In [ ]:



""" SEC Fine-Tuning Dataset Builder v21 — LOCAL RAG EDITION
================================================================
Replaces live EDGAR HTML scraping with local RAG over pre-downloaded filings.

v21 UPGRADES over v20:
──────────────────────────
V21-1 Local HTML Reader
    • Reads from filings/<TICKER>/<YEAR>/<FORM>_<PERIOD>_<ACCNO>.html
    • Zero EDGAR network calls for text — only XBRL still fetched live
    • Parses filename to extract ticker, year, form_type, period, accno

V21-2 iXBRL Cleaner
    • Full strip of ix:hidden, ix:nonFraction, ix:nonNumeric wrappers
    • Keeps all financial text including tables and inline numbers
    • 95%+ MD&A availability vs ~60% in v20

V21-3 Hierarchical SEC Chunker (3-level)
    • Level-1: Full section (mda, risk_factors, financial_statements, notes, business) — coarse retrieval
    • Level-2: Paragraph clusters 400-600 tokens with 50-token overlap
    • Level-3: Financial tables tagged separately (is_table=true)
    • Notes chunked by note number with note_subject metadata

V21-4 Rich Chunk Metadata (per the architecture spec)
    • ticker, company_name, year, period_end, form_type, fiscal_quarter
    • section, chunk_index, chunk_total, accession_no, filing_date
    • is_quantitative (≥3 $ or % figures), causal_connector_count
    • section_quality_score (pre-computed at index time)
    • is_table, table_type, note_subject

V21-5 Qdrant HNSW Vector Store (local, no cloud)
    • In-memory mode (no Docker required) for notebook use
    • HNSW index: m=16, ef_construct=200
    • Sentence-Transformers: all-MiniLM-L6-v2 (fast, good financial recall)
    • One collection per run: "sec_filings_v21"

V21-6 Hybrid Retrieval (dense + BM25 sparse)
    • BM25 for exact keyword matching (revenue grew vs declined)
    • Dense HNSW for semantic similarity
    • Reciprocal Rank Fusion (RRF) to combine scores
    • Mandatory metadata pre-filter: ticker + year + form_type
    • Quality gate: section_quality_score ≥ threshold

V21-7 scrape_text_sections() DROP-IN REPLACEMENT
    • New function: rag_retrieve_text_sections(ticker, year, form_type)
    • Returns identical dict shape: {mda, risk_factors, financial_statements, notes, business} — all downstream v20 code unchanged
    • Falls back to direct full-section text if retrieval returns < 500 chars

V21-8 Filing Manifest
    • Auto-scans filings/ folder on startup
    • Builds manifest: {ticker: {year: {form_type: [paths]}}}
    • Triggers index build for any filing not yet in Qdrant

All v20 classes retained: DerivedMetricEngine, CausalKeywordScanner, EliteQualityValidator,
build_stage_a_prompt, build_stage_b_prompt, etc. XBRL extraction pipeline unchanged.
"""

import os
import re
import json
import sys
import time
import hashlib
import random
import warnings
import threading
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional, List, Dict, Tuple

import requests
import pandas as pd
import urllib3
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from loguru import logger
from tqdm import tqdm
from dotenv import load_dotenv

# ── RAG-specific imports ──────────────────────────────────────────────────────
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    Filter,
    FieldCondition,
    MatchValue,
    MatchAny,
    HnswConfigDiff,
    OptimizersConfigDiff,
)
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

load_dotenv(override=True)

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ─── LOGURU ───────────────────────────────────────────────────────────────────
logger.remove()
logger.add(
    sys.stderr,
    format="<green>{time:HH:mm:ss}</green> | <level>{level: <7}</level> | {message}",
    level="INFO",
    colorize=True,
)
logger.add(
    "sec_builder_v21.log",
    format="{time:YYYY-MM-DD HH:mm:ss} | {level: <7} | {function}:{line} | {message}",
    level="DEBUG",
    rotation="50 MB",
    retention="7 days",
)

# ─── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_FILE = "sec_finetune_v21.jsonl"
GOLD_FILE = "sec_finetune_v21_gold.jsonl"
SILVER_FILE = "sec_finetune_v21_silver.jsonl"
PENDING_FILE = "sec_pending_v21.jsonl"
QUALITY_LOG_FILE = "sec_quality_v21.jsonl"
SANITY_LOG_FILE = "sec_sanity_v21.jsonl"
SKIP_LOG_FILE = "sec_skip_v21.jsonl"

# V21 RAG config
FILINGS_ROOT = "filings"  # root of your downloaded filings
QDRANT_COLLECTION = "sec_filings_v21"
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"  # 384-dim, fast, good quality
EMBED_DIM = 384
HNSW_M = 16
HNSW_EF_CONSTRUCT = 200
TOP_K_PER_QUERY = 5  # chunks per semantic query
BM25_TOP_K = 10  # BM25 candidates before RRF
RRF_K = 60  # RRF constant
MIN_CHUNK_QUALITY_SCORE = 1  # min quality score to index chunk
RAG_MIN_CONTEXT_CHARS = 500  # fallback threshold
CHUNK_SIZE_TOKENS = 500  # ~500 words per paragraph chunk
CHUNK_OVERLAP_TOKENS = 50  # overlap between adjacent chunks

# V20 constants (unchanged)
YEARS_BACK = 5
EDGAR_SLEEP = 0.12
MAX_RETRIES = 5
BACKOFF_BASE = 2.0
BACKOFF_MAX = 120.0
PERIOD_FUZZ_DAYS = 5
MAX_TEXT_CHARS = 5000  # V21: raised from 6000 (no truncation needed)
MIN_PARA_LEN = 40
MIN_PARA_ALPHA = 0.40
LLM_429_MAX_WAIT = 600
LLM_CIRCUIT_BREAK_N = 5
MIN_NARRATIVE_DOC_BYTES = 80_000
MIN_SECTION_CHARS = 50
MEGA_CAP_SWING_THRESHOLD = 60.0
MIN_MDA_CHARS_FOR_RICH = 1000
TABLE_NOISE_THRESHOLD = 0.40
MIN_NUMERICAL_GATE = 15
CAUSAL_CONNECTOR_MIN = 7
MAX_AUDITOR_CYCLES = 1
MIN_METRICS_FLOOR = 25
MIN_METRICS_RESCAN = 20
CROSS_REF_DELTA_PCT = 2.0
NVIDIA_RPM_LIMIT = 40
NVIDIA_MIN_INTERVAL = 60.0 / NVIDIA_RPM_LIMIT
ADAPTIVE_STEP_DOWN_ON_429 = True
ADAPTIVE_STEP_UP_AFTER_N = 10
QUEUE_TAIL_PCT = 0.10
TAIL_TEMPERATURE = 0.3
TAIL_MAX_TOKENS = 6144
BASE_TEMPERATURE = 0.2
BASE_MAX_TOKENS = 2048
PARALLEL_WORKERS = 2
UA_ROTATE_EVERY = 50
CONN_RESET_EVERY = 20
CAUSAL_KEYWORD_MIN = 3
GOLD_QUALITY_THRESHOLD = 90
SILVER_QUALITY_THRESHOLD = 75
MIN_REQUIRED_NUMBERS_IN_OUTPUT = 8
DELTA_PENALTY_PER_MISS = 10
GOLD_REQUIRES_ABNORMAL_RETURN = True
ABNORMAL_RETURN_DAYS = 5
SPY_TICKER = "SPY"
NVIDIA_MODEL = "meta/llama-3.1-70b-instruct"

# ─── THREAD-SAFE GLOBALS ──────────────────────────────────────────────────────
_file_lock = threading.Lock()
_cik_lock = threading.Lock()
_rl_lock = threading.Lock()
_queue_lock = threading.Lock()
_queue_state = {"total": 0, "processed": 0}

# ─── USER-AGENT POOL ──────────────────────────────────────────────────────────
_USER_AGENTS = [
    "SECDatasetBuilder/2.0 (ML research; research@university.edu)",
    "SECDatasetBuilder/2.1 (Academic use; ml-research@lab.edu)",
    "FinancialDataCollector/2.0 (Research; data@finresearch.org)",
    "SECFilingParser/2.0 (Non-commercial; nlp-lab@university.edu)",
    "AcademicSECBot/2.0 (Research dataset; sec-research@ml.edu)",
]
_ua_idx = 0
_ua_lock = threading.Lock()

EDGAR_HEADERS = {
    "User-Agent": _USER_AGENTS[0],
    "Accept-Encoding": "gzip, deflate, br",
    "Accept": "application/json, text/html, */*",
}

def _next_ua() -> str:
    global _ua_idx
    with _ua_lock:
        ua = _USER_AGENTS[_ua_idx % len(_USER_AGENTS)]
        _ua_idx += 1
        return ua

# ══════════════════════════════════════════════════════════════════════════════
# V21-1: LOCAL FILING MANIFEST
# ══════════════════════════════════════════════════════════════════════════════
# Filename pattern: FORM_PERIOD_ACCNO.html
# e.g. 10-K_2025-09-27_0000320193-25-000079.html
#      10-Q_A_2024-06-29_0000320193-24-000081.html (10-Q/A saved as 10-Q_A)

_FNAME_RE = re.compile(
    r"^(10-K(?:_A)?|10-Q(?:_A)?)"  # form type (/ replaced with _)
    r"_(\d{4}-\d{2}-\d{2})"  # period_end date
    r"_([0-9\-]+)\.html$",  # accession number
    re.IGNORECASE,
)

def _safe_form(raw: str) -> str:
    """Convert filename token back to standard form type."""
    return raw.replace("_A", "/A").replace("_", "-")

def build_filing_manifest(root: str = FILINGS_ROOT) -> Dict:
    """
    Walk the filings/ tree and build:
    manifest[ticker][year][form_type] = [
        { path, ticker, year, form_type, period_end, accession_no }
    ]
    """
    manifest: Dict = {}
    root_path = Path(root)
    if not root_path.exists():
        logger.warning(f"Filings root not found: {root_path.resolve()}")
        return manifest

    for ticker_dir in sorted(root_path.iterdir()):
        if not ticker_dir.is_dir():
            continue
        ticker = ticker_dir.name.upper()
        manifest[ticker] = {}

        for year_dir in sorted(ticker_dir.iterdir()):
            if not year_dir.is_dir():
                continue
            year = year_dir.name
            manifest[ticker][year] = {}

            for html_file in sorted(year_dir.glob("*.html")):
                m = _FNAME_RE.match(html_file.name)
                if not m:
                    continue
                form_raw, period_end, accno = m.group(1), m.group(2), m.group(3)
                form_type = _safe_form(form_raw)

                entry = {
                    "path": str(html_file),
                    "ticker": ticker,
                    "year": int(year),
                    "form_type": form_type,
                    "period_end": period_end,
                    "accession_no": accno,
                }
                manifest[ticker].setdefault(year, {})
                manifest[ticker][year].setdefault(form_type, [])
                manifest[ticker][year][form_type].append(entry)

    total_files = sum(len(entries) for t in manifest.values() for y in t.values() for entries in y.values())
    logger.info(f"Filing manifest: {len(manifest)} tickers, {total_files} HTML files")
    return manifest

# ══════════════════════════════════════════════════════════════════════════════
# V21-2: iXBRL CLEANER + HTML-TO-TEXT
# ══════════════════════════════════════════════════════════════════════════════

_RE_IX_HIDDEN = re.compile(r"<ix:hidden[^>]*>.*?</ix:hidden\s*>", re.DOTALL | re.IGNORECASE)
_RE_IX_NONFRAC = re.compile(r"<ix:nonFraction[^>]*>(.*?)</ix:nonFraction\s*>", re.DOTALL | re.IGNORECASE)
_RE_IX_NONNUM = re.compile(r"<ix:nonNumeric[^>]*>(.*?)</ix:nonNumeric\s*>", re.DOTALL | re.IGNORECASE)
_RE_IX_CONT = re.compile(r"<ix:continuation[^>]*>(.*?)</ix:continuation\s*>", re.DOTALL | re.IGNORECASE)
_RE_IX_EXCL = re.compile(r"<ix:exclude[^>]*>.*?</ix:exclude\s*>", re.DOTALL | re.IGNORECASE)
_RE_IX_TAGS = re.compile(r"</?ix:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>", re.IGNORECASE)
_RE_XBRL_TAGS = re.compile(r"</?xbrli?:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>", re.IGNORECASE)
_RE_LINK_TAGS = re.compile(r"</?link:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>", re.IGNORECASE)

def clean_ixbrl(html: str) -> str:
    """Strip iXBRL wrapper tags; preserve all visible financial content."""
    html = _RE_IX_HIDDEN.sub("", html)
    html = _RE_IX_EXCL.sub("", html)
    html = _RE_IX_NONFRAC.sub(r"\1", html)
    html = _RE_IX_NONNUM.sub(r"\1", html)
    html = _RE_IX_CONT.sub(r"\1", html)
    html = _RE_IX_TAGS.sub("", html)
    html = _RE_XBRL_TAGS.sub("", html)
    html = _RE_LINK_TAGS.sub("", html)
    return html

def html_to_plain(html: str) -> str:
    """Convert HTML (post-iXBRL cleaning) to plain text."""
    try:
        soup = BeautifulSoup(html, "lxml")
        for tag in soup(["script", "style", "head", "noscript", "meta", "link"]):
            tag.decompose()
        text = soup.get_text(separator="\n")
    except Exception:
        text = re.sub(r"<[^>]+>", " ", html)

    # Normalise whitespace
    text = re.sub(r"\xa0", " ", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def load_filing_text(path: str) -> str:
    """Load a local HTML filing and return clean plain text."""
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        html = f.read()
    html = clean_ixbrl(html)
    return html_to_plain(html)

# ══════════════════════════════════════════════════════════════════════════════
# V21-3: SECTION SPLITTER
# Uses the same regex dictionaries as v20, but works on full un-truncated text
# ══════════════════════════════════════════════════════════════════════════════

SECTIONS_10K = {
    "business": [r"item\s+1[\.\s]*[-–—]?\s*business\b", r"(?<!\w)item\s+1(?!\s*[aAbB])\b"],
    "risk_factors": [r"item\s+1a[\.\s]*[-–—]?\s*risk\s+factor", r"item\s+1a\b"],
    "mda": [r"item\s+7[\.\s]*[-–—]?\s*management.{0,120}discussion", r"(?<!\w)item\s+7(?!\s*a)\b"],
    "market_risk": [r"item\s+7a[\.\s]*[-–—]?\s*quantitative", r"item\s+7a\b"],
    "financial_statements": [r"item\s+8[\.\s]*[-–—]?\s*financial\s+statement", r"(?<!\w)item\s+8\b"],
    "legal_proceedings": [r"item\s+3[\.\s]*[-–—]?\s*legal\s+proceed", r"(?<!\w)item\s+3\b"],
}

SECTIONS_10Q = {
    "mda": [r"item\s+2[\.\s]*[-–—]?\s*management.{0,120}discussion", r"item\s+2\b"],
    "market_risk": [r"item\s+3[\.\s]*[-–—]?\s*quantitative", r"item\s+3\b"],
    "risk_factors": [r"item\s+1a[\.\s]*[-–—]?\s*risk\s+factor", r"item\s+1a\b"],
    "financial_statements": [r"item\s+1[\.\s]*[-–—]?\s*financial\s+statement", r"(?<!\w)item\s+1(?!\s*a)\b"],
    "legal_proceedings": [r"item\s+1[\.\s]*[-–—]?\s*legal\s+proceed"],
}

_NEXT_ITEM_RE = re.compile(r"\bitem\s+\d{1,2}[ab]?\s*[\.\s]", re.IGNORECASE)
_NUM_RE = re.compile(
    r"(\$[\d,.]+[BbMmKk]?|\d+\.?\d*\s*%|\b\d{1,3}(?:,\d{3})+\b)",
    re.IGNORECASE)

FINANCIAL_KEYWORDS = {
    "revenue", "net income", "earnings", "operating", "margin", "profit",
    "growth", "decline", "increased", "decreased", "compared", "fiscal",
    "quarter", "year", "segment", "billion", "million", "percent", "%",
    "cash flow", "capex", "ebitda", "gross", "cost", "expense", "loss",
    "gain", "acquisition", "guidance", "outlook", "demand", "supply",
    "customers", "market share", "competition", "results", "sales",
    "income", "debt", "equity", "assets", "liabilities", "dividend",
    "repurchase", "buyback", "restructur", "impairment", "write",
}

CAUSAL_CONNECTORS_LIST = [
    "driven by", "offset by", "primarily due to", "resulted from",
    "due to", "attributable to", "stemming from", "resulting in",
    "because", "consequently", "owing to", "which led to",
    "benefiting from", "weighed by", "impacted by", "partially offset",
]

_CAUSAL_RE = [re.compile(rf"\b{re.escape(c)}\b", re.IGNORECASE) for c in CAUSAL_CONNECTORS_LIST]

def _find_section_in_text(text: str, patterns: list, skip_start: int = 3000) -> Tuple[int, int]:
    """
    Return (start, end) byte positions of a section in plain text.
    Tries from skip_start first (avoids TOC), then from 0 as fallback.
    Returns (-1, -1) if not found.
    """
    lower = text.lower()

    def _search(sf: int) -> Tuple[int, int]:
        start = -1
        for pat in patterns:
            m = re.search(pat, lower[sf:], re.DOTALL)
            if m:
                start = sf + m.start()
                break
        if start < 0:
            return -1, -1

        # Find where next Item starts
        tail = lower[start + 30:]
        end_m = _NEXT_ITEM_RE.search(tail)
        end = (start + 30 + end_m.start()) if end_m else len(text)
        return start, end

    s, e = _search(skip_start)
    if s >= 0:
        return s, e
    return _search(0)

def extract_all_sections(plain_text: str, form_type: str) -> Dict[str, str]:
    """
    Extract all named sections from plain text.
    Returns dict of section_name → raw text (no truncation).
    """
    section_map = SECTIONS_10K if "10-K" in form_type else SECTIONS_10Q
    skip = max(3000, int(len(plain_text) * 0.03))

    result = {}
    for name, patterns in section_map.items():
        s, e = _find_section_in_text(plain_text, patterns, skip)
        if s >= 0:
            raw = plain_text[s:e].strip()
            if len(raw) >= MIN_SECTION_CHARS:
                result[name] = raw
    return result

# ══════════════════════════════════════════════════════════════════════════════
# V21-4 QUALITY SCORERS (pre-compute at index time)
# ══════════════════════════════════════════════════════════════════════════════

def score_text_quality(text: str) -> int:
    """Return integer quality score: kw_hits*2 + num_hits."""
    if not text:
        return 0
    lower = text.lower()
    kw_hits = sum(1 for kw in FINANCIAL_KEYWORDS if kw in lower)
    num_hits = len(_NUM_RE.findall(text))
    return kw_hits * 2 + num_hits

def count_causal_connectors(text: str) -> int:
    """Count causal connector phrases in text."""
    return sum(len(pat.findall(text)) for pat in _CAUSAL_RE)

def is_quantitative(text: str, threshold: int = 3) -> bool:
    """Return True if text has ≥ threshold dollar/percent figures."""
    return len(_NUM_RE.findall(text)) >= threshold

def detect_table_type(text: str) -> Optional[str]:
    """Heuristic: classify table type from surrounding text."""
    lower = text.lower()
    if any(k in lower for k in ["net sales", "total net revenue", "operating income", "net income", "earnings per share"]):
        return "income_statement"
    if any(k in lower for k in ["total assets", "total liabilities", "stockholders equity", "cash and cash equivalents"]):
        return "balance_sheet"
    if any(k in lower for k in ["operating activities", "investing activities", "financing activities", "free cash flow"]):
        return "cash_flow"
    return None

def extract_note_subject(text: str) -> Optional[str]:
    """Extract the topic of a Note to Financial Statements."""
    m = re.search(r"note\s+\d+[\.\s]*[-–—]?\s*([A-Z][A-Za-z\s&,]+)", text[:200], re.IGNORECASE)
    if m:
        return m.group(1).strip()[:60]
    return None

# ══════════════════════════════════════════════════════════════════════════════
# V21-3: HIERARCHICAL CHUNKER (continued)
# ══════════════════════════════════════════════════════════════════════════════

def _split_into_paragraphs(text: str) -> List[str]:
    """Split text on blank lines; filter very short lines."""
    paras = re.split(r"\n{2,}", text)
    return [p.strip() for p in paras if p.strip() and len(p.strip()) >= MIN_PARA_LEN]

def _approx_tokens(text: str) -> int:
    """Rough token count (1 token ≈ 4 chars)."""
    return len(text) // 4

def _cluster_paragraphs(
    paras: List[str],
    target_tokens: int = CHUNK_SIZE_TOKENS,
    overlap_tokens: int = CHUNK_OVERLAP_TOKENS
) -> List[str]:
    """
    Group paragraphs into chunks of ~target_tokens with overlap.
    Each chunk is a string (joined paragraphs).
    """
    chunks = []
    current = []
    cur_size = 0

    for para in paras:
        pt = _approx_tokens(para)
        if cur_size + pt > target_tokens and current:
            # Emit current chunk
            chunks.append("\n\n".join(current))

            # Keep tail for overlap
            overlap_size = 0
            overlap_paras = []
            for p in reversed(current):
                overlap_size += _approx_tokens(p)
                overlap_paras.insert(0, p)
                if overlap_size >= overlap_tokens:
                    break
            current = overlap_paras
            cur_size = sum(_approx_tokens(p) for p in current)

        current.append(para)
        cur_size += pt

    if current:
        chunks.append("\n\n".join(current))

    return chunks

def _is_table_region(text: str) -> bool:
    """Heuristic: returns True if text looks like a financial table."""
    lines = text.split("\n")
    num_lines = len(lines)
    if num_lines < 3:
        return False
    digit_lines = sum(1 for ln in lines if len(re.findall(r"\d", ln)) >= 3)
    return digit_lines / num_lines >= 0.5


def chunk_section(
    section_text: str,
    section_name: str,
    filing_meta: dict,
) -> List[dict]:
    ticker   = filing_meta["ticker"]
    year     = filing_meta["year"]
    form_type= filing_meta["form_type"]
    period_end= filing_meta["period_end"]
    accno    = filing_meta["accession_no"]
    fq       = filing_meta.get("fiscal_quarter")

    chunks: List[dict] = []

    # ── Level-1: full section (always include — never gate on quality score) ──
    l1_id   = hashlib.md5(f"{ticker}_{accno}_{section_name}_L1".encode()).hexdigest()
    l1_text = section_text[:12000]
    qs  = score_text_quality(l1_text)
    cc  = count_causal_connectors(l1_text)

    chunks.append({                        # ← no quality gate on L1
        "id": l1_id,
        "text": l1_text,
        "level": 1,
        "section": section_name,
        "chunk_index": 0,
        "chunk_total": 1,
        "ticker": ticker,
        "year": year,
        "form_type": form_type,
        "period_end": period_end,
        "fiscal_quarter": fq,
        "accession_no": accno,
        "is_quantitative": is_quantitative(l1_text),
        "causal_connector_count": cc,
        "section_quality_score": qs,
        "is_table": False,
        "table_type": None,
        "note_subject": None,
    })

    # ── Level-2: paragraph clusters ──────────────────────────────────────────
    paras     = _split_into_paragraphs(section_text)
    table_buf = []
    prose_buf = []

    for para in paras:
        if _is_table_region(para):
            table_buf.append(para)
        else:
            prose_buf.append(para)

    l2_chunks = _cluster_paragraphs(prose_buf)
    for ci, chunk_text in enumerate(l2_chunks):
        qs2 = score_text_quality(chunk_text)
        if qs2 < MIN_CHUNK_QUALITY_SCORE:   # MIN_CHUNK_QUALITY_SCORE=1 now
            continue
        cid = hashlib.md5(f"{ticker}_{accno}_{section_name}_L2_{ci}".encode()).hexdigest()
        chunks.append({
            "id": cid,
            "text": chunk_text,
            "level": 2,
            "section": section_name,
            "chunk_index": ci,
            "chunk_total": len(l2_chunks),
            "ticker": ticker,
            "year": year,
            "form_type": form_type,
            "period_end": period_end,
            "fiscal_quarter": fq,
            "accession_no": accno,
            "is_quantitative": is_quantitative(chunk_text),
            "causal_connector_count": count_causal_connectors(chunk_text),
            "section_quality_score": qs2,
            "is_table": False,
            "table_type": None,
            "note_subject": None,
        })

    # ── Level-3: table regions ────────────────────────────────────────────────
    for ti, tbl in enumerate(table_buf):
        qs3 = score_text_quality(tbl)
        if qs3 < MIN_CHUNK_QUALITY_SCORE:
            continue
        ttype = detect_table_type(tbl)
        cid = hashlib.md5(f"{ticker}_{accno}_{section_name}_L3_tbl_{ti}".encode()).hexdigest()
        chunks.append({
            "id": cid,
            "text": tbl,
            "level": 3,
            "section": section_name,
            "chunk_index": ti,
            "chunk_total": len(table_buf),
            "ticker": ticker,
            "year": year,
            "form_type": form_type,
            "period_end": period_end,
            "fiscal_quarter": fq,
            "accession_no": accno,
            "is_quantitative": True,
            "causal_connector_count": 0,
            "section_quality_score": qs3,
            "is_table": True,
            "table_type": ttype,
            "note_subject": None,
        })

    return chunks




def chunk_notes_section(
    notes_text: str,
    filing_meta: dict,
) -> List[dict]:
    """
    Special handling for Notes to Financial Statements: split by individual Note
    (Note 1, Note 2, ...) and tag note_subject.
    """
    ticker = filing_meta["ticker"]
    accno = filing_meta["accession_no"]
    year = filing_meta["year"]
    form_type = filing_meta["form_type"]
    period_end = filing_meta["period_end"]
    fq = filing_meta.get("fiscal_quarter")

    note_splits = re.split(r"\n(?=Note\s+\d+[\.\s])", notes_text, flags=re.IGNORECASE)
    chunks: List[dict] = []

    for ni, note_text in enumerate(note_splits):
        note_text = note_text.strip()
        if len(note_text) < MIN_SECTION_CHARS:
            continue
        qs = score_text_quality(note_text)
        if qs < MIN_CHUNK_QUALITY_SCORE:
            continue
        subj = extract_note_subject(note_text)
        cid = hashlib.md5(f"{ticker}_{accno}_notes_N{ni}".encode()).hexdigest()
        chunks.append({
            "id": cid,
            "text": note_text[:6000],
            "level": 2,
            "section": "notes",
            "chunk_index": ni,
            "chunk_total": len(note_splits),
            "ticker": ticker,
            "year": year,
            "form_type": form_type,
            "period_end": period_end,
            "fiscal_quarter": fq,
            "accession_no": accno,
            "is_quantitative": is_quantitative(note_text),
            "causal_connector_count": count_causal_connectors(note_text),
            "section_quality_score": qs,
            "is_table": False,
            "table_type": None,
            "note_subject": subj,
        })

    return chunks

def chunk_filing(plain_text: str, filing_meta: dict) -> List[dict]:
    """
    Full hierarchical chunking of one filing.
    Returns all chunks (L1 + L2 + L3) across all sections.
    """
    form_type = filing_meta["form_type"]
    sections = extract_all_sections(plain_text, form_type)

    all_chunks: List[dict] = []
    for section_name, section_text in sections.items():
        if section_name == "notes":
            notes_chunks = chunk_notes_section(section_text, filing_meta)
            all_chunks.extend(notes_chunks)
        else:
            sec_chunks = chunk_section(section_text, section_name, filing_meta)
            all_chunks.extend(sec_chunks)

    logger.debug(
        f"  Chunked {filing_meta['ticker']} {filing_meta['period_end']} "
        f"{form_type}: {len(sections)} sections → {len(all_chunks)} chunks"
    )
    return all_chunks

# ══════════════════════════════════════════════════════════════════════════════
# V21-5: QDRANT VECTOR STORE (in-memory, no Docker required)
# ══════════════════════════════════════════════════════════════════════════════

class SECVectorStore:
    """
    Manages Qdrant in-memory vector store for SEC filings.
    Uses sentence-transformers for dense embeddings.
    Supports hybrid retrieval via BM25 + dense vectors.
    """

    def __init__(
        self,
        collection: str = QDRANT_COLLECTION,
        embed_model: str = EMBED_MODEL_NAME,
    ):
        self.collection = collection
        self._client = None
        self._embedder = None
        self._embed_model = embed_model
        self._bm25_index: Dict[str, BM25Okapi] = {}  # key = ticker|year|form
        self._bm25_chunks: Dict[str, List[dict]] = {}
        self._indexed_ids: set = set()
        self._lock = threading.Lock()

    # ── Lazy init ─────────────────────────────────────────────────────────────
    def _get_client(self) -> QdrantClient:
        if self._client is None:
            self._client = QdrantClient(":memory:")
            self._ensure_collection()
        return self._client

    def _get_embedder(self) -> SentenceTransformer:
        if self._embedder is None:
            logger.info(f"Loading embedding model: {self._embed_model}")
            self._embedder = SentenceTransformer(self._embed_model)
        return self._embedder

    def _ensure_collection(self):
        client = self._client
        existing = [c.name for c in client.get_collections().collections]
        if self.collection not in existing:
            client.create_collection(
                collection_name=self.collection,
                vectors_config=VectorParams(
                    size=EMBED_DIM,
                    distance=Distance.COSINE,
                ),
                hnsw_config=HnswConfigDiff(
                    m=HNSW_M,
                    ef_construct=HNSW_EF_CONSTRUCT,
                    full_scan_threshold=10_000,
                ),
                optimizers_config=OptimizersConfigDiff(
                    indexing_threshold=20_000,
                ),
            )
            logger.info(f"Created Qdrant collection: {self.collection}")

    # ── Embedding ─────────────────────────────────────────────────────────────
    def _embed(self, texts: List[str]) -> List[List[float]]:
        embedder = self._get_embedder()
        return embedder.encode(texts, show_progress_bar=False, batch_size=32).tolist()

    # ── BM25 index key ────────────────────────────────────────────────────────
    @staticmethod
    def _bm25_key(ticker: str, year, form_type: str) -> str:
        return f"{ticker}|{year}|{form_type}"

    # ── Upsert ────────────────────────────────────────────────────────────────
    def upsert_chunks(self, chunks: List[dict]):
        if not chunks:
            return

        client = self._get_client()

        # Filter already-indexed
        with self._lock:
            new_chunks = [c for c in chunks if c["id"] not in self._indexed_ids]

        if not new_chunks:
            return

        texts = [c["text"] for c in new_chunks]
        vectors = self._embed(texts)

        points = []
        for chunk, vec in zip(new_chunks, vectors):
            # Payload = all metadata except the raw text (stored too for fallback)
            payload = {k: v for k, v in chunk.items() if k not in ("id",)}
            points.append(PointStruct(
                id=abs(int(chunk["id"][:8], 16)),  # Qdrant needs int or uuid
                vector=vec,
                payload=payload,
            ))

        client.upsert(collection_name=self.collection, points=points, wait=True)

        # Update BM25 indices grouped by ticker+year+form
        bm25_groups: Dict[str, List[dict]] = {}
        for chunk in new_chunks:
            key = self._bm25_key(chunk["ticker"], chunk["year"], chunk["form_type"])
            bm25_groups.setdefault(key, []).append(chunk)

        with self._lock:
            for key, group in bm25_groups.items():
                existing = self._bm25_chunks.get(key, [])
                existing.extend(group)
                self._bm25_chunks[key] = existing
                tokenised = [c["text"].lower().split() for c in existing]
                self._bm25_index[key] = BM25Okapi(tokenised)

            for c in new_chunks:
                self._indexed_ids.add(c["id"])

        logger.debug(f"  Upserted {len(new_chunks)} chunks to Qdrant")

    # ── BM25 Search ───────────────────────────────────────────────────────────
    def _bm25_search(
        self,
        query: str,
        key: str,
        top_k: int = BM25_TOP_K
    ) -> List[Tuple[dict, float]]:
        """Return list of (chunk, bm25_score) sorted descending."""
        bm25 = self._bm25_index.get(key)
        chunks = self._bm25_chunks.get(key, [])
        if not bm25 or not chunks:
            return []

        tokens = query.lower().split()
        scores = bm25.get_scores(tokens)
        ranked = sorted(
            zip(chunks, scores.tolist()),
            key=lambda x: x[1],
            reverse=True
        )
        return ranked[:top_k]

    # ── Dense Search ──────────────────────────────────────────────────────────

    def _dense_search(
        self,
        query: str,
        ticker: str,
        year: int,
        form_type: str,
        section: Optional[str] = None,
        quant_only: bool = False,
        top_k: int = TOP_K_PER_QUERY,
        period_end: Optional[str] = None, 
    ) -> List[Tuple[dict, float]]:
        """Return list of (chunk_payload, score) from Qdrant — compatible with qdrant-client ≥1.7."""
        client = self._get_client()
        vec = self._embed([query])[0]

        must_conditions = [
            FieldCondition(key="ticker",    match=MatchValue(value=ticker)),
            FieldCondition(key="year",      match=MatchValue(value=year)),
            FieldCondition(key="form_type", match=MatchValue(value=form_type)),
        ]
        if period_end:                          # ← ADD
            must_conditions.append(
                FieldCondition(key="period_end", match=MatchValue(value=period_end))
            )
        if section:
            must_conditions.append(
                FieldCondition(key="section", match=MatchValue(value=section))
            )
        if quant_only:
            must_conditions.append(
                FieldCondition(key="is_quantitative", match=MatchValue(value=True))
            )

        qfilter = Filter(must=must_conditions)

        # ── qdrant-client ≥ 1.7: use query_points ────────────────────────────────
        try:
            response = client.query_points(
                collection_name=self.collection,
                query=vec,
                query_filter=qfilter,
                limit=top_k,
                with_payload=True,
            )
            points = response.points          # QueryResponse.points → List[ScoredPoint]
        except AttributeError:
            # ── qdrant-client < 1.7 fallback: use search ─────────────────────────
            points = client.search(
                collection_name=self.collection,
                query_vector=vec,
                query_filter=qfilter,
                limit=top_k,
                with_payload=True,
            )

        return [(p.payload, p.score) for p in points]



    # ── Reciprocal Rank Fusion ────────────────────────────────────────────────
    @staticmethod
    def _rrf(
        dense_results: List[Tuple[dict, float]],
        bm25_results: List[Tuple[dict, float]],
        k: int = RRF_K,
    ) -> List[Tuple[dict, float]]:
        """Combine dense + BM25 rankings with RRF."""
        scores: Dict[str, float] = {}
        chunks: Dict[str, dict] = {}

        for rank, (chunk, _) in enumerate(dense_results):
            cid = chunk.get("accession_no", "") + chunk.get("section", "") + str(chunk.get("chunk_index", 0))
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
            chunks[cid] = chunk

        for rank, (chunk, _) in enumerate(bm25_results):
            cid = chunk.get("accession_no", "") + chunk.get("section", "") + str(chunk.get("chunk_index", 0))
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
            chunks[cid] = chunk

        merged = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [(chunks[cid], score) for cid, score in merged]

    # ── Public hybrid retrieval ───────────────────────────────────────────────
    def retrieve(
        self,
        query: str,
        ticker: str,
        year: int,
        form_type: str,
        section: Optional[str] = None,
        quant_only: bool = False,
        top_k: int = TOP_K_PER_QUERY,
        period_end: Optional[str] = None, 
    ) -> List[dict]:
        """
        Hybrid (dense + BM25) retrieval with RRF fusion.
        Returns list of chunk payloads sorted by relevance.
        Quality-filtered: only chunks with section_quality_score ≥ threshold.
        """
        # Dense retrieval
        dense = self._dense_search(query, ticker, year, form_type, section, quant_only, top_k * 2, period_end=period_end)

        # BM25 retrieval
        key = self._bm25_key(ticker, year, form_type)
        bm25 = self._bm25_search(query, key, BM25_TOP_K)

        # RRF fusion
        fused = self._rrf(dense, bm25)

        # Apply quality filter and return top_k
        filtered = [
            chunk for chunk, _ in fused
            if chunk.get("section_quality_score", 0) >= MIN_CHUNK_QUALITY_SCORE
        ]
        return filtered[:top_k]

    def is_indexed(self, accession_no: str) -> bool:
        """Check if a filing (by accession number) is already indexed."""
        with self._lock:
            return any(
                any(c.get("accession_no") == accession_no for c in chunks)
                for chunks in self._bm25_chunks.values()
            )


# ── Singleton vector store ─────────────────────────────────────────────────────
_vector_store: Optional[SECVectorStore] = None

def get_vector_store() -> SECVectorStore:
    global _vector_store
    if _vector_store is None:
        _vector_store = SECVectorStore()
    return _vector_store

# ══════════════════════════════════════════════════════════════════════════════
# V21-6: INDEX PIPELINE (index a single filing from local HTML)
# ══════════════════════════════════════════════════════════════════════════════

def _get_fiscal_quarter(period_end: str, form_type: str) -> Optional[int]:
    if not period_end:
        return None
    try:
        dt = datetime.strptime(period_end[:10], "%Y-%m-%d")
    except Exception:
        return None
    if "10-K" in form_type:
        return 4
    m = dt.month
    if m in (1, 2, 3, 4):
        return 1
    if m in (4, 5, 6, 7):
        return 2
    if m in (7, 8, 9, 10):
        return 3
    return 4

def index_filing(entry: dict, vs: Optional[SECVectorStore] = None) -> int:
    """
    Index one filing from its local HTML file.
    Returns number of chunks indexed.
    """
    vs = vs or get_vector_store()

    # Skip if already indexed
    if vs.is_indexed(entry["accession_no"]):
        return 0

    logger.info(
        f"  Indexing {entry['ticker']} {entry['form_type']} "
        f"{entry['period_end']} ({Path(entry['path']).name})"
    )

    try:
        plain_text = load_filing_text(entry["path"])
    except Exception as e:
        logger.warning(f"  Failed to load {entry['path']}: {e}")
        return 0

    filing_meta = {
        "ticker": entry["ticker"],
        "year": int(entry["year"]),
        "form_type": entry["form_type"],
        "period_end": entry["period_end"],
        "accession_no": entry["accession_no"],
        "fiscal_quarter": _get_fiscal_quarter(entry["period_end"], entry["form_type"]),
    }

    chunks = chunk_filing(plain_text, filing_meta)
    vs.upsert_chunks(chunks)
    logger.info(f"  → {len(chunks)} chunks indexed")
    return len(chunks)

def index_ticker(
    ticker: str,
    manifest: Dict,
    vs: Optional[SECVectorStore] = None,
    years: Optional[List[int]] = None,
    form_types: Optional[List[str]] = None,
) -> int:
    """Index all filings for a given ticker from the manifest."""
    vs = vs or get_vector_store()
    ticker_data = manifest.get(ticker.upper(), {})
    if not ticker_data:
        logger.warning(f"No filings in manifest for {ticker}")
        return 0

    total = 0
    for year_str, year_data in sorted(ticker_data.items()):
        yr = int(year_str)
        if years and yr not in years:
            continue
        for ft, entries in year_data.items():
            if form_types and ft not in form_types:
                continue
            for entry in entries:
                total += index_filing(entry, vs)
    return total

# ══════════════════════════════════════════════════════════════════════════════
# V21-7: RAG RETRIEVAL — DROP-IN REPLACEMENT FOR scrape_text_sections()
# ══════════════════════════════════════════════════════════════════════════════

# Query templates per section — adapted from the RAG architecture spec
_SECTION_QUERIES: Dict[str, List[dict]] = {
    "mda": [
        {"q": "revenue net sales growth increase decrease year over year", "quant": True},
        {"q": "gross margin cost of revenue COGS compression expansion driven by", "quant": False},
        {"q": "operating income loss from operations expense driven by", "quant": True},
        {"q": "net income earnings per share diluted attributable to", "quant": True},
        {"q": "free cash flow capital expenditures operating activities", "quant": True},
        {"q": "management discussion overview results of operations", "quant": False},
    ],
    "risk_factors": [
        {"q": "risk factor competition regulation supply chain cybersecurity", "quant": False},
        {"q": "geopolitical macroeconomic inflation interest rate litigation", "quant": False},
    ],
    "financial_statements": [
        {"q": "consolidated statements of operations net sales total revenue", "quant": True},
        {"q": "balance sheet total assets liabilities stockholders equity", "quant": True},
        {"q": "cash flow operating activities capital expenditures", "quant": True},
    ],
    "notes": [
        {"q": "segment revenue operating income reportable business segment", "quant": True},
        {"q": "notes to consolidated financial statements significant accounting", "quant": False},
    ],
    "business": [
        {"q": "business overview products services markets competition", "quant": False},
    ],
}

def rag_retrieve_text_sections(
    ticker: str,
    year: int,
    form_type: str,
    vs: Optional[SECVectorStore] = None,
    manifest: Optional[Dict] = None,
    period_end: Optional[str] = None,  
) -> Tuple[Dict[str, str], str]:
    """
    V21-7: Drop-in replacement for scrape_text_sections().
    1. Ensures the filing is indexed (indexes on-demand if not yet done).
    2. Runs multi-query retrieval for each section.
    3. Assembles context from retrieved chunks (deduplicated, position-sorted).
    4. Falls back to direct section extraction from local file if retrieval weak.
    Returns: (text_sections dict, plain_text of full filing)
        text_sections = {mda, risk_factors, financial_statements, notes, business}
    """
    vs = vs or get_vector_store()

    # ── Ensure indexed ────────────────────────────────────────────────────────
    if manifest:
        ticker_data = manifest.get(ticker.upper(), {})
        year_data = ticker_data.get(str(year), {})
        entries = []
        # Match base form (10-Q/A → 10-Q base)
        base_form = form_type.split("/")[0]
        for ft, elist in year_data.items():
            if ft == form_type or ft.startswith(base_form):
                entries.extend(elist)
        for entry in entries:
            if not vs.is_indexed(entry["accession_no"]):
                index_filing(entry, vs)

    result: Dict[str, str] = {}

    # ── Retrieve each section ─────────────────────────────────────────────────
    for section_name, query_list in _SECTION_QUERIES.items():
        seen_chunk_ids = set()
        section_chunks: List[dict] = []

        for qdef in query_list:
            retrieved = vs.retrieve(
                query=qdef["q"],
                ticker=ticker,
                year=year,
                form_type=form_type,
                section=section_name,
                quant_only=qdef["quant"],
                top_k=TOP_K_PER_QUERY,
                period_end=period_end,
            )
            for chunk in retrieved:
                cid = chunk.get("accession_no", "") + str(chunk.get("chunk_index", 0)) + chunk.get("section", "")
                if cid not in seen_chunk_ids:
                    seen_chunk_ids.add(cid)
                    section_chunks.append(chunk)

        if not section_chunks:
            continue

        # Sort by chunk_index to preserve reading order
        section_chunks.sort(key=lambda c: (c.get("level", 2), c.get("chunk_index", 0)))
        assembled = "\n\n".join(c["text"] for c in section_chunks)
        result[section_name] = assembled[:MAX_TEXT_CHARS]

    # ── Quality check + direct fallback ──────────────────────────────────────
    # If a critical section (mda) is too short, fall back to direct extraction
    if len(result.get("mda", "")) < RAG_MIN_CONTEXT_CHARS and manifest:
        logger.debug(f"  RAG fallback: MD&A too short for {ticker} {year} {form_type}")

        ticker_data = manifest.get(ticker.upper(), {})
        year_data = ticker_data.get(str(year), {})
        base_form = form_type.split("/")[0]

        for ft, entries in year_data.items():
            if not (ft == form_type or ft.startswith(base_form)):
                continue
            for entry in entries:
                try:
                    plain = load_filing_text(entry["path"])
                    sections = extract_all_sections(plain, form_type)
                    for sname, stext in sections.items():
                        if sname not in result or len(result[sname]) < RAG_MIN_CONTEXT_CHARS:
                            result[sname] = stext[:MAX_TEXT_CHARS]
                    # Return full plain text for segment extraction etc.
                    return result, plain
                except Exception as e:
                    logger.warning(f"  Fallback extraction failed: {e}")

    # Get full plain text for XBRL + segment extraction
    plain_text = ""
    if manifest:
        ticker_data = manifest.get(ticker.upper(), {})
        year_data = ticker_data.get(str(year), {})
        base_form = form_type.split("/")[0]
        for ft, entries in year_data.items():
            if ft == form_type or ft.startswith(base_form):
                for entry in entries:
                    try:
                        plain_text = load_filing_text(entry["path"])
                        break
                    except Exception:
                        pass
                if plain_text:
                    break

    # Log retrieval quality
    mda_len = len(result.get("mda", ""))
    rf_len = len(result.get("risk_factors", ""))
    fs_len = len(result.get("financial_statements", ""))
    logger.debug(
        f"  RAG {ticker}/{year}/{form_type}: "
        f"mda={mda_len}, rf={rf_len}, fs={fs_len}, "
        f"sections={len(result)}"
    )

    return result, plain_text

# ══════════════════════════════════════════════════════════════════════════════
# ALL v20 CLASSES AND FUNCTIONS (unchanged — copy verbatim from v20)
# Only scrape_text_sections is replaced with rag_retrieve_text_sections above
# ══════════════════════════════════════════════════════════════════════════════

# ─── Causal connectors & lexicons ────────────────────────────────────────────
CANONICAL_TERMS = {
    "profitability pressure": "margin pressure",
    "earnings pressure": "margin pressure",
    "profit decline": "margin pressure",
    "profit compression": "margin pressure",
    "margin deterioration": "margin pressure",
    "debt burden": "leverage pressure",
    "debt load": "leverage pressure",
    "heavy debt": "leverage pressure",
    "liquidity issue": "liquidity pressure",
    "liquidity concern": "liquidity pressure",
    "cash crunch": "liquidity pressure",
    "revenue contraction": "revenue decline",
    "top-line growth": "revenue growth",
    "top-line decline": "revenue decline",
    "bottom-line growth": "net income growth",
    "bottom-line pressure": "margin pressure",
    "operational efficiency": "operating leverage improvement",
    "cost optimization": "cost structure improvement",
    "capital efficiency": "asset utilization",
    "strong performance": "above-average financial results",
    "weak performance": "below-average financial results",
}

FILLER_TOKENS = {
    "suggesting", "indicating", "overall", "reflects", "demonstrates",
    "showcases", "underscores", "highlights", "illustrates", "notably",
    "importantly", "significantly", "meaningfully", "materially",
    "generally", "broadly", "largely", "essentially", "fundamentally",
    "incredibly", "remarkably", "substantially", "considerably",
    "improved", "declined", "strong", "weak", "robust", "solid",
    "healthy", "challenging", "difficult",
}

CAUSAL_CONNECTORS = [
    r"\bprimary driver being\b",
    r"\bresulting in\b",
    r"\bstemming from\b",
    r"\bconsequently\b",
    r"\battributable to\b",
    r"\bowing to\b",
    r"\bbecause\b",
    r"\bdriven by\b",
    r"\bdue to\b",
    r"\bwhich led to\b",
    r"\bprimarily from\b",
    r"\bwhich reflects\b",
    r"\bfollowing\b",
    r"\bresulting from\b",
    r"\boffset by\b",
    r"\bprimarily due to\b",
    r"\bresulted from\b",
    r"\bdriving\b",
    r"\bimpacted by\b",
    r"\bbenefiting from\b",
    r"\bweighed by\b",
    r"\bpartially offset by\b",
]

_CAUSAL_CONN_RE = [re.compile(p, re.IGNORECASE) for p in CAUSAL_CONNECTORS]

RISK_KW = [
    "competition", "regulation", "supply chain", "cybersecurity",
    "inflation", "interest rate", "liquidity", "litigation",
    "currency risk", "macroeconomic", "recession", "geopolitical",
    "tariff", "sanctions", "climate", "data privacy",
    "customer concentration", "patent", "labor shortage", "ai disruption",
]

ITEM_HEADING_RE = re.compile(r"\bitem\s+(?:1a?|2|3|7a?|8)\b", re.IGNORECASE)

NEGATIVE_REASONING_TEMPLATES = [
    {
        "trigger": "revenue_yoy_pct < 5 and revenue_yoy_pct > 0",
        "bad_conclusion": "Strong market demand drove revenue.",
        "correct_conclusion": "Revenue grew modestly at {revenue_yoy_pct:.1f}% YoY; no strong acceleration signal.",
        "label": "weak_growth_overclaim",
    },
    {
        "trigger": "revenue_yoy_pct > 15 and not mda_has_text",
        "bad_conclusion": "Product demand and market expansion drove growth.",
        "correct_conclusion": "Revenue grew {revenue_yoy_pct:.1f}% YoY. No MD&A; driver unidentifiable.",
        "label": "growth_without_driver_evidence",
    },
    {
        "trigger": "gross_margin_pct_delta < -2 and not mda_has_text",
        "bad_conclusion": "Supply chain disruption caused margin compression.",
        "correct_conclusion": "Gross margin compressed {gross_margin_pct_delta:.1f}pp YoY. No explicit cost driver identified.",
        "label": "margin_compression_unsupported",
    },
    {
        "trigger": "operating_margin_pct > 20 and net_margin_pct < 5",
        "bad_conclusion": "Strong operations indicate overall business health.",
        "correct_conclusion": "Operating margin {operating_margin_pct:.1f}% but net margin only {net_margin_pct:.1f}% — below-the-line costs reduce profitability.",
        "label": "operating_vs_net_margin_disconnect",
    },
    {
        "trigger": "fcf_yoy_pct > 0 and revenue_yoy_pct < -5",
        "bad_conclusion": "Business momentum is positive given improving cash flows.",
        "correct_conclusion": "FCF improved {fcf_yoy_pct:.1f}% YoY despite revenue declining {revenue_yoy_pct:.1f}%. May reflect cost-cutting, not acceleration.",
        "label": "fcf_improvement_revenue_decline",
    },
]

EDGE_CASE_ARCHETYPES = {
    "unprofitable_expansion": {
        "condition": lambda m, c: (c.get("revenue_yoy_pct", 0) or 0) > 15 and (m.get("operating_margin_pct", 0) or 0) < -5,
        "label": "growth_margin_collapse",
        "reasoning_focus": "High growth + negative margins = investment phase or structural cost problem.",
        "reasoning_path": "Income Statement → Operating Margin → Investment Phase Risk",
    },
    "efficiency_harvest": {
        "condition": lambda m, c: abs((c.get("revenue_yoy_pct", 0) or 0)) < 3 and (c.get("fcf_yoy_pct", 0) or 0) > 15,
        "label": "flat_revenue_fcf_improvement",
        "reasoning_focus": "Flat revenue + improving FCF = operational efficiency or CapEx reduction.",
        "reasoning_path": "Cash Flow → FCF Conversion → Efficiency Signal",
    },
    "restructuring_signal": {
        "condition": lambda m, c: (c.get("revenue_yoy_pct", 0) or 0) < -5 and (c.get("gross_margin_pct_delta", 0) or 0) > 2,
        "label": "revenue_decline_margin_improvement",
        "reasoning_focus": "Revenue decline + margin improvement = portfolio pruning or mix shift.",
        "reasoning_path": "Revenue → Mix Shift → Margin Structure",
    },
    "manageable_leverage": {
        "condition": lambda m, c: (m.get("debt_to_equity", 0) or 0) > 2.0 and (c.get("fcf_yoy_pct", 0) or 0) > 0 and (m.get("free_cash_flow", 0) or 0) > 0,
        "label": "high_leverage_strong_cashflow",
        "reasoning_focus": "High D/E alone is insufficient — FCF coverage determines debt serviceability.",
        "reasoning_path": "Balance Sheet → Leverage → FCF Coverage → Debt Risk",
    },
    "liquidity_paradox": {
        "condition": lambda m, c: (m.get("current_ratio", 99) or 99) < 1.0 and (m.get("cash_and_equivalents", 0) or 0) > 1e9,
        "label": "low_current_ratio_high_cash",
        "reasoning_focus": "Current ratio < 1.0 with large absolute cash — nuanced interpretation, not automatic crisis.",
        "reasoning_path": "Balance Sheet → Liquidity → Paradox → Risk Assessment",
    },
    "conflicting_earnings": {
        "condition": lambda m, c: (c.get("revenue_yoy_pct", 0) or 0) > 10 and (c.get("net_income_yoy_pct", 0) or 0) < -10,
        "label": "revenue_growth_income_decline",
        "reasoning_focus": "Revenue growth with net income decline = cost structure problem or tax/interest event.",
        "reasoning_path": "Revenue → Cost Structure → Below-the-Line → Net Income",
    },
}

REASONING_TYPES = {
    "trend_analysis": lambda m, c: c.get("revenue_yoy_pct") is not None,
    "margin_analysis": lambda m, c: m.get("gross_margin_pct") is not None and m.get("net_margin_pct") is not None,
    "risk_assessment": lambda m, c: m.get("debt_to_equity") is not None or m.get("current_ratio") is not None,
    "cash_flow_quality": lambda m, c: m.get("free_cash_flow") is not None and m.get("operating_cash_flow") is not None,
    "capital_allocation": lambda m, c: m.get("capex") is not None and m.get("share_repurchases") is not None,
    "profitability_driver": lambda m, c: c.get("gross_margin_pct_delta") is not None,
    "leverage_assessment": lambda m, c: m.get("long_term_debt") is not None and m.get("net_debt") is not None,
    "growth_quality": lambda m, c: c.get("revenue_yoy_pct") is not None and c.get("eps_yoy_pct") is not None,
    "multi_horizon": lambda m, c: False,
    "negative_inference": lambda m, c: False,
    "edge_case": lambda m, c: False,
    "market_impact": lambda m, c: False,
    "liquidity_paradox": lambda m, c: False,
    "predictive_signal": lambda m, c: False,
    "segment_analysis": lambda m, c: bool(m.get("segments")),
}

def is_8k_form(form_type: str) -> bool:
    return form_type.startswith("8-K")

# ─── CausalKeywordScanner ─────────────────────────────────────────────────────
class CausalKeywordScanner:
    KEYWORDS = [
        ("driven by", re.compile(r"\bdriven by\b", re.IGNORECASE)),
        ("offset by", re.compile(r"\boffset by\b", re.IGNORECASE)),
        ("primarily due to", re.compile(r"\bprimarily due to\b", re.IGNORECASE)),
        ("resulted from", re.compile(r"\bresulted from\b", re.IGNORECASE)),
        ("due to", re.compile(r"\bdue to\b", re.IGNORECASE)),
        ("attributable to", re.compile(r"\battributable to\b", re.IGNORECASE)),
        ("stemming from", re.compile(r"\bstemming from\b", re.IGNORECASE)),
        ("resulting in", re.compile(r"\bresulting in\b", re.IGNORECASE)),
        ("because", re.compile(r"\bbecause\b", re.IGNORECASE)),
        ("consequently", re.compile(r"\bconsequently\b", re.IGNORECASE)),

        ("owing to", re.compile(r"\bowings to\b", re.IGNORECASE)),
        ("which led to", re.compile(r"\bwhich led to\b", re.IGNORECASE)),
        ("benefiting from", re.compile(r"\bbenefiting from\b", re.IGNORECASE)),
        ("weighed by", re.compile(r"\bweighed by\b", re.IGNORECASE)),
        ("impacted by", re.compile(r"\bimpacted by\b", re.IGNORECASE)),
        ("partially offset", re.compile(r"\bpartially offset\b", re.IGNORECASE)),
    ]

    @classmethod
    def scan(cls, text: str) -> dict:
        if not text:
            return {"total": 0}
        hits = {}
        for name, pat in cls.KEYWORDS:
            c = len(pat.findall(text))
            if c:
                hits[name] = c
        hits["total"] = sum(hits.values())
        return hits

    @classmethod
    def count(cls, text: str) -> int:
        return cls.scan(text).get("total", 0)

def count_causal_connectors_v20(text: str) -> int:
    return CausalKeywordScanner.count(text)

# ─── ScalingNormaliser ────────────────────────────────────────────────────────
class ScalingNormaliser:
    _THOUSANDS_RE = re.compile(r"in\s+(thousands|000s)\b", re.IGNORECASE)
    _MILLIONS_RE = re.compile(r"in\s+millions?\b", re.IGNORECASE)
    _BILLIONS_RE = re.compile(r"in\s+billions?\b", re.IGNORECASE)

    @classmethod
    def detect(cls, fs_text: str) -> str:
        if not fs_text:
            return "unknown"
        sample = fs_text[:2000]
        if cls._THOUSANDS_RE.search(sample):
            return "thousands"
        if cls._MILLIONS_RE.search(sample):
            return "millions"
        if cls._BILLIONS_RE.search(sample):
            return "billions"
        return "millions"

    @classmethod
    def scale_to_absolute(cls, value: float, unit: str) -> float:
        if unit == "thousands":
            return value * 1_000
        if unit == "millions":
            return value * 1_000_000
        if unit == "billions":
            return value * 1_000_000_000
        return value

# ─── CrossRefGuard ────────────────────────────────────────────────────────────

class CrossRefGuard:
    _REV_TABLE_RE = re.compile(
        r"(?:net\s+revenues?|total\s+revenues?|net\s+sales)[\s\$]*?([\d,]+(?:\.\d+)?)",
        re.IGNORECASE)
    _NI_TABLE_RE = re.compile(
        r"net\s+(?:income|earnings|loss)\s+(?:attributable[^$]{0,40})?[\s\$]*?([\d,]+(?:\.\d+)?)",
        re.IGNORECASE)

    @classmethod
    def _delta_pct(cls, a: float, b: float) -> float:
        if b == 0:
            return 0.0
        return abs(a - b) / abs(b) * 100

    @classmethod
    def validate(cls, fs_text: str, mda_text: str, xbrl_rev: Optional[float],
                 xbrl_ni: Optional[float], scaling: str = "millions") -> dict:
        result = {
            "revenue_validated":       xbrl_rev,
            "revenue_source":          "xbrl",
            "revenue_discrepancy_pct": 0.0,
            "net_income_validated":    xbrl_ni,
            "net_income_source":       "xbrl",
            "net_income_discrepancy_pct": 0.0,
            "cross_ref_clean":         True,
        }
        scale = ScalingNormaliser.scale_to_absolute(1.0, scaling)

        rev_m = cls._REV_TABLE_RE.search(fs_text or "")
        ni_m  = cls._NI_TABLE_RE.search(fs_text or "")

        # ── Revenue cross-check ───────────────────────────────────────────────
        if rev_m and xbrl_rev and xbrl_rev > 0:
            try:
                tv = float(rev_m.group(1).replace(",", "")) * scale
                # CRITICAL: only override XBRL if text value is within 10× range.
                # Prevents "in thousands" mis-scaling from replacing $391B with $391M.
                if tv > 0 and 0.1 <= (tv / xbrl_rev) <= 10.0:
                    dp = cls._delta_pct(tv, xbrl_rev)
                    result["revenue_discrepancy_pct"] = round(dp, 2)
                    if dp > CROSS_REF_DELTA_PCT:
                        result.update({
                            "revenue_validated": tv,
                            "revenue_source":    "fs_table_override",
                            "cross_ref_clean":   False,
                        })
                else:
                    logger.debug(
                        f"  CrossRef: text_rev={tv:.0f} is >{10}× away from "
                        f"xbrl_rev={xbrl_rev:.0f} — XBRL value kept"
                    )
            except Exception:
                pass

        # ── Net income cross-check ────────────────────────────────────────────
        if ni_m and xbrl_ni is not None:
            try:
                tv = float(ni_m.group(1).replace(",", "")) * scale
                ref = abs(xbrl_ni) if xbrl_ni != 0 else 1.0
                # Allow signed net income but reject order-of-magnitude mismatches
                if abs(tv) > 0 and abs(tv) / ref <= 10.0:
                    dp = cls._delta_pct(tv, xbrl_ni)
                    result["net_income_discrepancy_pct"] = round(dp, 2)
                    if dp > CROSS_REF_DELTA_PCT:
                        result.update({
                            "net_income_validated": tv,
                            "net_income_source":    "fs_table_override",
                            "cross_ref_clean":      False,
                        })
            except Exception:
                pass

        return result

# ─── SegmentExtractor ────────────────────────────────────────────────────────
class SegmentExtractor:
    NOTE_PATTERNS = [
        re.compile(r"note\s+\d+.*?segment", re.IGNORECASE),
        re.compile(r"segment\s+(information|reporting|data|results)", re.IGNORECASE),
        re.compile(r"operating\s+segment", re.IGNORECASE),
        re.compile(r"reportable\s+segment", re.IGNORECASE),
        re.compile(r"business\s+segment", re.IGNORECASE),
    ]

    AMOUNT_RE = re.compile(
        r"([A-Z][A-Za-z &\-]+?)\s{1,40}"
        r"(?:revenues?|net\s+revenues?|sales)?"
        r"\s{0,20}\$?\s*([\d,\.]+)\s*(billion|million|thousand)?",
        re.IGNORECASE)

    OP_INCOME_RE = re.compile(
        r"([A-Z][A-Za-z &\-]+?)\s{1,40}"
        r"(?:operating\s+income|segment\s+(?:profit|income|earnings))"
        r"\s{0,20}\$?\s*([\d,\.]+)\s*(billion|million|thousand)?",
        re.IGNORECASE)


    EXCLUDE_NAMES = {
        "total", "consolidated", "corporate", "other", "eliminations",
        "unallocated", "intersegment", "all other", "reconciling",
        # add these:
        "form", "and", "net", "for", "note", "includes", "item",
        "as of", "within", "there was", "the company",
    }


    @classmethod
    def _parse_amount(cls, val_str: str, unit_str: Optional[str]) -> Optional[float]:
        try:
            val = float(val_str.replace(",", ""))
            unit = (unit_str or "").lower()
            if "billion" in unit:
                return val * 1e9
            if "million" in unit:
                return val * 1e6
            if "thousand" in unit:
                return val * 1e3
            return val * 1e6 if val < 10_000 else val
        except ValueError:
            return None

    @classmethod
    def _find_segment_section(cls, text: str) -> str:
        lower = text.lower()
        best_pos, best_score = -1, -1
        for pat in cls.NOTE_PATTERNS:
            for m in pat.finditer(lower):
                window = text[m.start(): m.start() + 500]
                score = sum(1 for kw in ["revenue", "income", "profit", "segment"] if kw in window.lower())
                if score > best_score:
                    best_score = score
                    best_pos = m.start()
        if best_pos < 0:
            return ""
        return text[best_pos: best_pos + 4000]

    @classmethod
    def extract(cls, plain_text: str) -> list:
        section = cls._find_segment_section(plain_text)
        if not section:
            return []

        segments: dict = {}
        for m in cls.AMOUNT_RE.finditer(section):
            name = m.group(1).strip().rstrip("–—-").strip()
            if name.lower() in cls.EXCLUDE_NAMES or not (8 <= len(name) <= 60):
                continue
            val = cls._parse_amount(m.group(2), m.group(3))
            if val and val > 0:
                segments.setdefault(name, {"name": name, "revenue": None, "operating_income": None})
                if segments[name]["revenue"] is None:
                    segments[name]["revenue"] = val

        for m in cls.OP_INCOME_RE.finditer(section):
            name = m.group(1).strip().rstrip("–—-").strip()
            if name.lower() in cls.EXCLUDE_NAMES or not (8 <= len(name) <= 60):
                continue
            val = cls._parse_amount(m.group(2), m.group(3))
            if val is not None:
                segments.setdefault(name, {"name": name, "revenue": None, "operating_income": None})
                segments[name]["operating_income"] = val

        valid = [s for s in segments.values() if s.get("revenue") is not None]
        if len(valid) < 2:
            return []
        return valid[:8]

# ─── DerivedMetricEngine ─────────────────────────────────────────────────────
class DerivedMetricEngine:
    @classmethod
    def enrich(cls, metrics: dict, changes: dict, signals: dict) -> dict:
        m = dict(metrics)

        if m.get("ebitda") is None:
            oi = m.get("operating_income")
            da = m.get("depreciation_amortization")
            if oi is not None and da is not None:
                m["ebitda"] = round(oi + abs(da), 4)

        if m.get("free_cash_flow") is None:
            ocf = m.get("operating_cash_flow")
            cap = m.get("capex")
            if ocf is not None and cap is not None:
                m["free_cash_flow"] = round(ocf - abs(cap), 4)

        rev = m.get("revenue")
        gp = m.get("gross_profit")
        ni = m.get("net_income")
        oi2 = m.get("operating_income")

        if rev and rev > 0:
            if gp is not None and m.get("gross_margin_pct") is None:
                m["gross_margin_pct"] = round(gp / rev * 100, 2)
            if ni is not None and m.get("net_margin_pct") is None:
                m["net_margin_pct"] = round(ni / rev * 100, 2)
            if oi2 is not None and m.get("operating_margin_pct") is None:
                m["operating_margin_pct"] = round(oi2 / rev * 100, 2)

            cor = m.get("cost_of_revenue")
            sga = m.get("selling_general_admin")
            rnd = m.get("research_development")
            if cor is not None:
                m["cogs_ratio_pct"] = round(cor / rev * 100, 2)
            if sga is not None:
                m["sga_ratio_pct"] = round(sga / rev * 100, 2)
            if rnd is not None:
                m["rd_ratio_pct"] = round(rnd / rev * 100, 2)

        dbt = m.get("long_term_debt")
        csh = m.get("cash_and_equivalents")
        if dbt is not None and csh is not None and m.get("net_debt") is None:
            m["net_debt"] = round(dbt - csh, 4)

        eq = m.get("total_equity")
        if dbt is not None and eq and eq > 0 and m.get("debt_to_equity") is None:
            m["debt_to_equity"] = round(dbt / eq, 3)

        ca = m.get("current_assets")
        cl = m.get("current_liabilities")
        if ca and cl and cl > 0 and m.get("current_ratio") is None:
            m["current_ratio"] = round(ca / cl, 3)

        ta = m.get("total_assets")
        if ta and ta > 0 and ni is not None and m.get("return_on_assets_pct") is None:
            m["return_on_assets_pct"] = round(ni / ta * 100, 2)

        if eq and eq > 0 and ni is not None and m.get("return_on_equity_pct") is None:
            m["return_on_equity_pct"] = round(ni / eq * 100, 2)

        fcf_v = m.get("free_cash_flow")
        ni_v = m.get("net_income")
        if ni_v and ni_v > 0 and fcf_v is not None and m.get("fcf_conversion_ratio") is None:
            m["fcf_conversion_ratio"] = round(fcf_v / ni_v, 3)

        return m

    @classmethod
    def count_numerical_points(cls, metrics: dict, changes: dict) -> int:
        count = 0
        for v in list(metrics.values()) + list(changes.values()):
            if v is not None and isinstance(v, (int, float)) and not isinstance(v, bool):
                count += 1
        return count

    @classmethod
    def passes_gate(cls, metrics: dict, changes: dict) -> tuple:
        n = cls.count_numerical_points(metrics, changes)
        return n >= MIN_NUMERICAL_GATE, n

# ─── RegexTextSearch ─────────────────────────────────────────────────────────
class RegexTextSearch:
    PATTERNS = [
        ("revenue", r"(?:net\s+(?:revenues?|sales)|total\s+revenues?)\s+(?:were|of|was|totaled?)\s+\$?([\d,\.]+)\s*(billion|million|thousand)?", 1),
        ("gross_profit", r"gross\s+profit\s+(?:was|of|were|totaled?)\s+\$?([\d,\.]+)\s*(billion|million|thousand)?", 1),
        ("operating_income", r"(?:operating\s+income|income\s+from\s+operations)\s+(?:was|of|were|totaled?)\s+\$?([\d,\.]+)\s*(billion|million|thousand)?", 1),
        ("net_income", r"net\s+(?:income|earnings|loss)\s+(?:was|of|were|totaled?|attributable[^$]*?)\s+\$?([\d,\.]+)\s*(billion|million|thousand)?", 1),
        ("eps_diluted", r"(?:diluted\s+)?(?:earnings|loss)\s+per\s+(?:diluted\s+)?share\s+(?:was|of|were)\s+\$?([\d,\.]+)", 1),
        ("capex", r"capital\s+expenditures?\s+(?:were|of|was|totaled?)\s+\$?([\d,\.]+)\s*(billion|million|thousand)?", 1),
        ("free_cash_flow", r"free\s+cash\s+flow\s+(?:was|of|were|totaled?)\s+\$?([\d,\.]+)\s*(billion|million|thousand)?", 1),
        ("ebitda", r"ebitda\s+(?:was|of|were|totaled?)\s+\$?([\d,\.]+)\s*(billion|million|thousand)?", 1),
        ("cash_and_equivalents", r"cash\s+(?:and\s+cash\s+equivalents?|equivalents?)\s+(?:was|of|were|totaled?|at\s+end)\s+\$?([\d,\.]+)\s*(billion|million|thousand)?", 1),
        ("long_term_debt", r"long.?term\s+debt\s+(?:was|of|were|totaled?)\s+\$?([\d,\.]+)\s*(billion|million|thousand)?", 1),
    ]
    _compiled = [(k, re.compile(p, re.IGNORECASE), g) for k, p, g in PATTERNS]

    YOY_PATTERNS = [
        ("revenue_yoy_pct", r"revenues?\s+(?:increased|decreased|grew|declined)\s+([\d\.]+)\s*(?:percent|%)", 1),
        ("gross_margin_pct_delta", r"gross\s+margin\s+(?:expanded|compressed|improved|declined|increased|decreased)\s+([\d\.]+)\s*(?:percentage points?|pp|basis)", 1),
        ("net_income_yoy_pct", r"net\s+(?:income|earnings)\s+(?:increased|decreased|grew|declined)\s+([\d\.]+)\s*(?:percent|%)", 1),
        ("operating_income_yoy_pct", r"(?:operating\s+income|income\s+from\s+operations)\s+(?:increased|decreased|grew|declined)\s+([\d\.]+)\s*(?:percent|%)", 1),
        ("eps_yoy_pct", r"(?:diluted\s+)?(?:earnings|loss)\s+per\s+share\s+(?:increased|decreased|grew|declined)\s+([\d\.]+)\s*(?:percent|%)", 1),
    ]
    _yoy_compiled = [(k, re.compile(p, re.IGNORECASE), g) for k, p, g in YOY_PATTERNS]

    @classmethod
    def _parse_amount(cls, val_str: str, unit_str: Optional[str]) -> Optional[float]:
        try:
            val = float(val_str.replace(",", ""))
            unit = (unit_str or "").lower()
            if "billion" in unit:
                return val * 1e9
            if "million" in unit:
                return val * 1e6
            if "thousand" in unit:
                return val * 1e3
            return val * 1e6 if val < 10_000 else val
        except ValueError:
            return None

    @classmethod
    def extract(cls, text: str, existing_metrics: dict) -> tuple:
        if not text:
            return existing_metrics, []
        enriched = dict(existing_metrics)
        text_keys = []

        for key, pattern, grp in cls._compiled:
            if enriched.get(key) is not None:
                continue
            m = pattern.search(text)
            if m:
                groups = m.groups()
                val_str = groups[grp - 1] if len(groups) >= grp else None
                unit_str = groups[grp] if len(groups) > grp else None
                val = cls._parse_amount(val_str or "", unit_str)
                if val is not None:
                    enriched[key] = val
                    text_keys.append(key)

        for key, pattern, grp in cls._yoy_compiled:
            if enriched.get(key) is not None:
                continue
            m = pattern.search(text)
            if m:
                try:
                    pct = float(m.group(grp))
                    if any(w in m.group(0).lower() for w in ("decreased", "declined", "dropped")):
                        pct = -pct
                    enriched[key] = round(pct, 2)
                    text_keys.append(key)
                except (ValueError, IndexError):
                    pass

        return enriched, text_keys


# ----------------- New _validate_metrics_for_llm — gate before any LLM call

def _validate_metrics_for_llm(
    metrics: dict,
    changes: dict,
    ticker: str,
    period: str,
) -> tuple:
    """
    Final sanity gate before sending data to the LLM.
    Returns (is_valid: bool, errors: list[str]).
    A record with is_valid=False must NOT be sent to the LLM.
    """
    errors = []

    rev = metrics.get("revenue")
    if rev is not None and rev <= 0:
        errors.append(f"Revenue={rev:.0f} is non-positive (scaling/decumulation artifact)")

    gm = metrics.get("gross_margin_pct")
    if gm is not None and (gm < -100.0 or gm > 100.0):
        errors.append(f"Gross margin={gm:.1f}% is outside [-100, 100] — impossible")

    nm = metrics.get("net_margin_pct")
    if nm is not None and nm < -500.0:
        errors.append(f"Net margin={nm:.1f}% is extreme — likely artifact")

    om = metrics.get("operating_margin_pct")
    if om is not None and (om < -500.0 or om > 200.0):
        errors.append(f"Operating margin={om:.1f}% is extreme")

    # Catch the cross-ref override gone wrong: revenue changed sign
    rev_yoy = changes.get("revenue_yoy_pct")
    if rev_yoy is not None and abs(rev_yoy) > 200:
        errors.append(f"Revenue YoY={rev_yoy:+.1f}% exceeds ±200% — artifact")

    if errors:
        logger.warning(
            f"  ✗ DATA_VALIDITY [{ticker} {period}]: {len(errors)} error(s) — "
            f"{'; '.join(errors)}"
        )

    return len(errors) == 0, errors

# ─── AdaptiveLimiter ─────────────────────────────────────────────────────────
class AdaptiveLimiter:
    def __init__(self, rpm: int = NVIDIA_RPM_LIMIT):
        self._base_rpm = rpm
        self._current_rpm = rpm
        self._lock = threading.Lock()
        self._next_slot = 0.0
        self._call_count = 0
        self._throttled = 0
        self._consecutive_ok = 0
        self._backoff_coeff = 1.0
        self._current_workers = PARALLEL_WORKERS

    @property
    def _interval(self) -> float:
        return 60.0 / max(self._current_rpm, 1) * self._backoff_coeff

    def acquire(self):
        with self._lock:
            now = time.monotonic()
            slot = max(now, self._next_slot)
            self._next_slot = slot + self._interval
            sleep_time = slot - now
            if sleep_time > 0:
                self._throttled += 1
            self._call_count += 1
            if sleep_time > 0:
                time.sleep(sleep_time)

    def record_success(self):
        with self._lock:
            self._consecutive_ok += 1
            if self._consecutive_ok >= ADAPTIVE_STEP_UP_AFTER_N:
                self._step_up()

    def record_429(self, wait_seconds: float):
        with self._lock:
            self._consecutive_ok = 0
            if ADAPTIVE_STEP_DOWN_ON_429:
                self._step_down()
            self._backoff_coeff = min(self._backoff_coeff * 1.5, 8.0)

    def _step_down(self):
        if self._current_workers > 1:
            self._current_workers -= 1
        if self._current_rpm > 10:
            self._current_rpm = max(10, int(self._current_rpm * 0.75))

    def _step_up(self):
        self._consecutive_ok = 0
        if self._current_rpm < self._base_rpm:
            self._current_rpm = min(self._base_rpm, int(self._current_rpm * 1.25))
        if self._current_workers < PARALLEL_WORKERS:
            self._current_workers += 1
        self._backoff_coeff = max(1.0, self._backoff_coeff * 0.9)

    def stats(self) -> str:
        return (f"llm_calls={self._call_count}, throttled={self._throttled}, "
                f"rpm={self._current_rpm}/{self._base_rpm}, "
                f"backoff={self._backoff_coeff:.2f}, workers={self._current_workers}")

_llm_rate_limiter = AdaptiveLimiter()

def get_llm(temperature: float = BASE_TEMPERATURE, max_tokens: int = BASE_MAX_TOKENS):
    from langchain_nvidia_ai_endpoints import ChatNVIDIA
    api_key = os.getenv("NVIDIA_API_KEY_LLAMA_70_INSTRUCT")
    if not api_key:
        raise EnvironmentError("NVIDIA_API_KEY_LLAMA_70_INSTRUCT not found in .env")
    return ChatNVIDIA(
        model=NVIDIA_MODEL,
        api_key=api_key,
        temperature=temperature,
        top_p=1,
        max_completion_tokens=max_tokens,
    )

def _is_queue_tail() -> bool:
    with _queue_lock:
        total = _queue_state["total"]
        processed = _queue_state["processed"]
        if total <= 0:
            return False
        return processed >= total * (1.0 - QUEUE_TAIL_PCT)

def _increment_queue_processed():
    with _queue_lock:
        _queue_state["processed"] += 1

def get_llm_for_context(temperature_override: Optional[float] = None, strict: bool = False):
    if temperature_override is not None:
        return get_llm(temperature=temperature_override, max_tokens=BASE_MAX_TOKENS)
    if strict:
        return get_llm(temperature=0.1, max_tokens=BASE_MAX_TOKENS)
    if _is_queue_tail():
        return get_llm(temperature=TAIL_TEMPERATURE, max_tokens=TAIL_MAX_TOKENS)
    return get_llm(temperature=BASE_TEMPERATURE, max_tokens=BASE_MAX_TOKENS)

# ─── SessionManager ──────────────────────────────────────────────────────────
class SessionManager:
    def __init__(self):
        self._request_count = 0
        self._company_count = 0
        self._ua_index = 0
        self._lock = threading.Lock()

    def record_request(self):
        with self._lock:
            self._request_count += 1
            if self._request_count % UA_ROTATE_EVERY == 0:
                self._ua_index = (self._ua_index + 1) % len(_USER_AGENTS)

    def record_company_done(self) -> bool:
        with self._lock:
            self._company_count += 1
            return self._company_count % CONN_RESET_EVERY == 0

    def current_ua(self) -> str:
        return _USER_AGENTS[self._ua_index]

    def stats(self) -> str:
        return (f"requests={self._request_count}, companies={self._company_count}, "
                f"ua_idx={self._ua_index}")

_session_manager = SessionManager()

# ─── Utility functions ───────────────────────────────────────────────────────
def normalize_financial_terms(text: str) -> str:
    for non_canonical, canonical in CANONICAL_TERMS.items():
        text = re.sub(re.escape(non_canonical), canonical, text, flags=re.IGNORECASE)
    return text

def _usd(v) -> str:
    if v is None:
        return "N/A"
    if abs(v) >= 1e12:
        return f"${v/1e12:.2f}T"
    if abs(v) >= 1e9:
        return f"${v/1e9:.1f}B"
    if abs(v) >= 1e6:
        return f"${v/1e6:.0f}M"
    return f"${v:,.0f}"

def _pct(v, sfx: str = "%") -> str:
    if v is None:
        return "N/A"
    try:
        return f"{float(v):+.1f}{sfx}"
    except:
        return "N/A"

def _pct_plain(v) -> str:
    if v is None:
        return "N/A"
    try:
        return f"{float(v):.1f}%"
    except:
        return "N/A"

def _table_noise_ratio(text: str) -> float:
    if not text:
        return 0.0
    num_chars = sum(1 for c in text if c.isdigit() or c in ",$|%-.()[]")
    alpha_chars = sum(1 for c in text if c.isalpha())
    total = num_chars + alpha_chars
    return num_chars / total if total > 0 else 0.0

def enrich_thin_mda(text_sections: dict, plain_text: str) -> dict:
    mda = text_sections.get("mda", "") or ""
    if len(mda) >= MIN_MDA_CHARS_FOR_RICH:
        return text_sections
    enriched = mda
    risk = text_sections.get("risk_factors", "") or ""
    if len(risk) > MIN_SECTION_CHARS:
        enriched += "\n\n[RISK FACTORS SUPPLEMENT]\n" + risk[:2000]
    if len(enriched) > len(mda):
        text_sections = dict(text_sections)
        text_sections["mda"] = enriched[:MAX_TEXT_CHARS]
    return text_sections

# ─── EDGAR SESSION (still needed for XBRL) ───────────────────────────────────
class EDGARSession:
    ARCHIVES = "https://www.sec.gov/Archives/edgar/data"
    DATA_BASE = "https://data.sec.gov"
    WWW_BASE = "https://www.sec.gov"

    def __init__(self):
        self.session = self._make_session()

    def _make_session(self) -> requests.Session:
        sess = requests.Session()
        retry = Retry(total=MAX_RETRIES, backoff_factor=BACKOFF_BASE,
                      status_forcelist=[429, 500, 502, 503, 504],
                      allowed_methods=["GET", "HEAD"], raise_on_status=False)
        adapter = HTTPAdapter(max_retries=retry)
        sess.mount("https://", adapter)
        sess.mount("http://", adapter)
        sess.headers.update({**EDGAR_HEADERS, "User-Agent": _session_manager.current_ua()})
        return sess

    def reset_connection(self):
        try:
            self.session.close()
        except Exception:
            pass
        self.session = self._make_session()

    def get(self, url: str, timeout: int = 30, is_html: bool = False) -> Optional[requests.Response]:
        time.sleep(EDGAR_SLEEP)
        _session_manager.record_request()
        self.session.headers.update({"User-Agent": _session_manager.current_ua()})
        self.session.headers.update({"Accept": "text/html,*/*;q=0.8"} if is_html else
                                     {"Accept": "application/json, text/html, */*"})

        for attempt in range(MAX_RETRIES):
            try:
                r = self.session.get(url, timeout=timeout)
                if r.status_code == 429:
                    wait = BACKOFF_BASE ** (attempt + 2) * (0.8 + random.random() * 0.4)
                    time.sleep(min(wait, BACKOFF_MAX))
                    continue
                if r.status_code in (403, 404):
                    return None
                r.raise_for_status()
                return r
            except requests.exceptions.Timeout:
                time.sleep(min(BACKOFF_BASE ** (attempt + 1), BACKOFF_MAX))
            except Exception as e:
                logger.error(f"GET: {e} | {url[-60:]}")
                return None
        return None

# ─── CIK LOOKUP (still needed) ───────────────────────────────────────────────
_cik_cache: dict = {}
_tickers_data: Optional[dict] = None

TITAN_CIKS = {
    "BRK.B": "0001067983", "BRK.A": "0001067983", "AAPL": "0000320193",
    "MSFT": "0000789019", "NVDA": "0001045810", "AVGO": "0001730168",
    "ORCL": "0001341439", "ADBE": "0000796343", "CRM": "0001108524",
    "AMD": "0000002488", "INTC": "0000050863", "QCOM": "0000804328",
    "TXN": "0000097476", "IBM": "0000051143", "CSCO": "0000858877",
    "NOW": "0001373715", "PANW": "0001327567", "INTU": "0000896878",
    "SNOW": "0001640147", "CRWD": "0001535527", "DELL": "0001571123",
    "HPQ": "0000047217", "GOOGL": "0001652044", "GOOG": "0001652044",
    "META": "0001326801", "AMZN": "0001018724", "TSLA": "0001318605",
    "NFLX": "0001065280", "JPM": "0000019617", "BAC": "0000070858",
    "WFC": "0000072971", "GS": "0000886982", "MS": "0000895421",
    "V": "0001403161", "MA": "0001141391", "WMT": "0000104169",
    "COST": "0000909832", "HD": "0000354950", "LLY": "0000059478",
    "JNJ": "0000200406", "UNH": "0000731766", "MRK": "0000310158",
    "ABBV": "0001551152", "PFE": "0000078003", "XOM": "0000034088",
    "CVX": "0000093410", "GE": "0000040533", "CAT": "0000018230",
    "BA": "0000012927", "UPS": "0001090727", "DIS": "0001001039",
    "T": "0000732717", "VZ": "0000732712", "KO": "0000021344",
    "PEP": "0000077476", "PG": "0000080424", "NEE": "0000753308",
    "DUK": "0001326428", "SO": "0000092122", "ADSK": "0000769397",
}

def _ticker_variants(ticker: str) -> list:
    variants = [ticker.upper()]
    t = ticker.upper()
    if "." in t:
        base, cls_ = t.rsplit(".", 1)
        variants += [f"{base}-{cls_}", f"{base}/{cls_}", f"{base}{cls_}"]
    elif "-" in t:
        base, cls_ = t.rsplit("-", 1)
        variants += [f"{base}.{cls_}", f"{base}/{cls_}", f"{base}{cls_}"]
    return list(dict.fromkeys(variants))

def ticker_to_cik(ticker: str, sess: EDGARSession, company_name: Optional[str] = None) -> Optional[str]:
    global _tickers_data

    titan_key = ticker.upper()
    if titan_key in TITAN_CIKS:
        cik = str(TITAN_CIKS[titan_key]).zfill(10)
        with _cik_lock:
            _cik_cache[ticker] = cik
        return cik

    with _cik_lock:
        if ticker in _cik_cache:
            return _cik_cache[ticker]

    if _tickers_data is None:
        r = sess.get(f"{sess.WWW_BASE}/files/company_tickers.json")
        _tickers_data = r.json() if r else {}

    variants = _ticker_variants(ticker)
    for variant in variants:
        for entry in (_tickers_data or {}).values():
            if entry.get("ticker", "").upper() == variant:
                found_cik = str(entry["cik_str"]).zfill(10)
                with _cik_lock:
                    _cik_cache[ticker] = found_cik
                return found_cik

    logger.warning(f"No CIK for {ticker}")
    return None

# ─── XBRL METRICS (unchanged from v20) ──────────────────────────────────────
FIELD_MAP = {
    "revenue": ["RevenueFromContractWithCustomerExcludingAssessedTax", "NetSales", "RevenueFromContractWithCustomerIncludingAssessedTax", "Revenues", "SalesRevenueNet"],
    "cost_of_revenue": ["CostOfGoodsAndServicesSold", "CostOfRevenue", "CostOfGoodsSold"],
    "gross_profit": ["GrossProfit"],
    "operating_expenses": ["OperatingExpenses"],
    "research_development": ["ResearchAndDevelopmentExpense"],
    "selling_general_admin": ["SellingGeneralAndAdministrativeExpense"],
    "operating_income": ["OperatingIncomeLoss"],
    "interest_expense": ["InterestExpense", "InterestAndDebtExpense"],
    "pretax_income": ["IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest"],
    "income_tax": ["IncomeTaxExpenseBenefit"],
    "net_income": ["NetIncomeLoss"],
    "eps_basic": ["EarningsPerShareBasic"],
    "eps_diluted": ["EarningsPerShareDiluted"],
    "shares_diluted": ["WeightedAverageNumberOfDilutedSharesOutstanding"],
    "total_assets": ["Assets"],
    "current_assets": ["AssetsCurrent"],
    "total_liabilities": ["Liabilities"],
    "current_liabilities": ["LiabilitiesCurrent"],
    "total_equity": ["StockholdersEquity", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"],
    "cash_and_equivalents": ["CashAndCashEquivalentsAtCarryingValue", "CashCashEquivalentsAndShortTermInvestments", "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents"],
    "short_term_investments": ["ShortTermInvestments", "MarketableSecuritiesCurrent"],
    "accounts_receivable": ["AccountsReceivableNetCurrent"],
    "inventory": ["InventoryNet"],
    "goodwill": ["Goodwill"],
    "long_term_debt": ["LongTermDebt", "LongTermDebtNoncurrent"],
    "retained_earnings": ["RetainedEarningsAccumulatedDeficit"],
    "operating_cash_flow": ["NetCashProvidedByUsedInOperatingActivities"],
    "investing_cash_flow": ["NetCashProvidedByUsedInInvestingActivities"],
    "financing_cash_flow": ["NetCashProvidedByUsedInFinancingActivities"],
    "capex": ["PaymentsToAcquirePropertyPlantAndEquipment"],
    "depreciation_amortization": ["DepreciationDepletionAndAmortization", "DepreciationAndAmortization"],
    "stock_based_compensation": ["ShareBasedCompensation", "AllocatedShareBasedCompensationExpense"],
    "dividends_paid": ["PaymentsOfDividends", "PaymentsOfDividendsCommonStock"],
    "share_repurchases": ["PaymentsForRepurchaseOfCommonStock"],
}

DEI_MAP = {
    "employees": ["EntityNumberOfEmployees"],
    "entity_public_float": ["EntityPublicFloat"],
}

def load_xbrl_facts(cik: str, sess: EDGARSession) -> dict:
    r = sess.get(f"{sess.DATA_BASE}/api/xbrl/companyfacts/CIK{cik}.json", timeout=30)
    if not r:
        return {}
    facts = r.json().get("facts", {})
    gaap = facts.get("us-gaap", {})
    dei = facts.get("dei", {})
    logger.info(f"  XBRL: {len(gaap)} GAAP + {len(dei)} DEI")
    return {"us-gaap": gaap, "dei": dei}

def _is_consolidated(e: dict) -> bool:
    return e.get("segment") is None

def xbrl_val(ns: dict, concepts: list, period_end: str, base_form: str) -> Optional[float]:
    try:
        td = datetime.strptime(period_end, "%Y-%m-%d")
        window = {(td + timedelta(days=d)).strftime("%Y-%m-%d") for d in range(-PERIOD_FUZZ_DAYS, PERIOD_FUZZ_DAYS + 1)}
    except Exception:
        window = {period_end}

    for concept in concepts:
        for unit in ("USD", "USD/shares", "shares", "pure"):
            matches = [e for e in ns.get(concept, {}).get("units", {}).get(unit, [])
                       if e.get("end") in window and e.get("form", "").startswith(base_form)
                       and e.get("val") is not None and _is_consolidated(e)]
            if matches:
                matches.sort(key=lambda e: e.get("filed", ""), reverse=True)
                try:
                    return float(matches[0]["val"])
                except:
                    pass
    return None

def extract_metrics(xbrl: dict, period_end: str, form_type: str) -> dict:
    gaap = xbrl.get("us-gaap", {})
    dei = xbrl.get("dei", {})
    bf = form_type.split("/")[0]

    m = {}
    for field, concepts in FIELD_MAP.items():
        m[field] = xbrl_val(gaap, concepts, period_end, bf)
    for field, concepts in DEI_MAP.items():
        m[field] = xbrl_val(dei, concepts, period_end, bf)

    ocf = m.get("operating_cash_flow")
    cap = m.get("capex")
    if ocf is not None and cap is not None:
        m["free_cash_flow"] = ocf - abs(cap)

    oi = m.get("operating_income")
    da = m.get("depreciation_amortization")
    if oi is not None and da is not None:
        m["ebitda"] = oi + abs(da)

    return {k: (round(v, 4) if isinstance(v, float) and abs(v) > 0.01 else v) for k, v in m.items() if v is not None}

def force_fetch_xbrl_tag(xbrl: dict, field: str, period_end: str, form_type: str) -> Optional[float]:
    gaap = xbrl.get("us-gaap", {})
    concepts = FIELD_MAP.get(field, [])
    if not concepts:
        return None

    try:
        td = datetime.strptime(period_end, "%Y-%m-%d")
        window = {(td + timedelta(days=d)).strftime("%Y-%m-%d") for d in range(-15, 16)}
    except Exception:
        window = {period_end}

    for concept in concepts:
        for unit in ("USD", "USD/shares", "shares", "pure"):
            matches = [e for e in gaap.get(concept, {}).get("units", {}).get(unit, [])
                       if e.get("end") in window and e.get("val") is not None]
            if matches:
                matches.sort(key=lambda e: e.get("filed", ""), reverse=True)
                try:
                    return float(matches[0]["val"])
                except (TypeError, ValueError):
                    pass
    return None

def force_fetch_all_xbrl_fields(xbrl: dict, period_end: str, form_type: str, existing_metrics: dict) -> dict:
    gaap = xbrl.get("us-gaap", {})

    try:
        td = datetime.strptime(period_end, "%Y-%m-%d")
        window = {(td + timedelta(days=d)).strftime("%Y-%m-%d") for d in range(-20, 21)}
    except Exception:
        window = {period_end}

    new = dict(existing_metrics)
    filled = 0

    for field, concepts in FIELD_MAP.items():
        if new.get(field) is not None:
            continue
        for concept in concepts:
            for unit in ("USD", "USD/shares", "shares", "pure"):
                matches = [e for e in gaap.get(concept, {}).get("units", {}).get(unit, [])
                           if e.get("end") in window and e.get("val") is not None]
                if matches:
                    matches.sort(key=lambda e: e.get("filed", ""), reverse=True)
                    try:
                        new[field] = float(matches[0]["val"])
                        filled += 1
                        break
                    except (TypeError, ValueError):
                        pass
            if new.get(field) is not None:
                break

    if filled:
        logger.info(f"  force-fetch-all: {filled} additional XBRL fields")
    return new

def get_fiscal_quarter(period_end: str, form_type: str) -> Optional[int]:
    if not period_end:
        return None
    try:
        dt = datetime.strptime(period_end[:10], "%Y-%m-%d")
    except:
        return None
    if "10-K" in form_type:
        return 4
    m = dt.month
    if m in (1, 2, 3, 4):
        return 1
    if m in (4, 5, 6, 7):
        return 2
    if m in (7, 8, 9, 10):
        return 3
    return 4


def decumulate_10q_metrics(
    current_metrics: dict,
    prev_ytd_metrics: dict,
    fiscal_quarter: int,
    form_type: str,
) -> tuple:
    if "10-Q" not in form_type or fiscal_quarter == 1 or not prev_ytd_metrics:
        return current_metrics, False

    FLOW_METRICS = {
        "revenue", "cost_of_revenue", "gross_profit", "operating_expenses",
        "research_development", "selling_general_admin", "operating_income",
        "interest_expense", "pretax_income", "income_tax", "net_income",
        "depreciation_amortization", "stock_based_compensation",
        "operating_cash_flow", "investing_cash_flow", "financing_cash_flow",
        "capex", "dividends_paid", "share_repurchases",
    }

    # These metrics are economically impossible to be negative from decumulation
    MUST_BE_POSITIVE = {
        "revenue", "gross_profit", "cost_of_revenue", "operating_expenses",
        "research_development", "selling_general_admin", "depreciation_amortization",
    }

    discrete = dict(current_metrics)
    applied = False

    for field in FLOW_METRICS:
        cur  = current_metrics.get(field)
        prev = prev_ytd_metrics.get(field)
        if cur is None or prev is None:
            continue

        val = round(cur - prev, 4)

        # Guard: if decumulation produces an impossible negative for a
        # non-negative field, revert to the raw YTD value rather than
        # poisoning downstream metrics and CAGR calculations.
        if val < 0 and field in MUST_BE_POSITIVE:
            logger.debug(
                f"  Decumulation artifact: {field}={val:.0f} < 0 "
                f"(cur={cur:.0f}, prev={prev:.0f}) → reverted to raw YTD"
            )
            discrete[field] = cur   # keep raw cumulative value
        else:
            discrete[field] = val

        applied = True

    return discrete, applied

def extract_metrics_with_decumulation(xbrl: dict, period_end: str, form_type: str,
                                       prev_ytd_metrics: Optional[dict], fiscal_quarter: Optional[int]) -> tuple:
    raw = extract_metrics(xbrl, period_end, form_type)
    if not raw:
        return raw, False
    return decumulate_10q_metrics(raw, prev_ytd_metrics or {}, fiscal_quarter or 0, form_type)

def compute_changes(current: dict, previous: dict) -> dict:
    PAIRS = [
        ("revenue", "revenue_yoy_pct"),
        ("net_income", "net_income_yoy_pct"),
        ("gross_profit", "gross_profit_yoy_pct"),
        ("operating_income", "operating_income_yoy_pct"),
        ("ebitda", "ebitda_yoy_pct"),
        ("free_cash_flow", "fcf_yoy_pct"),
        ("operating_cash_flow", "ocf_yoy_pct"),
        ("long_term_debt", "debt_yoy_pct"),
        ("total_assets", "assets_yoy_pct"),
        ("total_equity", "equity_yoy_pct"),
        ("research_development", "rd_yoy_pct"),
        ("eps_diluted", "eps_yoy_pct"),
        ("capex", "capex_yoy_pct"),
    ]

    c = {}
    for metric, label in PAIRS:
        cur = current.get(metric)
        prev = previous.get(metric)
        if cur is not None and prev and prev != 0:
            c[label] = round(((cur - prev) / abs(prev)) * 100, 2)

    for margin in ("gross_margin_pct", "net_margin_pct", "operating_margin_pct",
                   "cogs_ratio_pct", "sga_ratio_pct", "rd_ratio_pct"):
        cur = current.get(margin)
        prev = previous.get(margin)
        if cur is not None and prev is not None:
            c[f"{margin}_delta"] = round(cur - prev, 2)

    rev = c.get("revenue_yoy_pct")
    if rev is not None:
        c["trend_revenue"] = ("hypergrowth" if rev > 50 else
                              "strong_growth" if rev > 20 else
                              "moderate_growth" if rev > 5 else
                              "slowing_growth" if rev > 0 else
                              "mild_decline" if rev > -10 else
                              "significant_decline")

    gm = c.get("gross_margin_pct_delta")
    if gm is not None:
        c["trend_margin"] = "expanding" if gm > 1 else "compressing" if gm < -1 else "stable"

    fcf = c.get("fcf_yoy_pct")
    if fcf is not None:
        c["trend_fcf"] = "strong_generation" if fcf > 20 else "improving" if fcf > 0 else "deteriorating"

    dbt = c.get("debt_yoy_pct")
    if dbt is not None:
        c["trend_leverage"] = ("rapidly_increasing" if dbt > 30 else
                               "moderately_increasing" if dbt > 10 else
                               "stable" if -10 <= dbt <= 10 else
                               "decreasing")
    return c

def compute_quarter_aligned_yoy(current_metrics: dict, current_period: str, current_quarter: int,
                                 historical_records: list, form_type: str) -> tuple:
    if not current_period or not historical_records:
        return {}, False

    try:
        cur_dt = datetime.strptime(current_period[:10], "%Y-%m-%d")
    except:
        return {}, False

    target_dt = cur_dt - timedelta(days=365)
    window_low = target_dt - timedelta(days=30)
    window_high = target_dt + timedelta(days=30)

    for rec in historical_records:
        rec_form = rec.get("meta", {}).get("form_type", "")
        rec_period = rec.get("meta", {}).get("period_of_report", "")
        rec_q = rec.get("meta", {}).get("fiscal_quarter")

        if form_type.split("/")[0] not in rec_form:
            continue
        if rec_q is not None and rec_q != current_quarter:
            continue

        try:
            rec_dt = datetime.strptime(rec_period[:10], "%Y-%m-%d")
        except:
            continue

        if window_low <= rec_dt <= window_high:
            return compute_changes(current_metrics, rec.get("metrics", {})), True

    return {}, False

def sanity_check_metrics(metrics: dict, changes: dict, ticker: str, period: str,
                          form_type: str, is_mega_cap: bool = True) -> dict:
    flags = []
    action = "pass"

    rev_yoy = changes.get("revenue_yoy_pct")
    ni_yoy = changes.get("net_income_yoy_pct")
    gm_d = changes.get("gross_margin_pct_delta")

    threshold = MEGA_CAP_SWING_THRESHOLD if is_mega_cap else 50.0

    if rev_yoy is not None and abs(rev_yoy) > threshold:
        flags.append(f"REVENUE_SWING: {rev_yoy:+.1f}%")
        action = "flag"

    if gm_d is not None and abs(gm_d) > 40:
        flags.append(f"MARGIN_DELTA_IMPOSSIBLE: Δ={gm_d:+.1f}pp")
        action = "block"

    if rev_yoy is not None and rev_yoy < -50:
        flags.append(f"REVENUE_COLLAPSE: {rev_yoy:+.1f}%")
        action = "block"

    if rev_yoy is not None and rev_yoy > 100 and is_mega_cap:
        flags.append(f"HYPERGROWTH_ARTIFACT: {rev_yoy:+.1f}%")
        action = "block"

    if rev_yoy is not None and ni_yoy is not None and abs(ni_yoy - rev_yoy) > 80:
        flags.append(f"INCOME_REVENUE_DIVERGENCE: Δ={abs(ni_yoy-rev_yoy):.1f}pp")

    if action == "pass":
        action = "flag"

    confidence = "high" if not flags else ("medium" if action == "flag" else "low")

    return {
        "action": action,
        "flags": flags,
        "confidence": confidence,
        "note": "Data validated." if not flags else f"{len(flags)} rule(s). Action: {action.upper()}.",
        "ticker": ticker,
        "period": period,
        "form_type": form_type
    }

def enrich_period_type_metadata(meta: dict, fiscal_quarter: Optional[int],
                                 decumulation_applied: bool, quarter_aligned_yoy: bool,
                                 sanity_result: dict) -> dict:
    ft = meta.get("form_type", "")

    if ft in ("10-K", "10-K/A"):
        period_type = "full_year"
    elif fiscal_quarter == 1:
        period_type = "discrete_quarter"
    elif fiscal_quarter in (2, 3):
        period_type = "discrete_quarter" if decumulation_applied else "ytd_cumulative"
    elif fiscal_quarter == 4:
        period_type = "full_year"
    else:
        period_type = "unknown"

    meta.update({
        "period_type": period_type,
        "fiscal_quarter": fiscal_quarter,
        "decumulation_applied": decumulation_applied,
        "quarter_aligned_yoy": quarter_aligned_yoy,
        "sanity_action": sanity_result.get("action", "unknown"),
        "sanity_confidence": sanity_result.get("confidence", "unknown"),
        "sanity_flags_count": len(sanity_result.get("flags", [])),
    })
    return meta

def compute_signals(m: dict, c: dict) -> dict:
    s = {}

    nm = m.get("net_margin_pct", 0)
    s["profitability_tier"] = ("exceptional" if nm > 30 else
                                "strong" if nm > 15 else
                                "average" if nm > 5 else
                                "weak" if nm > 0 else "loss_making")

    ni = m.get("net_income", 0) or 0
    fcf = m.get("free_cash_flow", 0) or 0
    if ni > 0:
        r = fcf / ni
        s["cash_conversion"] = ("excellent" if r > 1.1 else
                                "good" if r > 0.8 else
                                "moderate" if r > 0.5 else "poor")

    dr = m.get("debt_to_equity")
    cr = m.get("current_ratio")

    if dr is not None:
        s["leverage_health"] = ("minimal_debt" if dr < 0.3 else
                                "conservative" if dr < 1.0 else
                                "moderate_leverage" if dr < 2.0 else
                                "high_leverage" if dr < 4.0 else "overleveraged")

    if cr is not None:
        s["liquidity_health"] = ("very_strong" if cr > 3.0 else
                                 "strong" if cr > 2.0 else
                                 "adequate" if cr > 1.0 else "tight")

    rev_g = c.get("revenue_yoy_pct", 0) or 0
    op_m = m.get("operating_margin_pct", 0) or 0
    s["rule_of_40_score"] = round(rev_g + op_m, 1)
    s["rule_of_40_pass"] = (rev_g + op_m) >= 40

    rev = m.get("revenue", 0) or 0
    rd = m.get("research_development", 0) or 0
    if rev > 0 and rd > 0:
        rdi = rd / rev * 100
        s["rd_intensity_pct"] = round(rdi, 2)
        s["innovation_investment"] = ("heavy" if rdi > 20 else
                                      "moderate" if rdi > 10 else
                                      "light" if rdi > 3 else "minimal")

    edge_cases = []
    for name, archetype in EDGE_CASE_ARCHETYPES.items():
        try:
            if archetype["condition"](m, c):
                edge_cases.append(name)
        except Exception:
            pass
    if edge_cases:
        s["edge_cases_detected"] = edge_cases

    return s

def infer_reasoning_path(metrics: dict, changes: dict, signals: dict, edge_cases: list) -> str:
    if edge_cases and edge_cases[0] in EDGE_CASE_ARCHETYPES:
        return EDGE_CASE_ARCHETYPES[edge_cases[0]].get("reasoning_path", "")

    parts = []
    if changes.get("revenue_yoy_pct") is not None:
        parts.append("Revenue Trend")
    if changes.get("gross_margin_pct_delta") is not None:
        parts.append("Margin Structure")
    if changes.get("fcf_yoy_pct") is not None:
        parts.append("Cash Generation")
    if metrics.get("debt_to_equity") is not None:
        parts.append("Leverage")
    if metrics.get("current_ratio") is not None:
        parts.append("Liquidity")
    if metrics.get("segments"):
        parts.append("Segment Mix")

    tier = signals.get("profitability_tier", "")
    parts.append("Investment Quality" if tier == "exceptional" else
                 "Viability Risk" if tier == "loss_making" else "Risk-Return Assessment")

    return " → ".join(parts) if parts else "Quantitative Analysis → Signal Assessment"

def build_negative_examples(metrics: dict, changes: dict, text_sections: dict) -> list:
    mda_text = text_sections.get("mda", "") or ""
    mda_has_text = len(mda_text) > MIN_SECTION_CHARS

    negatives = []
    for tmpl in NEGATIVE_REASONING_TEMPLATES:
        try:
            ctx = {**metrics, **changes, "mda_has_text": mda_has_text}
            result = eval(tmpl["trigger"], {"__builtins__": {}}, ctx)
            if result:
                fmt_ctx = {k: (v if v is not None else 0) for k, v in ctx.items()}
                try:
                    good = tmpl["correct_conclusion"].format(**fmt_ctx)
                except:
                    good = tmpl["correct_conclusion"]
                negatives.append({
                    "bad_conclusion": tmpl["bad_conclusion"],
                    "correct_conclusion": good,
                    "label": tmpl["label"]
                })
        except Exception:
            continue
    return negatives

def classify_reasoning_types(metrics: dict, changes: dict, extra_flags: Optional[dict] = None) -> list:
    types = [k for k, fn in REASONING_TYPES.items() if fn(metrics, changes)]
    if extra_flags:
        for flag, active in extra_flags.items():
            if active and flag in REASONING_TYPES and flag not in types:
                types.append(flag)
    return types

# ─── Quality validators (v20 EliteQualityValidator) ──────────────────────────
class RequiredNumbersInjector:
    @classmethod
    def validate_numbers_in_output(cls, output: str, required_numbers: list) -> dict:
        if not required_numbers:
            return {"found": 0, "total": 0, "missing": [], "passes": True}

        found = []
        missing = []
        for num_str in required_numbers:
            try:
                val = float(num_str)
                pattern = rf"\b({max(0,val-1):.0f}|{val:.0f}|{val+1:.0f}|{val:.1f})\b"
                (found if re.search(pattern, output) else missing).append(num_str)
            except ValueError:
                missing.append(num_str)

        return {
            "found": len(found),
            "total": len(required_numbers),
            "missing": missing[:5],
            "passes": len(found) >= MIN_REQUIRED_NUMBERS_IN_OUTPUT
        }

class RateLimitManager:
    def __init__(self):
        self.consecutive_failures = 0
        self.total_429s = 0
        self.total_skipped = 0
        self.last_wait = 0.0
        self.circuit_open = False
        self.circuit_open_until = 0.0

    def record_success(self):
        with _rl_lock:
            self.consecutive_failures = 0
            self.circuit_open = False

    def record_429(self, wait_seconds: float):
        with _rl_lock:
            self.consecutive_failures += 1
            self.total_429s += 1
            self.last_wait = wait_seconds
            if self.consecutive_failures >= LLM_CIRCUIT_BREAK_N:
                cool_down = min(wait_seconds * 2, 600)
                self.circuit_open = True
                self.circuit_open_until = time.time() + cool_down

    def record_skip(self):
        with _rl_lock:
            self.total_skipped += 1

    def is_circuit_open(self) -> bool:
        with _rl_lock:
            if self.circuit_open and time.time() > self.circuit_open_until:
                self.circuit_open = False
                self.consecutive_failures = 0
            return self.circuit_open

    def wait_circuit(self):
        with _rl_lock:
            remaining = self.circuit_open_until - time.time()
            is_open = self.circuit_open
        if is_open and remaining > 0:
            time.sleep(remaining + 1)
            with _rl_lock:
                self.circuit_open = False
                self.consecutive_failures = 0

    @staticmethod
    def parse_wait_from_error(err_str: str) -> int:
        m = re.search(r"try again in (\d+)m([\d.]+)s", err_str)
        if m:
            return int(m.group(1)) * 60 + int(float(m.group(2))) + 5
        m = re.search(r"try again in ([\d.]+)s", err_str)
        if m:
            return int(float(m.group(1))) + 5
        return 60

    @staticmethod
    def jitter(base: float) -> float:
        return base * (0.8 + random.random() * 0.4)

    def summary(self) -> str:
        return (f"total_429s={self.total_429s}, skipped={self.total_skipped}, "
                f"consecutive={self.consecutive_failures}")

_rate_limiter = RateLimitManager()

class QualityValidator:
    HALLUCINATION_PATTERNS = [
        r"\b(iphone|ipad|mac|services|aws|azure|ads)\b.{0,60}\b(drove|boosted|caused|led to)\b",
        r"\b(strong|robust|surging) demand\b",
        r"\boperational efficiency (drove|boosted|improved) (revenue|margin)\b",
    ]
    _hall_compiled = [re.compile(p, re.IGNORECASE) for p in HALLUCINATION_PATTERNS]
    _filler_compiled = [re.compile(rf"\b{tok}\b", re.IGNORECASE) for tok in FILLER_TOKENS]

    @classmethod
    def check_hallucination(cls, output: str, has_mda: bool) -> dict:
        if has_mda:
            return {"flag": False, "violations": [], "note": "MD&A available"}
        violations = [p.search(output).group()[:80] for p in cls._hall_compiled if p.search(output)]
        return {"flag": len(violations) > 0, "violations": violations, "note": f"{len(violations)} unsupported claims" if violations else "Clean"}

    @classmethod
    def check_style(cls, output: str) -> dict:
        filler_hits = [p.pattern for p in cls._filler_compiled if p.search(output)]
        word_count = len(output.split())
        num_count = len(_NUM_RE.findall(output))
        density = round(num_count / max(word_count, 1), 3)
        return {
            "filler_tokens": filler_hits[:5],
            "word_count": word_count,
            "numerical_density": density,
            "density_ok": density >= 0.06,
            "length_ok": 80 <= word_count <= 600
        }

    @classmethod
    def validate(cls, output: str, metrics: dict, changes: dict, has_mda: bool,
                 required_numbers: Optional[list] = None, paradox_detected: bool = False) -> dict:
        hall = cls.check_hallucination(output, has_mda)
        style = cls.check_style(output)
        num_check = RequiredNumbersInjector.validate_numbers_in_output(output, required_numbers or [])

        score = 100
        if hall["flag"]:
            score -= 20 * len(hall["violations"])
        if not style["density_ok"]:
            score -= 10
        if not style["length_ok"]:
            score -= 5
        if not num_check["passes"]:
            score -= DELTA_PENALTY_PER_MISS * max(0, MIN_REQUIRED_NUMBERS_IN_OUTPUT - num_check["found"])
        if paradox_detected and ("paradox" in output.lower() or "current ratio" in output.lower()):
            score = min(100, score + 5)

        score = max(0, score)
        tier = "gold" if score >= GOLD_QUALITY_THRESHOLD else "silver" if score >= SILVER_QUALITY_THRESHOLD else "bronze"

        return {
            "quality_score": score,
            "hallucination": hall,
            "style": style,
            "required_numbers": num_check,
            "passes_threshold": score >= SILVER_QUALITY_THRESHOLD,
            "tier": tier
        }

class EliteQualityValidator(QualityValidator):
    MANDATORY_METRICS_PATTERNS = {
        "net_income_yoy": [r"net income.{0,40}(yoy|year|%|grew|declin|chang)", r"net income.{0,60}\d+\.?\d*\s*%"],
        "ebitda": [r"ebitda.{0,60}\d", r"ebitda.{0,40}(grew|declin|chang|yoy)"],
        "capex": [r"cap.?ex.{0,60}\d", r"capital expenditure.{0,60}\d"],
        "eps": [r"\beps\b.{0,60}\d", r"earnings per share.{0,60}\d"],
        "gross_margin_delta": [r"gross margin.{0,40}(expand|compress|improv|declin|\d+\s*pp|\d+\s*basis)"],
    }
    _mandatory_compiled = {k: [re.compile(p, re.IGNORECASE) for p in pats] for k, pats in MANDATORY_METRICS_PATTERNS.items()}

    CAUSAL_PATTERNS = [r"\bdue to\b", r"\bbecause\b", r"\bdriven by\b", r"\bowing to\b",
                       r"\bresulting from\b", r"\battributable to\b", r"\bprimarily from\b",
                       r"\bwhich reflects\b", r"\bfollowing\b"]
    _causal_compiled = [re.compile(p, re.IGNORECASE) for p in CAUSAL_PATTERNS]

    CAGR_PATTERN = re.compile(r"(cagr|compound|end/start|\^\s*\(|1/\d|annuali[sz]ed)", re.IGNORECASE)
    VALID_SIGNALS = {"STRONG_BUY", "BUY", "HOLD", "REDUCE", "SELL"}

    @classmethod
    def check_numerical_density(cls, output: str) -> dict:
        pct = re.findall(r"[+-]?\d+\.?\d*\s*(?:pp|%|basis points|bps)", output, re.IGNORECASE)
        usd = re.findall(r"\$\d+\.?\d*\s*[BbMmTt]", output, re.IGNORECASE)
        total = len(pct) + len(usd)
        return {"pct_count": len(pct), "usd_count": len(usd), "total_nums": total, "passes_req1": total >= 10, "pct_samples": pct[:5]}

    @classmethod
    def check_mandatory_metrics(cls, output: str) -> dict:
        results = {}
        missing = []
        for metric, patterns in cls._mandatory_compiled.items():
            found = any(p.search(output) for p in patterns)
            results[metric] = found
            if not found:
                missing.append(metric)
        return {"metrics_found": results, "missing": missing, "passes_req2": len(missing) == 0}

    @classmethod
    def check_causal_reasoning(cls, output: str) -> dict:
        hits = [p.search(output).group() for p in cls._causal_compiled if p.search(output)]
        conn_count = CausalKeywordScanner.count(output)
        total_hits = max(len(hits), conn_count)
        return {"causal_hits": hits[:5], "count": total_hits, "passes_req3": total_hits >= CAUSAL_KEYWORD_MIN,
                "connector_count": conn_count, "passes_v20_req": conn_count >= CAUSAL_CONNECTOR_MIN}

    @classmethod
    def check_cagr_formula(cls, output: str, has_prior: bool) -> dict:
        if not has_prior:
            return {"passes_req4": True, "note": "Single period — CAGR not required"}
        found = bool(cls.CAGR_PATTERN.search(output))
        return {"cagr_found": found, "passes_req4": found, "note": "CAGR formula found" if found else "CAGR formula MISSING"}

    @classmethod
    def check_investment_signal(cls, output: str) -> dict:
        sm = re.search(r'"investment_signal"\s*:\s*"([^"]+)"', output, re.IGNORECASE)
        rm = re.search(r'"signal_rationale"\s*:\s*"([^"]+)"', output, re.IGNORECASE)
        cm = re.search(r'"conflict_rationale"\s*:\s*"([^"]+)"', output, re.IGNORECASE)

        signal = sm.group(1).strip().upper() if sm else None
        rationale = rm.group(1) if rm else None
        conflict = cm.group(1) if cm else None

        num_in_r = len(re.findall(r"\d+\.?\d*\s*%", rationale or ""))
        has_cr = bool(conflict and len(conflict) > 20)
        needs_cr = signal == "HOLD" if signal else False
        cr_ok = has_cr if needs_cr else True

        return {"signal": signal, "signal_valid": signal in cls.VALID_SIGNALS if signal else False,
                "rationale_present": bool(rationale and len(rationale) > 20), "rationale_nums": num_in_r,
                "conflict_present": has_cr, "conflict_rationale_ok": cr_ok,
                "passes_req7": (signal in cls.VALID_SIGNALS and bool(rationale) and num_in_r >= 2 and cr_ok) if signal else False}

    @classmethod
    def elite_validate(cls, output: str, metrics: dict, changes: dict, has_mda: bool, has_prior: bool = False) -> dict:
        base = cls.validate(output, metrics, changes, has_mda)
        num_chk = cls.check_numerical_density(output)
        man_chk = cls.check_mandatory_metrics(output)
        caus_chk = cls.check_causal_reasoning(output)
        cagr_chk = cls.check_cagr_formula(output, has_prior)
        sig_chk = cls.check_investment_signal(output)
        causal_density = CausalKeywordScanner.count(output)

        score = base["quality_score"]
        if not num_chk["passes_req1"]:
            score -= max(0, 10 - num_chk["total_nums"]) * 1.5
        if not man_chk["passes_req2"]:
            score -= len(man_chk["missing"]) * 10
        if not caus_chk["passes_req3"]:
            score -= 10
        if not caus_chk.get("passes_v20_req", True):
            score -= 5
        if not cagr_chk["passes_req4"]:
            score -= 10
        if not sig_chk["passes_req7"]:
            score -= 10

        score = max(0, round(score, 1))
        tier = "gold" if score >= GOLD_QUALITY_THRESHOLD else "silver" if score >= SILVER_QUALITY_THRESHOLD else "bronze"

        return {
            **base,
            "quality_score": score,
            "tier": tier,
            "passes_threshold": score >= SILVER_QUALITY_THRESHOLD,
            "causal_density_score": causal_density,
            "elite_checks": {
                "req1_numerical_density": num_chk,
                "req2_mandatory_metrics": man_chk,
                "req3_causal_reasoning": caus_chk,
                "req4_cagr_formula": cagr_chk,
                "req7_investment_signal": sig_chk,
            },
            "gold_ready": score >= GOLD_QUALITY_THRESHOLD,
            "rejection_reasons": [
                f"REQ-1: only {num_chk['total_nums']} numbers (need ≥10)" if not num_chk["passes_req1"] else None,
                f"REQ-2: missing {man_chk['missing']}" if not man_chk["passes_req2"] else None,
                f"REQ-3: insufficient causal ({caus_chk['count']} hits)" if not caus_chk["passes_req3"] else None,
                f"REQ-V20: connectors {causal_density} (need ≥{CAUSAL_CONNECTOR_MIN})" if not caus_chk.get("passes_v20_req", True) else None,
                "REQ-4: CAGR formula missing" if not cagr_chk["passes_req4"] else None,
                "REQ-7: invalid/incomplete signal" if not sig_chk["passes_req7"] else None,
            ],
        }

# ─── CAUSAL / MDA helpers ─────────────────────────────────────────────────────
class CausalGuard:
    MDA_DRIVER_PATTERNS = {
        "cost": [r"cost(s)?\s+(increased|higher|rose|surge)", r"input cost", r"logistics cost", r"labor cost"],
        "demand": [r"demand (increased|higher|strong|grew)", r"customer (demand|volume)", r"order volume"],
        "pricing": [r"price (increase|higher)", r"pricing power", r"ASP"],
        "volume": [r"unit (volume|sales|shipped)", r"shipment(s)?", r"units sold"],
        "fx": [r"foreign (exchange|currency)", r"currency (headwind|tailwind|impact)"],
        "acquisition": [r"acqui(red|sition|re)", r"merger", r"inorganic"],
        "restructuring": [r"restructur", r"headcount reduction", r"cost reduction program"],
        "macro": [r"macroeconomic", r"recession", r"inflation", r"interest rate"],
    }
    _mda_compiled = {k: [re.compile(p, re.IGNORECASE) for p in pats] for k, pats in MDA_DRIVER_PATTERNS.items()}

    @classmethod
    def detect_drivers_in_mda(cls, mda_text: str) -> dict:
        if not mda_text:
            return {}
        return {driver: any(p.search(mda_text) for p in patterns) for driver, patterns in cls._mda_compiled.items()}

    @classmethod
    def build_causality_instruction(cls, metrics: dict, changes: dict, mda_text: str) -> str:
        has_mda = bool(mda_text and len(mda_text) > MIN_SECTION_CHARS)
        drivers = cls.detect_drivers_in_mda(mda_text) if has_mda else {}

        if has_mda and drivers:
            allowed = [k for k, v in drivers.items() if v]
            return (f"CAUSALITY RULE: MD&A available. Attribute drivers ONLY to: "
                    f"{', '.join(allowed) or 'none identified'}. No invented attribution.")
        elif has_mda:
            return "CAUSALITY RULE: MD&A available but no clear drivers found. Metric-level causality only."
        else:
            return ("CAUSALITY RULE: NO MD&A. Base ALL claims on quantitative metrics. "
                    "FORBIDDEN: product names, segment names, demand narratives.")

class MDAGrounder:
    ALIGNMENT_MAP = [
        ("margin_decline", ["cost", "expense", "higher", "increase", "pressure"], "Margin decline consistent with MD&A: elevated {mda_driver} costs."),
        ("margin_expansion", ["efficiency", "scale", "leverage", "reduction", "savings"], "Margin expansion supported by MD&A: {mda_driver} improvements."),
        ("revenue_growth", ["demand", "volume", "customer", "market", "growth"], "Revenue growth corroborated by MD&A: {mda_driver}."),
        ("revenue_decline", ["lower", "decline", "weak", "soft", "headwind"], "Revenue decline consistent with MD&A: {mda_driver} headwinds."),
        ("capex_increase", ["expansion", "invest", "capacity", "infrastructure", "build"], "CapEx increase aligned with MD&A investment narrative: {mda_driver}."),
        ("debt_increase", ["acqui", "borrow", "financing", "credit", "facility"], "Leverage increase linked to MD&A: {mda_driver}."),
    ]

    @classmethod
    def build_grounded_context(cls, metrics: dict, changes: dict, mda_text: str) -> str:
        if not mda_text or len(mda_text) < MIN_SECTION_CHARS:
            return "[MD&A not available — quantitative causality only]"

        states = []
        rev_d = changes.get("revenue_yoy_pct", 0) or 0
        gm_d = changes.get("gross_margin_pct_delta", 0) or 0

        if rev_d > 3:
            states.append("revenue_growth")
        if rev_d < -3:
            states.append("revenue_decline")
        if gm_d < -1:
            states.append("margin_decline")
        if gm_d > 1:
            states.append("margin_expansion")

        mda_low = mda_text.lower()
        grounded = []

        for state in states:
            for condition, keywords, template in cls.ALIGNMENT_MAP:
                if condition != state:
                    continue
                found_kw = [kw for kw in keywords if kw in mda_low]
                if found_kw:
                    grounded.append(template.format(mda_driver=", ".join(found_kw[:3])))
                    break

        if not grounded:
            return "[MD&A available but no strong alignment — use quantitative causality]"

        return "MD&A ALIGNED CONTEXT:\n" + "\n".join(f"• {s}" for s in grounded)

class EvidenceConfidence:
    @classmethod
    def score(cls, metrics: dict, changes: dict, text_sections: dict) -> dict:
        mda = text_sections.get("mda", "") or ""
        has_mda = len(mda) > MIN_SECTION_CHARS
        has_risk = len(text_sections.get("risk_factors", "") or "") > MIN_SECTION_CHARS

        key_metrics = ["revenue", "gross_profit", "operating_income", "net_income",
                       "free_cash_flow", "total_assets", "long_term_debt"]
        key_changes = ["revenue_yoy_pct", "gross_margin_pct_delta", "net_income_yoy_pct"]

        m_cov = sum(1 for m in key_metrics if m in metrics) / len(key_metrics)
        c_cov = sum(1 for c in key_changes if c in changes) / len(key_changes)

        if m_cov >= 0.85 and c_cov >= 0.8 and has_mda:
            level = "HIGH"
            qualifier = ""
        elif m_cov >= 0.6 and c_cov >= 0.5:
            level = "MEDIUM"
            qualifier = "Note: Partial metric coverage."
        else:
            level = "LOW"
            qualifier = "IMPORTANT: Limited data. Conclusions preliminary."

        return {"level": level, "metric_coverage": round(m_cov, 2), "change_coverage": round(c_cov, 2),
                "has_mda": has_mda, "has_risk": has_risk, "qualifier": qualifier}

# ─── CAGREngine ───────────────────────────────────────────────────────────────
class CAGREngine:
    @staticmethod
    def compute(start_val: float, end_val: float, n_years: float) -> Optional[dict]:
        if not start_val or start_val <= 0 or n_years <= 0:
            return None
        try:
            cagr = ((end_val / start_val) ** (1.0 / n_years) - 1.0) * 100
            sign = "+" if cagr >= 0 else ""
            return {
                "cagr_pct": round(cagr, 2),
                "formula_str": f"({_usd(end_val)}/{_usd(start_val)})^(1/{n_years:.1f})-1 = {sign}{cagr:.2f}%",
                "direction": "growth" if cagr > 1 else "decline" if cagr < -1 else "flat",
                "start_val": start_val,
                "end_val": end_val,
                "n_years": round(n_years, 1),
            }
        except Exception:
            return None

    @staticmethod
    def years_between(period_a: str, period_b: str) -> float:
        try:
            d1 = datetime.strptime(period_a[:10], "%Y-%m-%d")
            d2 = datetime.strptime(period_b[:10], "%Y-%m-%d")
            return max(abs((d2 - d1).days) / 365.25, 0.25)
        except Exception:
            return 1.0

    @classmethod
    def from_records(cls, records: list, metric: str = "revenue") -> Optional[dict]:
        valid = [r for r in records if r.get("metrics", {}).get(metric) and r["metrics"][metric] > 0]
        if len(valid) < 2:
            return None

        oldest = valid[0]
        newest = valid[-1]

        result = cls.compute(oldest["metrics"][metric], newest["metrics"][metric],
                             cls.years_between(oldest["meta"].get("period_of_report", ""),
                                               newest["meta"].get("period_of_report", "")))
        if result:
            result["start_period"] = oldest["meta"].get("period_of_report", "")
            result["end_period"] = newest["meta"].get("period_of_report", "")
            result["metric"] = metric
        return result

class LiquidityParadoxDetector:
    @classmethod
    def detect(cls, metrics: dict) -> Optional[dict]:
        cr = metrics.get("current_ratio")
        cash = metrics.get("cash_and_equivalents", 0) or 0
        cl = metrics.get("current_liabilities", 0) or 0
        fcf = metrics.get("free_cash_flow", 0) or 0

        if cr is None or cr >= 1.0:
            return None
        if cash < 5e9:
            return None

        cash_cov = round(cash / cl * 100, 1) if cl > 0 else None
        fcf_cov = round(cl / fcf, 2) if (fcf and fcf > 0 and cl > 0) else None
        risk_lvl = "LOW" if cash > cl else ("MEDIUM" if cash > cl * 0.5 else "HIGH")

        return {
            "paradox_detected": True,
            "current_ratio": cr,
            "cash_balance": cash,
            "current_liabilities": cl,
            "cash_coverage_pct": cash_cov,
            "fcf_payoff_years": fcf_cov,
            "risk_level": risk_lvl,
            "explanation": f"LIQUIDITY PARADOX: CR={cr:.2f} but cash={_usd(cash)}. Risk={risk_lvl}.",
        }

def _segment_block(segments: list) -> str:
    if not segments:
        return "[No segment data available]"
    lines = ["SEGMENT DATA (from Notes):"]
    for seg in segments:
        rev_str = _usd(seg.get("revenue"))
        oi_str = _usd(seg.get("operating_income")) if seg.get("operating_income") is not None else "N/A"
        lines.append(f"  • {seg['name']}: Revenue={rev_str} Op.Income={oi_str}")
    lines.append("REQUIRED: reference ≥2 segments by name + revenue in your analysis.")
    return "\n".join(lines)

# ─── FEW-SHOT GOLD ────────────────────────────────────────────────────────────
FEW_SHOT_GOLD = """
═══ GOLD EXAMPLE (Score 97/100) — MIMIC THIS STYLE ═══
Input: Revenue $42.3B (+8.1% YoY), GM 43.2% (Δ-1.8pp), Net Income $5.1B (-12.3% YoY),
EBITDA $8.9B (+2.1% YoY), EPS $3.21 (-11.0% YoY), FCF $6.4B (+18.7% YoY), CapEx $4.2B (+31.0% YoY),
D/E 1.8x, CR 0.87x, Cash $29.4B
Output snippet:
{
  "trend_summary": "Revenue grew +8.1% YoY to $42.3B, primary driver being sustained enterprise volume, with CAGR ($42.3B/$38.9B)^(1/1.0)-1=+8.7% signalling deceleration vs prior 3yr CAGR of +14.2%. Gross margin compressed -1.8pp to 43.2% attributable to labour and input cost inflation, while FCF surged +18.7% to $6.4B owing to improved working capital management.",
  "investment_signal": "HOLD",
  "signal_rationale": "Revenue growing +8.1% YoY but net margin compressed to 12.1% (-1.8pp) stemming from cost inflation and D/E=1.8x leverage drag."
}
═══ REJECTED EXAMPLE (Score 31/100) — NEVER DO THIS ═══
{
  "trend_summary": "The company showed strong performance with revenue growing significantly.",
  "investment_signal": "HOLD",
  "signal_rationale": "The company faces headwinds."
}
REJECTION REASON: Zero % figures. No causal language. No EBITDA/EPS/CapEx values.
"""

# ─── PROMPT BUILDERS (v21: same as v20, RAG context is already assembled) ────

def build_stage_a_prompt(record: dict, prior_record: Optional[dict] = None) -> str:
    m, c, s = record.get("metrics", {}), record.get("changes", {}), record.get("signals", {})
    txt, meta = record.get("text_sections", {}), record.get("meta", {})
    mda_text = (txt.get("mda", "") or "")[:800]
    segments = m.get("segments", [])

    # ── CAGR hint — only when both revenues are positive and valid ────────────
    cagr_hint = ""
    cur_rev  = m.get("revenue")
    prev_rev = (prior_record or {}).get("metrics", {}).get("revenue") if prior_record else None
    if cur_rev and prev_rev and cur_rev > 0 and prev_rev > 0:
        try:
            pp  = prior_record["meta"].get("period_of_report", "")
            cp  = meta.get("period_of_report", "")
            y1  = datetime.strptime(pp[:10], "%Y-%m-%d")
            y2  = datetime.strptime(cp[:10], "%Y-%m-%d")
            n   = max((y2 - y1).days / 365.25, 0.5)
            val = ((cur_rev / prev_rev) ** (1 / n) - 1) * 100
            if abs(val) < 200:   # sanity: reject impossible CAGR values
                cagr_hint = (
                    f"Pre-computed CAGR: ({_usd(cur_rev)}/{_usd(prev_rev)})"
                    f"^(1/{n:.1f})-1 = {val:+.2f}%"
                )
        except Exception:
            pass

    return f"""You are a financial reasoning expert. Extract CAUSAL CHAINS from the data below.

COMPANY: {meta.get('company_name')} ({meta.get('ticker')}) | PERIOD: {meta.get('period_of_report')}
{cagr_hint}

KEY METRICS:
- Revenue: {_usd(m.get('revenue'))} (YoY {_pct(c.get('revenue_yoy_pct'))})
- Gross Margin:{_pct_plain(m.get('gross_margin_pct'))} Δ:{_pct(c.get('gross_margin_pct_delta'),'pp')}
- Op Margin: {_pct_plain(m.get('operating_margin_pct'))}
- Net Income: {_usd(m.get('net_income'))} (YoY {_pct(c.get('net_income_yoy_pct'))})
- EBITDA: {_usd(m.get('ebitda'))} (YoY {_pct(c.get('ebitda_yoy_pct'))})
- EPS: {m.get('eps_diluted','N/A')} (YoY {_pct(c.get('eps_yoy_pct'))})
- FCF: {_usd(m.get('free_cash_flow'))} (YoY {_pct(c.get('fcf_yoy_pct'))})
- CapEx: {_usd(m.get('capex'))} (YoY {_pct(c.get('capex_yoy_pct'))})
- D/E: {m.get('debt_to_equity','N/A')} | CR: {m.get('current_ratio','N/A')}
- Cash: {_usd(m.get('cash_and_equivalents'))} | LT Debt: {_usd(m.get('long_term_debt'))}

{_segment_block(segments)}

MD&A excerpt (RAG-retrieved, full quality):
{mda_text or '[not available]'}

TASK: Write 5-7 causal chains: [ROOT FACTOR] → [MECHANISM] → [FINANCIAL RESULT with exact %]

V20-4 REQUIRED connector words (use ≥1 per chain):
driven by / offset by / primarily due to / resulted from / due to / attributable to /
stemming from / resulting in / because / consequently / owing to / which led to /
benefiting from / weighed by / impacted by / partially offset by

Rules:
- Every chain MUST end with a specific metric and its % change
- Cover: revenue driver, margin driver, EPS divergence, FCF quality, EBITDA bridge,
  segment performance (if available), one risk factor

Output ONLY the numbered list. No JSON."""



def build_stage_b_prompt(record: dict, causal_chains: str, prior_record: Optional[dict] = None) -> str:

    
    m, c, s = record.get("metrics", {}), record.get("changes", {}), record.get("signals", {})
    txt, meta = record.get("text_sections", {}), record.get("meta", {})
    mda_text = (txt.get("mda", "") or "")
    risk_text = (txt.get("risk_factors", "") or "")[:600]
    has_mda = len(mda_text) > MIN_SECTION_CHARS
    segments = m.get("segments", [])

    # ── Data integrity gate — refuse analysis on impossible metrics ───────────
    _rev = m.get("revenue")
    _gm  = m.get("gross_margin_pct")
    _data_warning = ""
    if _rev is not None and _rev <= 0:
        _data_warning = (
            f"\n⛔ DATA ERROR: Revenue={_usd(_rev)} is non-positive. "
            f"If this is confirmed, output ONLY: "
            f'{{\"error\": \"DATA_ERROR\", \"reason\": \"Non-positive revenue — '
            f'cannot produce valid analysis\"}}'
        )
    elif _gm is not None and (_gm < -100 or _gm > 100):
        _data_warning = (
            f"\n⛔ DATA ERROR: Gross margin={_gm:.1f}% is outside [-100%, 100%]. "
            f"If confirmed, output ONLY: "
            f'{{\"error\": \"DATA_ERROR\", \"reason\": \"Impossible gross margin\"}}'
        )

    causality_instr = CausalGuard.build_causality_instruction(m, c, mda_text)
    grounded_ctx = MDAGrounder.build_grounded_context(m, c, mda_text)
    conf = EvidenceConfidence.score(m, c, txt)

    gm_delta_str = _pct(c.get("gross_margin_pct_delta"), "pp")
    ni_yoy_str = _pct(c.get("net_income_yoy_pct"))
    ebitda_yoy_str = _pct(c.get("ebitda_yoy_pct"))
    eps_yoy_str = _pct(c.get("eps_yoy_pct"))
    capex_yoy_str = _pct(c.get("capex_yoy_pct"))
    rev_yoy_str = _pct(c.get("revenue_yoy_pct"))
    fcf_str = _usd(m.get("free_cash_flow"))
    fcf_yoy_str = _pct(c.get("fcf_yoy_pct"))
    nm_str = _pct_plain(m.get("net_margin_pct"))
    ebitda_str = _usd(m.get("ebitda"))
    eps_str = str(m.get("eps_diluted", "N/A"))
    capex_str = _usd(m.get("capex"))
    fcf_conv_str = str(m.get("fcf_conversion_ratio", "N/A"))

    cagr_hint = ""
    if prior_record and prior_record.get("metrics", {}).get("revenue") and m.get("revenue"):
        try:
            pp = prior_record["meta"].get("period_of_report", "")
            cp = meta.get("period_of_report", "")
            y1 = datetime.strptime(pp[:10], "%Y-%m-%d")
            y2 = datetime.strptime(cp[:10], "%Y-%m-%d")
            n = max((y2 - y1).days / 365.25, 0.5)
            val = ((m["revenue"] / prior_record["metrics"]["revenue"]) ** (1 / n) - 1) * 100
            cagr_hint = f"\nPRE-COMPUTED CAGR ({pp[:7]}→{cp[:7]}): ({_usd(m['revenue'])}/{_usd(prior_record['metrics']['revenue'])})^(1/{n:.1f})-1 = {val:+.2f}%"
        except Exception:
            pass

    liq_note = ""
    cr = m.get("current_ratio")
    csh = m.get("cash_and_equivalents")
    if cr is not None and cr < 1.0 and csh is not None and csh > 1e9:
        liq_note = f"\n⚠ LIQUIDITY PARADOX: CR={cr:.2f} BUT cash={_usd(csh)}. MUST address."

    edge_cases = s.get("edge_cases_detected", [])
    edge_instr = ""
    if edge_cases:
        focuses = [EDGE_CASE_ARCHETYPES[ec]["reasoning_focus"] for ec in edge_cases if ec in EDGE_CASE_ARCHETYPES]
        edge_instr = "\n⚠ EDGE CASES:\n" + "\n".join(f"  • {f}" for f in focuses)

    segment_str = _segment_block(segments)

    return f"""You are an Elite Senior Financial Analyst producing Gold-tier LLM fine-tuning data.

SOURCE: v21 RAG (local HTML, full section coverage, no truncation)

{FEW_SHOT_GOLD}

COMPANY: {meta.get('company_name')} ({meta.get('ticker')}) | PERIOD: {meta.get('period_of_report')} | FORM: {meta.get('form_type')}
EVIDENCE: {conf['level']} (metric_cov={conf['metric_coverage']:.0%}, MD&A={'YES (RAG-retrieved, full quality)' if has_mda else 'NO'})
{cagr_hint}{liq_note}

STAGE-A CAUSAL CHAINS — integrate ALL:
{causal_chains or '[No causal chains — derive from quantitative data only]'}

━━━ FINANCIAL DATA (v21: XBRL + RAG cross-referenced) ━━━
Revenue: {_usd(m.get('revenue'))} YoY:{rev_yoy_str}
Gross Margin: {_pct_plain(m.get('gross_margin_pct'))} Δ:{gm_delta_str}
COGS ratio: {_pct_plain(m.get('cogs_ratio_pct'))} | SG&A:{_pct_plain(m.get('sga_ratio_pct'))}
Op Margin: {_pct_plain(m.get('operating_margin_pct'))} YoY:{_pct(c.get('operating_income_yoy_pct'))}
EBITDA: {ebitda_str} YoY:{ebitda_yoy_str}
Net Income: {_usd(m.get('net_income'))} Margin:{nm_str} YoY:{ni_yoy_str}
EPS: {eps_str} YoY:{eps_yoy_str}
R&D: {_usd(m.get('research_development'))} ({_pct_plain(s.get('rd_intensity_pct'))} of rev)
FCF: {fcf_str} YoY:{fcf_yoy_str} FCF/NI:{fcf_conv_str}
OCF: {_usd(m.get('operating_cash_flow'))} | CapEx:{capex_str} YoY:{capex_yoy_str}
Buybacks: {_usd(m.get('share_repurchases'))} | SBC:{_usd(m.get('stock_based_compensation'))}
Cash: {_usd(m.get('cash_and_equivalents'))} | Net Debt:{_usd(m.get('net_debt'))}
LT Debt: {_usd(m.get('long_term_debt'))} YoY:{_pct(c.get('debt_yoy_pct'))}
D/E:{m.get('debt_to_equity','N/A')} Curr Ratio: {m.get('current_ratio','N/A')} | ROA:{_pct_plain(m.get('return_on_assets_pct'))}
Rule-of-40: {s.get('rule_of_40_score','N/A')} ({'PASS' if s.get('rule_of_40_pass') else 'FAIL'})
{edge_instr}
{segment_str}

MD&A ({len(mda_text)} chars, RAG-retrieved):
{mda_text[:1200] or '[NOT AVAILABLE]'}

Risks:
{risk_text or '[NOT AVAILABLE]'}

{grounded_ctx}

━━━ V21 MANDATORY REQUIREMENTS ━━━
REQ-1: ≥10 specific % or $ figures
REQ-2: MUST reference NI YoY ({ni_yoy_str}), EBITDA YoY ({ebitda_yoy_str}), CapEx YoY ({capex_yoy_str}), EPS ({eps_yoy_str}), GM-delta ({gm_delta_str})
REQ-3: ≥{CAUSAL_KEYWORD_MIN} causal connectors minimum
REQ-4: CAGR formula if prior data exists{cagr_hint}
REQ-7: investment_signal ∈ {{STRONG_BUY,BUY,HOLD,REDUCE,SELL}} with ≥2 metrics
REQ-HOLD: conflict_rationale with bull + bear cases and numbers
REQ-V21-RAG: segment_analysis MUST use MD&A text evidence where available

{causality_instr}

Return ONLY valid JSON. No markdown. No preamble:

{{
  "trend_summary": "<2-3 sentences, ≥5 specific numbers, causal connectors, cite MD&A evidence>",
  "key_inflection_points": ["<period: exact metric Δ + WHY>", "<second>"],
  "revenue_cagr": "<formula: (End/Start)^(1/N)-1 = X%>",
  "margin_evolution": "<gross {gm_delta_str} + net {nm_str} — driver>",
  "cash_generation_quality": "<FCF={fcf_str} YoY {fcf_yoy_str}, FCF/NI={fcf_conv_str}, CapEx={capex_str}>",
  "ebitda_analysis": "<EBITDA={ebitda_str} YoY {ebitda_yoy_str} — D&A bridge + margin>",
  "eps_analysis": "<EPS={eps_str} YoY {eps_yoy_str} — divergence explanation>",
  "segment_analysis": "<≥2 segments by name + revenue + causal driver from MD&A text — or N/A>",
  "key_risks": ["<risk + metric>", "<leverage/liquidity>", "<trend risk>"],
  "conflict_rationale": "<HOLD only: bull case with metrics vs bear case with metrics>",
  "investment_signal": "STRONG_BUY | BUY | HOLD | REDUCE | SELL",
  "signal_rationale": "<Causal Triad: [Metric Δ] → [Root Cause] → [Forward Impact] with ≥2 metrics>",
  "causal_density_score": <integer>,
  "rag_context_chars": {len(mda_text)},
  "cross_ref_validated": true,
  "scaling_unit": "millions",
  "data_source_mix": {{"xbrl_pct": <0-100>, "text_pct": <0-100>, "rag_retrieved": true}}
}}"""

def build_auditor_prompt(analyst_output: str, metrics: dict, changes: dict, qr: dict) -> str:
    num_chk = qr.get("elite_checks", {}).get("req1_numerical_density", {})
    man_chk = qr.get("elite_checks", {}).get("req2_mandatory_metrics", {})
    caus_chk = qr.get("elite_checks", {}).get("req3_causal_reasoning", {})
    cagr_chk = qr.get("elite_checks", {}).get("req4_cagr_formula", {})
    sig_chk = qr.get("elite_checks", {}).get("req7_investment_signal", {})
    causal_density = qr.get("causal_density_score", 0)

    return f"""You are a HOSTILE FINANCIAL DATA AUDITOR. Find EVERY violation. Do NOT praise.

ANALYST OUTPUT:
{analyst_output[:3000]}

SCORES:
REQ-1 (numerical density): {num_chk.get('total_nums',0)}/10 needed
REQ-2 (mandatory metrics): missing={man_chk.get('missing',[])}
REQ-3 (causal phrases): {caus_chk.get('count',0)}/3 minimum
V21-4 (connectors): {causal_density}/{CAUSAL_CONNECTOR_MIN} minimum
REQ-4 (CAGR formula): {'PRESENT' if cagr_chk.get('cagr_found') else 'MISSING'}
REQ-7 (signal quality): signal={sig_chk.get('signal')}, nums={sig_chk.get('rationale_nums',0)}

AUDIT TASK:
1. Verify every % figure is supported by data (not invented)
2. Check every causal claim has a connector word
3. Flag vague language with no numbers
4. Verify conflict_rationale has bull AND bear with numbers (HOLD signals)

Return ONLY valid JSON:
{{
  "approved": true | false,
  "violations": ["<specific violation>"],
  "missing_items": ["<item that MUST be added>"],
  "rewrite_instructions": ["<precise fix>"],
  "score_estimate": <0-100>,
  "approval_blocking_reasons": ["<reason>"]
}}"""

def build_analyst_rewrite_prompt(original_output: str, auditor_report: str, metrics: dict, changes: dict, causal_chains: str) -> str:
    m, c = metrics, changes
    return f"""You are an Elite Financial Analyst. An auditor REJECTED your output. Fix ALL violations.

AUDITOR REJECTION:
{auditor_report[:2000]}

PREVIOUS OUTPUT (partially wrong):
{original_output[:2000]}

CAUSAL CHAINS:
{causal_chains[:1000] or '[derive from data]'}

MANDATORY FIXES:
1. Add ≥{CAUSAL_CONNECTOR_MIN} causal connectors: driven by / offset by / primarily due to / resulted from / due to / attributable to / stemming from / resulting in / because / consequently / owing to / which led to / benefiting from / weighed by
2. Reference ALL: NI YoY {_pct(c.get('net_income_yoy_pct'))}, EBITDA YoY {_pct(c.get('ebitda_yoy_pct'))}, CapEx YoY {_pct(c.get('capex_yoy_pct'))}, EPS YoY {_pct(c.get('eps_yoy_pct'))}, GM Δ {_pct(c.get('gross_margin_pct_delta'),'pp')}
3. Every directional claim needs a number AND a connector
4. If HOLD: add conflict_rationale with bull + bear + numbers

Return ONLY valid JSON fixing ALL violations:"""

def build_self_correction_prompt(output: str, qr: dict, metrics: dict, changes: dict) -> str:
    missing = qr.get("elite_checks", {}).get("req2_mandatory_metrics", {}).get("missing", [])
    conn_ct = qr.get("causal_density_score", 0)

    corrections = []
    if conn_ct < CAUSAL_CONNECTOR_MIN:
        corrections.append(
            f"CAUSAL LANGUAGE: only {conn_ct} connectors (need ≥{CAUSAL_CONNECTOR_MIN}). "
            "Use: driven by / offset by / primarily due to / resulted from"
        )
    if missing:
        metric_hints = {
            "net_income_yoy": f"Net Income YoY: {_pct(changes.get('net_income_yoy_pct'))}",
            "ebitda": f"EBITDA: {_usd(metrics.get('ebitda'))} YoY {_pct(changes.get('ebitda_yoy_pct'))}",
            "capex": f"CapEx: {_usd(metrics.get('capex'))} YoY {_pct(changes.get('capex_yoy_pct'))}",
            "eps": f"EPS: {metrics.get('eps_diluted','N/A')} YoY {_pct(changes.get('eps_yoy_pct'))}",
            "gross_margin_delta": f"Gross Margin Δ: {_pct(changes.get('gross_margin_pct_delta'),'pp')}",
        }
        hints = [metric_hints[k] for k in missing if k in metric_hints]
        corrections.append(f"MISSING METRICS — add: {' | '.join(hints)}")

    return f"""Fix these specific issues in your previous output:
{chr(10).join(f'• {corr}' for corr in corrections)}

PREVIOUS OUTPUT:
{output[:2000]}

Rewrite the COMPLETE JSON fixing all issues. Return ONLY valid JSON:"""

# ─── LLM CALL HELPER ─────────────────────────────────────────────────────────
def _call_llm_once(prompt: str, llm_instance, label: str = "") -> str:
    for attempt in range(1, MAX_RETRIES + 1):
        _llm_rate_limiter.acquire()
        try:
            use_prompt = prompt
            if attempt > 1:
                use_prompt = (prompt +
                              f"\n\n⚠ RETRY {attempt}/{MAX_RETRIES}: "
                              f"include ≥10 % figures and ≥{CAUSAL_CONNECTOR_MIN} causal connectors.")
            text = llm_instance.invoke(use_prompt).content.strip()
            m_ = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
            if m_:
                text = m_.group(1)
            json.loads(text)
            text = normalize_financial_terms(text)
            _rate_limiter.record_success()
            _llm_rate_limiter.record_success()
            return text
        except json.JSONDecodeError:
            _rate_limiter.record_success()
            _llm_rate_limiter.record_success()
            return text if text else ""
        except Exception as e:
            err = str(e)
            if "429" in err or "rate_limit" in err.lower():
                wait = _rate_limiter.parse_wait_from_error(err)
                _rate_limiter.record_429(wait)
                _llm_rate_limiter.record_429(wait)
                if wait > LLM_429_MAX_WAIT:
                    _rate_limiter.record_skip()
                    return ""
                time.sleep(_rate_limiter.jitter(max(wait, NVIDIA_MIN_INTERVAL * 2)))
                if _rate_limiter.is_circuit_open():
                    _rate_limiter.wait_circuit()
            else:
                if attempt == MAX_RETRIES:
                    break
                time.sleep(_rate_limiter.jitter(BACKOFF_BASE ** attempt))

    _rate_limiter.record_skip()
    return ""

def generate_analysis_v21(
    record: dict,
    llm,
    llm_strict,
    prior_record: Optional[dict] = None,
    metrics: Optional[dict] = None,
    changes: Optional[dict] = None,
    has_prior: bool = False,
    text_sourced_keys: Optional[list] = None,
) -> tuple:
    """V21 multi-stage pipeline — identical structure to v20 generate_analysis_v20
    but uses RAG-retrieved text_sections (richer, no truncation)."""
    if _rate_limiter.is_circuit_open():
        _rate_limiter.wait_circuit()

    m = metrics or record.get("metrics", {})
    c = changes or record.get("changes", {})

    causal_chains = ""
    try:
        raw = _call_llm_once(build_stage_a_prompt(record, prior_record), llm, "Stage-A")
        causal_chains = raw[:1500] if raw else ""
    except Exception as e:
        logger.warning(f"  Stage A failed: {e}")

    text = _call_llm_once(build_stage_b_prompt(record, causal_chains, prior_record), llm, "Stage-B")
    if not text:
        return "", 0

    has_mda = any(v for v in record.get("text_sections", {}).values() if v and len(v) >= MIN_SECTION_CHARS)

    if text_sourced_keys:
        try:
            parsed = json.loads(text)
            total_keys = len(m)
            text_pct = round(len(text_sourced_keys) / max(total_keys, 1) * 100)
            parsed["data_source_mix"] = {"xbrl_pct": 100 - text_pct, "text_pct": text_pct, "rag_retrieved": True}
            text = json.dumps(parsed)
        except Exception:
            pass

    auditor_cycles = 0
    for cycle in range(MAX_AUDITOR_CYCLES):
        qr = EliteQualityValidator.elite_validate(text, m, c, has_mda, has_prior=has_prior)
        needs_audit = (
            not qr["gold_ready"] or
            qr["causal_density_score"] < CAUSAL_CONNECTOR_MIN or
            qr["elite_checks"]["req3_causal_reasoning"]["count"] < CAUSAL_KEYWORD_MIN or
            qr["elite_checks"]["req2_mandatory_metrics"]["missing"]
        )
        if not needs_audit:
            break

        auditor_llm = get_llm_for_context(strict=True)
        auditor_raw = _call_llm_once(build_auditor_prompt(text, m, c, qr), auditor_llm, f"Auditor-{cycle+1}")
        auditor_cycles += 1
        if not auditor_raw:
            break

        try:
            ar = json.loads(auditor_raw)
            if ar.get("approved", False):
                break
        except Exception:
            ar = {"violations": [auditor_raw[:200]], "rewrite_instructions": []}

        rewritten = _call_llm_once(
            build_analyst_rewrite_prompt(text, auditor_raw, m, c, causal_chains),
            llm, f"Rewrite-{cycle+1}"
        )
        if rewritten:
            qr2 = EliteQualityValidator.elite_validate(rewritten, m, c, has_mda, has_prior=has_prior)
            if qr2["quality_score"] >= qr["quality_score"]:
                text = rewritten

    final_qr = EliteQualityValidator.elite_validate(text, m, c, has_mda, has_prior=has_prior)
    causal_ct = final_qr["elite_checks"]["req3_causal_reasoning"]["count"]
    missing = final_qr["elite_checks"]["req2_mandatory_metrics"]["missing"]

    if causal_ct < CAUSAL_KEYWORD_MIN or missing:
        corrected = _call_llm_once(build_self_correction_prompt(text, final_qr, m, c), llm_strict, "Self-Correction")
        if corrected:
            qr_sc = EliteQualityValidator.elite_validate(corrected, m, c, has_mda, has_prior=has_prior)
            if qr_sc["quality_score"] >= final_qr["quality_score"]:
                text = corrected

    return text, auditor_cycles

def generate_analysis(prompt: str, llm) -> str:
    if _rate_limiter.is_circuit_open():
        _rate_limiter.wait_circuit()
    return _call_llm_once(prompt, llm, "single-stage")

# ─── PAIR BUILDERS ────────────────────────────────────────────────────────────
def build_single_pair(
    record: dict,
    analysis: str,
    prior_record: Optional[dict] = None,
    quality_result: Optional[dict] = None,
    cagr_data: Optional[dict] = None,
    paradox_data: Optional[dict] = None,
    abnormal_return: Optional[dict] = None,
    reasoning_path: str = "",
    auditor_cycles: int = 0,
    data_source_mix: Optional[dict] = None,
    cross_ref_result: Optional[dict] = None,
    scaling_unit: str = "millions",
    segments: Optional[list] = None,
    rag_context_chars: int = 0,
) -> dict:
    m, c, s = record.get("metrics", {}), record.get("changes", {}), record.get("signals", {})
    txt, meta = record.get("text_sections", {}), record.get("meta", {})
    has_text = any(v for v in txt.values() if v and len(v) >= MIN_SECTION_CHARS)
    risks = [kw for kw in RISK_KW if kw in ((txt.get("risk_factors", "") or "") + (txt.get("mda", "") or "")).lower()][:6]
    conf = EvidenceConfidence.score(m, c, txt)
    reasoning_types = classify_reasoning_types(m, c)
    edge_cases = s.get("edge_cases_detected", [])

    if edge_cases:
        reasoning_types.append("edge_case")
    if paradox_data:
        reasoning_types.append("liquidity_paradox")
    if abnormal_return:
        reasoning_types.extend(["market_impact", "predictive_signal"])
    if segments:
        reasoning_types.append("segment_analysis")

    ar = abnormal_return or meta.get("abnormal_return", {})
    causal_density = 0
    try:
        parsed_out = json.loads(analysis)
        causal_density = parsed_out.get("causal_density_score", CausalKeywordScanner.count(analysis))
    except Exception:
        causal_density = CausalKeywordScanner.count(analysis)

    prior_section = ""
    if prior_record and prior_record.get("metrics"):
        pm = prior_record["metrics"]
        pp = prior_record["meta"].get("period_of_report", "prior")
        prior_section = (
            f"\nPRIOR ({pp}): Rev={_usd(pm.get('revenue'))} | "
            f"GM={_pct_plain(pm.get('gross_margin_pct'))} | "
            f"NM={_pct_plain(pm.get('net_margin_pct'))} | "
            f"FCF={_usd(pm.get('free_cash_flow'))} | EPS={pm.get('eps_diluted', 'N/A')}"
        )

    ar_section = ""
    if ar:
        ar_section = (
            f"\nMARKET REACTION: {ar.get('abnormal_return_pct', 0):+.3f}% abnormal "
            f"({ar.get('window_days', 5)}d) | {ar.get('market_reaction', '')}"
        )

    seg_section = ""
    if segments:
        lines = ["\nSEGMENTS:"]
        for seg in segments[:4]:
            lines.append(f"  {seg['name']}: Rev={_usd(seg.get('revenue'))} OI={_usd(seg.get('operating_income'))}")
        seg_section = "\n".join(lines)

    inp = (
        f"COMPANY: {meta.get('company_name')} ({meta.get('ticker')}) | "
        f"FORM: {meta.get('form_type')} | PERIOD: {meta.get('period_of_report')} | "
        f"period_type={meta.get('period_type', 'unknown')} | "
        f"fiscal_Q={meta.get('fiscal_quarter', 'N/A')} | "
        f"scaling={scaling_unit} | rag_context_chars={rag_context_chars}"
        f"{prior_section}{ar_section}{seg_section}\n\n"
        f"── INCOME ──\n"
        f"Revenue: {_usd(m.get('revenue'))} (YoY {_pct(c.get('revenue_yoy_pct'))})\n"
        f"Gross Margin: {_pct_plain(m.get('gross_margin_pct'))} Δ{_pct(c.get('gross_margin_pct_delta'), 'pp')}\n"
        f"Op Margin: {_pct_plain(m.get('operating_margin_pct'))}\n"
        f"Net Income: {_usd(m.get('net_income'))} | NM:{_pct_plain(m.get('net_margin_pct'))} | YoY:{_pct(c.get('net_income_yoy_pct'))}\n"
        f"EBITDA: {_usd(m.get('ebitda'))} | YoY:{_pct(c.get('ebitda_yoy_pct'))}\n"
        f"EPS: {m.get('eps_diluted', 'N/A')} (YoY {_pct(c.get('eps_yoy_pct'))})\n\n"
        f"── CASH FLOW ──\n"
        f"FCF: {_usd(m.get('free_cash_flow'))} (YoY {_pct(c.get('fcf_yoy_pct'))}) | FCF/NI:{m.get('fcf_conversion_ratio', 'N/A')}\n"
        f"CapEx: {_usd(m.get('capex'))} YoY {_pct(c.get('capex_yoy_pct'))} | Buybacks:{_usd(m.get('share_repurchases'))}\n\n"
        f"── BALANCE SHEET ──\n"
        f"Net Debt:{_usd(m.get('net_debt'))} | D/E:{m.get('debt_to_equity', 'N/A')} | CR:{m.get('current_ratio', 'N/A')}\n\n"
        f"── SIGNALS ──\n"
        f"{s.get('profitability_tier', 'N/A')} | "
        f"Rule-of-40:{s.get('rule_of_40_score', 'N/A')} "
        f"({'PASS' if s.get('rule_of_40_pass') else 'FAIL'}) | "
        f"Trend:{c.get('trend_revenue', 'N/A')} | Margin:{c.get('trend_margin', 'N/A')}"
    )

    if has_text:
        inp += f"\n\n── MD&A (RAG-retrieved, {rag_context_chars} chars) ──\n{(txt.get('mda', '') or '')[:600]}"
        inp += f"\n\n── RISKS ──\n{(txt.get('risk_factors', '') or '')[:300]}"
    else:
        inp += "\n\n[MD&A not available — quantitative analysis only]"

    if edge_cases:
        inp += f"\n\n── EDGE CASES ──\n{', '.join(edge_cases)}"

    cr_validated = (cross_ref_result or {}).get("cross_ref_clean", True)

    return {
        "instruction": (
            "You are an Elite Senior Financial Analyst (v21 RAG). Analyze the SEC filing data and return "
            "a structured JSON assessment with evidence-grounded causal reasoning. "
            "Include ≥10 specific numerical percentages and reference "
            "Net Income, EBITDA, CapEx, and EPS deltas. Use causal connectors: "
            "driven by / offset by / primarily due to / resulted from / attributable to."
        ),
        "input": inp,
        "output": analysis,
        "record_type": "single_period",
        "metadata": {
            "ticker": meta.get("ticker"),
            "company": meta.get("company_name"),
            "period": meta.get("period_of_report"),
            "form_type": meta.get("form_type"),
            "output_schema": "v21_rag_elite",
            "analysis_mode": "full_rag" if has_text else "quantitative_only",
            "evidence_confidence": conf["level"],
            "period_type": meta.get("period_type", "unknown"),
            "fiscal_quarter": meta.get("fiscal_quarter"),
            "decumulation_applied": meta.get("decumulation_applied", False),
            "quarter_aligned_yoy": meta.get("quarter_aligned_yoy", False),
            "sanity_action": meta.get("sanity_action", "unknown"),
            "profitability": s.get("profitability_tier"),
            "trend_revenue": c.get("trend_revenue"),
            "trend_margin": c.get("trend_margin"),
            "trend_fcf": c.get("trend_fcf"),
            "edge_cases": edge_cases,
            "reasoning_types": reasoning_types,
            "reasoning_path": reasoning_path,
            "risk_keywords": risks,
            "rule_of_40_pass": s.get("rule_of_40_pass"),
            "metrics_count": len(m),
            "text_sections_filled": sum(1 for v in txt.values() if v and len(v) >= MIN_SECTION_CHARS),
            "has_mda": has_text,
            "rag_context_chars": rag_context_chars,
            "quality_score": quality_result.get("quality_score") if quality_result else None,
            "quality_tier": quality_result.get("tier") if quality_result else None,
            "gold_ready": quality_result.get("gold_ready") if quality_result else None,
            "paradox_detected": bool(paradox_data),
            "cagr_precomputed": cagr_data.get("formula_str") if cagr_data else None,
            "abnormal_return_pct": ar.get("abnormal_return_pct") if ar else None,
            "market_reaction": ar.get("market_reaction") if ar else None,
            "causal_density_score": causal_density,
            "auditor_cycles": auditor_cycles,
            "data_source_mix": data_source_mix or {"xbrl_pct": 100, "text_pct": 0, "rag_retrieved": True},
            "cross_ref_validated": cr_validated,
            "scaling_unit": scaling_unit,
            "segment_count": len(segments) if segments else 0,
            "rag_source": FILINGS_ROOT,
        },
    }

def build_negative_pair(record: dict, neg_analysis: str, negatives: list) -> dict:
    m, c, meta = record.get("metrics", {}), record.get("changes", {}), record.get("meta", {})

    inp = (
        f"COMPANY: {meta.get('company_name')} ({meta.get('ticker')}) | "
        f"FORM: {meta.get('form_type')} | PERIOD: {meta.get('period_of_report')}\n"
        f"Revenue YoY:{_pct(c.get('revenue_yoy_pct'))} | "
        f"GM Δ:{_pct(c.get('gross_margin_pct_delta'), 'pp')} | "
        f"NM:{_pct_plain(m.get('net_margin_pct'))}\n\n"
        "Identify what conclusions CANNOT be drawn:\n" +
        "\n".join(f"Scenario {i+1}: {n['label']}" for i, n in enumerate(negatives))
    )

    return {
        "instruction": (
            "You are a financial reasoning teacher. Identify logically unsupported "
            "conclusions and explain epistemic boundaries."
        ),
        "input": inp,
        "output": neg_analysis,
        "record_type": "negative_reasoning",
        "metadata": {
            "ticker": meta.get("ticker"),
            "company": meta.get("company_name"),
            "period": meta.get("period_of_report"),
            "form_type": meta.get("form_type"),
            "output_schema": "v21_negative",
            "reasoning_types": ["negative_inference"],
            "reasoning_path": "Evidence → Epistemic Boundary → Correct Inference",
            "scenarios": [n["label"] for n in negatives],
        },
    }

def build_negative_example_prompt(record: dict, negatives: list) -> str:
    m, c = record.get("metrics", {}), record.get("changes", {})
    meta = record.get("meta", {})

    neg_block = "\n".join(
        f"  SCENARIO {i+1}: {neg['label'].replace('_', ' ')}.\n"
        f"  ❌ INVALID: \"{neg['bad_conclusion']}\"\n"
        f"  ✓ CORRECT: \"{neg['correct_conclusion']}\""
        for i, neg in enumerate(negatives)
    )

    return f"""You are a financial reasoning teacher. Explain WHY each invalid conclusion is logically unsupported.

COMPANY: {meta.get('company_name')} ({meta.get('ticker')}) | PERIOD: {meta.get('period_of_report')}
Revenue YoY: {_pct(c.get('revenue_yoy_pct'))} | GM Δ: {_pct(c.get('gross_margin_pct_delta'), 'pp')} | NM: {_pct_plain(m.get('net_margin_pct'))}
FCF YoY: {_pct(c.get('fcf_yoy_pct'))} | D/E: {m.get('debt_to_equity', 'N/A')} | CR: {m.get('current_ratio', 'N/A')}

{neg_block}

Return ONLY valid JSON:
{{
  "negative_examples": [
    {{
      "scenario_label": "<label>",
      "why_invalid": "<1-2 sentences with specific numbers>",
      "correct_reasoning": "<1-2 sentences with exact metric values>",
      "reasoning_boundary": "<what evidence WOULD support the invalid claim>"
    }}
  ],
  "epistemic_principle": "<1 sentence general principle>"
}}"""

# ─── FILE WRITE HELPERS ───────────────────────────────────────────────────────
def _write_tiered(record: dict, qr: dict, quality_log, gold_out, silver_out, has_abnormal_return: bool = False):
    tier = qr.get("tier", "bronze")
    if tier == "gold" and GOLD_REQUIRES_ABNORMAL_RETURN and not has_abnormal_return:
        tier = "silver"
        logger.debug("  Demoted gold→silver: missing abnormal return label")

    with _file_lock:
        if quality_log:
            quality_log.write(json.dumps({
                "ticker": record.get("meta", {}).get("ticker"),
                "period": record.get("meta", {}).get("period_of_report"),
                **{k: v for k, v in qr.items() if not isinstance(v, dict)}
            }, ensure_ascii=False) + "\n")
            quality_log.flush()

        if tier == "gold" and gold_out:
            gold_out.write(json.dumps(record, ensure_ascii=False) + "\n")
            gold_out.flush()
        elif tier in ("gold", "silver") and silver_out:
            silver_out.write(json.dumps(record, ensure_ascii=False) + "\n")
            silver_out.flush()

def _write_record_to_file(record: dict, out_file, pf, already_tiered: bool = False):
    if out_file is None:
        return
    with _file_lock:
        if record.pop("_needs_llm", False):
            if pf:
                pf.write(json.dumps(record, ensure_ascii=False) + "\n")
                pf.flush()
        else:
            out_file.write(json.dumps(record, ensure_ascii=False) + "\n")
            out_file.flush()

# ─── ABNORMAL RETURN ─────────────────────────────────────────────────────────
def fetch_abnormal_return(ticker: str, filing_date: str, window_days: int = ABNORMAL_RETURN_DAYS) -> Optional[dict]:
    try:
        import yfinance as yf
    except ImportError:
        return None

    try:
        start_dt = datetime.strptime(filing_date[:10], "%Y-%m-%d")
        fetch_start = (start_dt - timedelta(days=2)).strftime("%Y-%m-%d")
        fetch_end = (start_dt + timedelta(days=window_days + 5)).strftime("%Y-%m-%d")

        stock_data = yf.download(ticker, start=fetch_start, end=fetch_end, auto_adjust=True, progress=False, timeout=15)
        spy_data = yf.download(SPY_TICKER, start=fetch_start, end=fetch_end, auto_adjust=True, progress=False, timeout=15)

        if stock_data.empty or spy_data.empty:
            return None

        def _close(df):
            if isinstance(df.columns, pd.MultiIndex):
                cols = [c for c in df.columns if c[0] == "Close"]
                return df[cols[0]] if cols else df["Close"]
            return df["Close"]

        sc = _close(stock_data)
        sp = _close(spy_data)
        common = sc.index.intersection(sp.index)
        if len(common) < 2:
            return None

        sc = sc.loc[common]
        sp = sp.loc[common]

        avail = [d for d in common if d.date() >= start_dt.date()]
        if len(avail) < 2:
            return None

        t0 = avail[0]
        te = avail[min(window_days, len(avail) - 1)]

        def _sf(val):
            if isinstance(val, pd.Series):
                return float(val.iloc[0])
            return float(val)

        p0s = _sf(sc.loc[t0])
        pts = _sf(sc.loc[te])
        p0m = _sf(sp.loc[t0])
        ptm = _sf(sp.loc[te])

        if p0s <= 0 or p0m <= 0:
            return None

        sr = round((pts / p0s - 1) * 100, 3)
        mr = round((ptm / p0m - 1) * 100, 3)
        ar = round(sr - mr, 3)

        return {
            "stock_return_pct": sr,
            "spy_return_pct": mr,
            "abnormal_return_pct": ar,
            "window_start": t0.strftime("%Y-%m-%d"),
            "window_end": te.strftime("%Y-%m-%d"),
            "window_days": window_days,
            "data_source": "yahoo_finance",
            "market_reaction": ("strong_positive" if ar > 5 else
                                "positive" if ar > 2 else
                                "neutral" if ar > -2 else
                                "negative" if ar > -5 else "strong_negative"),
        }
    except Exception as e:
        logger.debug(f"  yfinance error {ticker}: {e}")
        return None

# ══════════════════════════════════════════════════════════════════════════════
# PROCESS COMPANY — V21 VERSION
# Key change: uses rag_retrieve_text_sections() instead of scrape_text_sections()
# XBRL pipeline unchanged.
# ══════════════════════════════════════════════════════════════════════════════

def process_company_v21(
    company: dict,
    sess: EDGARSession,
    llm,
    llm_strict,
    manifest: Dict,
    vs: SECVectorStore,
    forms: tuple = ("10-K", "10-Q"),
    max_per_form: int = 8,
    done_accessions: set = None,
    quality_log=None,
    sanity_log=None,
    gold_out=None,
    silver_out=None,
    out_file=None,
    pf=None,
) -> list:
    ticker = company["ticker"]
    company_name = company["name"]
    done_accessions = done_accessions or set()

    logger.info(f"══ {ticker} ({company_name}) ══")

    cik = ticker_to_cik(ticker, sess, company_name=company_name)
    if not cik:
        return []

    xbrl = load_xbrl_facts(cik, sess)
    prev_ytd_metrics: dict = {}
    prev_records: dict = {}
    all_records: list = []
    all_historical: list = []

    # Get filings from manifest
    ticker_manifest = manifest.get(ticker.upper(), {})

    for form_type in forms:
        if is_8k_form(form_type):
            continue

        # Build filing list from manifest
        filings = []
        base_form = form_type.split("/")[0]

        for year_str in sorted(ticker_manifest.keys()):
            for ft, entries in ticker_manifest[year_str].items():
                if not (ft == form_type or ft.startswith(base_form)):
                    continue
                for entry in entries:
                    filings.append({
                        "form_type": ft,
                        "filed_date": entry["period_end"],  # use period as proxy
                        "period_of_report": entry["period_end"],
                        "accession_no": entry["accession_no"],
                        "primary_document": "",
                        "_path": entry["path"],
                        "_year": entry["year"],
                    })

        # Sort by period ascending
        filings.sort(key=lambda f: f["period_of_report"])
        logger.info(f"  {form_type}: {len(filings)} filings (from manifest)")

        for filing in filings:
            accno = filing["accession_no"]
            period = filing["period_of_report"]
            ft = filing["form_type"]
            year = filing["_year"]

            if accno in done_accessions:
                logger.debug(f"  Skip (done): {accno}")
                continue

            logger.info(f"  ▶ {ft} | {period} | {accno}")

            rec_id = hashlib.md5(f"{ticker}_{ft}_{period}_{accno}".encode()).hexdigest()[:12]

            base_meta = {
                "ticker": ticker,
                "company_name": company_name,
                "cik": cik,
                "form_type": ft,
                "filed_date": period,
                "period_of_report": period,
                "fiscal_year": int(period[:4]) if period else None,
                "accession_no": accno,
            }

            ar = fetch_abnormal_return(ticker, period)
            if ar:
                base_meta["abnormal_return"] = ar
                logger.info(f"  AR: {ar['abnormal_return_pct']:+.2f}% ({ar['market_reaction']})")

            fiscal_quarter = get_fiscal_quarter(period, ft)
            prev_key = f"{ticker}_{ft}"
            prev_ytd = prev_ytd_metrics.get(prev_key)

            metrics, decum_applied = (
                extract_metrics_with_decumulation(xbrl, period, ft, prev_ytd, fiscal_quarter)
                if xbrl else ({}, False)
            )

            if xbrl and metrics:
                prev_ytd_metrics[prev_key] = extract_metrics(xbrl, period, ft)

            # ── V21-7: RAG text retrieval (replaces scrape_text_sections) ─────
            text_sections_raw, plain_text = rag_retrieve_text_sections(
                ticker=ticker,
                year=year,
                form_type=ft,
                vs=vs,
                manifest=manifest,
                period_end=period,   
            )
            text_sections = enrich_thin_mda(text_sections_raw, plain_text)

            # Count total RAG context chars for metadata
            rag_context_chars = sum(len(v) for v in text_sections.values() if v)

            # Scaling unit
            fs_text = text_sections.get("financial_statements", "") or plain_text[:3000]
            scaling_unit = ScalingNormaliser.detect(fs_text)

            # Segments
            segments: list = []
            if plain_text:
                segments = SegmentExtractor.extract(plain_text)
                if segments:
                    logger.info(f"  Segments found: {len(segments)} — {[s['name'] for s in segments]}")
                metrics["segments"] = segments

            # RegexTextSearch fallback for any missing metrics
            text_sourced_keys: list = []
            combined_text = " ".join(v for v in text_sections.values() if v and len(v) >= MIN_SECTION_CHARS)
            if combined_text:
                metrics, text_sourced_keys = RegexTextSearch.extract(combined_text, metrics)

            metrics = DerivedMetricEngine.enrich(metrics, {}, {})

            # XBRL force-fetch if below floor
            raw_count = DerivedMetricEngine.count_numerical_points(metrics, {})
            if xbrl and raw_count < MIN_METRICS_RESCAN:
                metrics = force_fetch_all_xbrl_fields(xbrl, period, ft, metrics)
                metrics = DerivedMetricEngine.enrich(metrics, {}, {})
                raw_count = DerivedMetricEngine.count_numerical_points(metrics, {})

            logger.info(
                f"  Metrics:{len(metrics)} | decum:{decum_applied} | "
                f"rag_chars:{rag_context_chars} | scaling:{scaling_unit} | "
                f"segments:{len(segments)} | "
                f"sections:{sum(1 for v in text_sections.values() if v and len(v) >= MIN_SECTION_CHARS)}"
            )

            prior_rec = prev_records.get(prev_key)
            qa_changes, quarter_aligned = compute_quarter_aligned_yoy(
                metrics, period, fiscal_quarter or 0, all_historical, ft
            )
            if not qa_changes and prior_rec:
                qa_changes = compute_changes(metrics, prior_rec.get("metrics", {}))
                quarter_aligned = False

            changes = qa_changes
            metrics = DerivedMetricEngine.enrich(metrics, changes, {})

            # Cross-ref guard
            cross_ref_result = CrossRefGuard.validate(
                fs_text=text_sections.get("financial_statements", "") or "",
                mda_text=text_sections.get("mda", "") or "",
                xbrl_rev=metrics.get("revenue"),
                xbrl_ni=metrics.get("net_income"),
                scaling=scaling_unit,
            )

            if not cross_ref_result["cross_ref_clean"]:
                if cross_ref_result["revenue_source"] == "fs_table_override":
                    metrics["revenue"] = cross_ref_result["revenue_validated"]
                if cross_ref_result["net_income_source"] == "fs_table_override":
                    metrics["net_income"] = cross_ref_result["net_income_validated"]
                metrics = DerivedMetricEngine.enrich(metrics, changes, {})

            sanity_result = sanity_check_metrics(metrics, changes, ticker, period, ft)
            if sanity_log and sanity_result["flags"]:
                with _file_lock:
                    sanity_log.write(json.dumps(sanity_result, ensure_ascii=False) + "\n")
                    sanity_log.flush()

            if sanity_result["action"] == "block":
                logger.warning(f"  ✗ SANITY BLOCK: {sanity_result['note']}")
                base_meta["sanity_blocked"] = True

            base_meta = enrich_period_type_metadata(
                base_meta, fiscal_quarter, decum_applied, quarter_aligned, sanity_result
            )
            base_meta["scaling_unit"] = scaling_unit

            signals = compute_signals(metrics, changes)

            paradox_data = LiquidityParadoxDetector.detect(metrics)
            if paradox_data:
                signals.setdefault("edge_cases_detected", [])
                if "liquidity_paradox" not in signals["edge_cases_detected"]:
                    signals["edge_cases_detected"].append("liquidity_paradox")

            cagr_data = None
            if len(all_historical) >= 2:
                cagr_data = CAGREngine.from_records(
                    [r for r in all_historical if r["meta"].get("form_type") == ft], "revenue"
                )

            edge_cases = signals.get("edge_cases_detected", [])
            reasoning_path = infer_reasoning_path(metrics, changes, signals, edge_cases)

            record = {
                "id": rec_id,
                "meta": base_meta,
                "metrics": metrics,
                "changes": changes,
                "signals": signals,
                "text_sections": text_sections,
            }

            passes_gate, num_count = DerivedMetricEngine.passes_gate(metrics, changes)
            if not passes_gate and xbrl and raw_count < MIN_METRICS_FLOOR:
                for field in ["revenue", "net_income", "gross_profit", "operating_income",
                              "free_cash_flow", "ebitda", "capex", "eps_diluted", "total_assets",
                              "total_equity", "long_term_debt", "cash_and_equivalents",
                              "operating_cash_flow"]:
                    if metrics.get(field) is None:
                        val = force_fetch_xbrl_tag(xbrl, field, period, ft)
                        if val is not None:
                            metrics[field] = val
                metrics = DerivedMetricEngine.enrich(metrics, changes, signals)
                passes_gate, num_count = DerivedMetricEngine.passes_gate(metrics, changes)

            total_keys = max(len([k for k in metrics if k != "segments"]), 1)
            text_pct = round(len(text_sourced_keys) / total_keys * 100)
            data_source_mix = {"xbrl_pct": 100 - text_pct, "text_pct": text_pct, "rag_retrieved": True}


            # ── Pre-LLM data validity gate ────────────────────────────────────
            is_data_valid, validity_errors = _validate_metrics_for_llm(
                metrics, changes, ticker, period
            )

            if len(metrics) >= 5 and sanity_result["action"] != "block" and is_data_valid:
                has_prior = prior_rec is not None and bool(prior_rec.get("metrics"))
                analysis, auditor_cycles = generate_analysis_v21(
                    record=record,
                    llm=llm,
                    llm_strict=llm_strict,
                    prior_record=prior_rec,
                    metrics=metrics,
                    changes=changes,
                    has_prior=has_prior,
                    text_sourced_keys=text_sourced_keys,
                )

                if analysis:
                    has_mda = any(v for v in text_sections.values() if v and len(v) >= MIN_SECTION_CHARS)
                    qr = EliteQualityValidator.elite_validate(analysis, metrics, changes, has_mda, has_prior=has_prior)

                    if quality_log:
                        log_entry = {
                            "ticker": ticker,
                            "period": period,
                            "accno": accno,
                            "form": ft,
                            "v21_rag": True,
                            "rag_context_chars": rag_context_chars,
                            "decum": decum_applied,
                            "q_aligned": quarter_aligned,
                            "sanity": sanity_result["action"],
                            "has_mda": has_mda,
                            "has_prior": has_prior,
                            "reasoning_path": reasoning_path,
                            "num_data_points": num_count,
                            "raw_metric_count": raw_count,
                            "auditor_cycles": auditor_cycles,
                            "causal_density": qr.get("causal_density_score", 0),
                            "data_source_mix": data_source_mix,
                            "scaling_unit": scaling_unit,
                            "cross_ref_clean": cross_ref_result.get("cross_ref_clean", True),
                            "segment_count": len(segments),
                            **{k: v for k, v in qr.items() if k != "elite_checks"},
                            "elite_summary": {
                                "req1_nums": qr["elite_checks"]["req1_numerical_density"]["total_nums"],
                                "req2_missing": qr["elite_checks"]["req2_mandatory_metrics"]["missing"],
                                "req3_causal": qr["elite_checks"]["req3_causal_reasoning"]["count"],
                                "req4_cagr": qr["elite_checks"]["req4_cagr_formula"].get("cagr_found", False),
                                "req7_signal": qr["elite_checks"]["req7_investment_signal"]["signal"],
                                "gold_ready": qr["gold_ready"],
                                "rejections": [r for r in qr["rejection_reasons"] if r],
                            },
                        }
                        with _file_lock:
                            quality_log.write(json.dumps(log_entry, ensure_ascii=False) + "\n")
                            quality_log.flush()

                    record["instruction_pair"] = build_single_pair(
                        record,
                        analysis,
                        prior_record=prior_rec,
                        quality_result=qr,
                        cagr_data=cagr_data,
                        paradox_data=paradox_data,
                        abnormal_return=ar,
                        reasoning_path=reasoning_path,
                        auditor_cycles=auditor_cycles,
                        data_source_mix=data_source_mix,
                        cross_ref_result=cross_ref_result,
                        scaling_unit=scaling_unit,
                        segments=segments if segments else None,
                        rag_context_chars=rag_context_chars,
                    )

                    _write_tiered(record, qr, None, gold_out, silver_out, has_abnormal_return=bool(ar))

                    mode = "RAG-FULL" if has_mda else "QUANT"
                    logger.info(
                        f"  ✓ {mode} [{ft}|Q={qr['quality_score']:.0f}|"
                        f"tier={qr['tier']}|auditor={auditor_cycles}|"
                        f"causal={qr.get('causal_density_score', 0)}|"
                        f"rag={rag_context_chars}chars]: {len(analysis)} chars"
                    )
                else:
                    logger.warning(f"  ✗ No analysis generated for {ticker} {period}")

            else:
                if not is_data_valid:
                    logger.warning(
                        f"  ✗ DATA_VALIDITY SKIP [{ft} {period}]: "
                        f"{'; '.join(validity_errors)}"
                    )
                record["_needs_llm"] = False
                record["_validity_errors"] = validity_errors if not is_data_valid else []

            _write_record_to_file(record, out_file, pf)
            all_records.append(record)
            all_historical.append(record)
            prev_records[prev_key] = record
            done_accessions.add(accno)

        # Negative examples
        neg_eligible = [r for r in all_records if r["meta"].get("form_type") == form_type and r.get("metrics") and r.get("changes")]
        for rec in neg_eligible[-3:]:
            negatives = build_negative_examples(rec["metrics"], rec["changes"], rec.get("text_sections", {}))
            if not negatives:
                continue

            neg_analysis = generate_analysis(build_negative_example_prompt(rec, negatives), llm)
            if neg_analysis:
                neg_pair = build_negative_pair(rec, neg_analysis, negatives)
                neg_rec = {
                    "id": hashlib.md5(f"{ticker}_{form_type}_neg_{rec['meta']['period_of_report']}".encode()).hexdigest()[:12],
                    "meta": {**rec["meta"], "record_type": "negative_reasoning"},
                    "metrics": rec["metrics"],
                    "changes": rec["changes"],
                    "signals": rec["signals"],
                    "text_sections": {},
                    "instruction_pair": neg_pair,
                }
                _write_record_to_file(neg_rec, out_file, pf)
                all_records.append(neg_rec)
                logger.info(f"  ✓ Negative example [{form_type}]: {len(neg_analysis)} chars")

    if _session_manager.record_company_done():
        sess.reset_connection()

    _increment_queue_processed()
    logger.info(f"✓ {ticker}: {len(all_records)} records | llm={_llm_rate_limiter.stats()}")
    return all_records

# ─── RESUME HELPERS ───────────────────────────────────────────────────────────
def load_done_accessions(output_file: str) -> set:
    done = set()
    if not os.path.exists(output_file):
        return done

    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                accno = rec.get("meta", {}).get("accession_no")
                if accno:
                    done.add(accno)
            except Exception:
                pass
    logger.info(f"Resume: {len(done)} accession numbers already processed")
    return done

def load_done_tickers(output_file: str) -> set:
    done = set()
    if not os.path.exists(output_file):
        return done

    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                ticker = rec.get("meta", {}).get("ticker")
                if ticker:
                    done.add(ticker)
            except Exception:
                pass
    return done

# ══════════════════════════════════════════════════════════════════════════════
# DATASET BUILDER v21
# ══════════════════════════════════════════════════════════════════════════════

def build_dataset_v21(
    tickers: list,
    output_file: str = OUTPUT_FILE,
    forms: tuple = ("10-K", "10-Q"),
    max_per_form: int = 8,
    resume: bool = True,
    filings_root: str = FILINGS_ROOT,
) -> int:
    with _queue_lock:
        _queue_state["total"] = len(tickers)
        _queue_state["processed"] = 0

    llm = get_llm_for_context()
    llm_strict = get_llm_for_context(strict=True)

    sess = EDGARSession()
    vs = get_vector_store()

    # Build manifest from local HTML files
    manifest = build_filing_manifest(filings_root)

    # Pre-index all filings for requested tickers
    logger.info("Pre-indexing local filings into Qdrant…")
    total_indexed = 0
    for company in tickers:
        t = company["ticker"] if isinstance(company, dict) else company
        total_indexed += index_ticker(t, manifest, vs)
    logger.info(f"Total chunks indexed: {total_indexed}")

    done_accessions: set = set()
    remaining = tickers

    if resume:
        done_accessions = load_done_accessions(output_file)
        done_tickers = load_done_tickers(output_file)
        remaining = [t for t in tickers if (t["ticker"] if isinstance(t, dict) else t) not in done_tickers]
        logger.info(f"Resume mode: {len(done_tickers)} tickers done, {len(remaining)} remaining")

    fmode = "a" if resume else "w"
    out_file = open(output_file, fmode, encoding="utf-8")
    pf = open(PENDING_FILE, fmode, encoding="utf-8")
    quality_log = open(QUALITY_LOG_FILE, fmode, encoding="utf-8")
    sanity_log = open(SANITY_LOG_FILE, fmode, encoding="utf-8")
    gold_out = open(GOLD_FILE, fmode, encoding="utf-8")
    silver_out = open(SILVER_FILE, fmode, encoding="utf-8")

    total_records = 0

    try:
        with tqdm(total=len(remaining), desc="Companies", unit="co") as pbar:
            with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as executor:
                futures = {
                    executor.submit(
                        process_company_v21,
                        company=company,
                        sess=sess,
                        llm=llm,
                        llm_strict=llm_strict,
                        manifest=manifest,
                        vs=vs,
                        forms=forms,
                        max_per_form=max_per_form,
                        done_accessions=done_accessions,
                        quality_log=quality_log,
                        sanity_log=sanity_log,
                        gold_out=gold_out,
                        silver_out=silver_out,
                        out_file=out_file,
                        pf=pf,
                    ): company for company in remaining
                }

                for future in as_completed(futures):
                    company = futures[future]
                    ticker = company["ticker"] if isinstance(company, dict) else company
                    try:
                        records = future.result()
                        total_records += len(records)
                        pbar.set_postfix({"last": ticker, "records": total_records})
                    except Exception as e:
                        logger.error(f"  ✗ {ticker} failed: {e}", exc_info=True)
                    finally:
                        pbar.update(1)

    except KeyboardInterrupt:
        logger.warning("Interrupted — progress saved, safe to resume")
    finally:
        for fh in (out_file, pf, quality_log, sanity_log, gold_out, silver_out):
            try:
                fh.flush()
                fh.close()
            except Exception:
                pass

    return total_records

def report_dataset_stats_v21(
    output_file: str,
    quality_log_file: str,
    sanity_log_file: str,
    gold_file: str,
    silver_file: str,
):
    def _count(path: str) -> int:
        if not os.path.exists(path):
            return 0
        with open(path, "r", encoding="utf-8") as f:
            return sum(1 for ln in f if ln.strip())

    total = _count(output_file)
    gold_count = _count(gold_file)
    silver_count = _count(silver_file)
    bronze_count = max(0, total - gold_count - silver_count)

    sanity_flags = _count(sanity_log_file)

    causal_all: list = []
    rac_all: list = []
    scores_all: dict = {}

    if os.path.exists(quality_log_file):
        with open(quality_log_file, "r", encoding="utf-8") as qf:
            for line in qf:
                line = line.strip()
                if not line:
                    continue
                try:
                    entry = json.loads(line)
                    tk = entry.get("ticker", "?")
                    scores_all.setdefault(tk, []).append(entry.get("quality_score", 0))
                    cd = entry.get("causal_density", 0)
                    if cd:
                        causal_all.append(cd)
                    rc = entry.get("rag_context_chars", 0)
                    if rc:
                        rac_all.append(rc)
                except Exception:
                    pass

    avg_causal = sum(causal_all) / max(len(causal_all), 1)
    avg_rag = sum(rac_all) / max(len(rac_all), 1)

    gold_pct = gold_count / max(total, 1) * 100

    avg_scores = {tk: round(sum(v) / len(v), 1) for tk, v in scores_all.items()}
    top5 = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)[:5]
    bottom5 = sorted(avg_scores.items(), key=lambda x: x[1])[:5]

    print("=" * 70)
    print("SEC Fine-Tuning Dataset Builder v21 RAG — QUALITY REPORT")
    print("=" * 70)
    print(f"  Total records written    : {total}")
    print(f"  Tickers covered          : {len(scores_all)}")
    print(f"  Gold tier (≥{GOLD_QUALITY_THRESHOLD})       : {gold_count} ({gold_pct:.1f}%)")
    print(f"  Silver tier (≥{SILVER_QUALITY_THRESHOLD})   : {silver_count} ({silver_count / max(total,1)*100:.1f}%)")
    print(f"  Bronze tier (<{SILVER_QUALITY_THRESHOLD})   : {bronze_count} ({bronze_count / max(total,1)*100:.1f}%)")
    print(f"  Sanity-flagged records   : {sanity_flags}")
    print("-" * 70)
    print(f"  V21 avg RAG context      : {avg_rag:.0f} chars per filing (target >3000)")
    print(f"  V21 avg causal density   : {avg_causal:.1f} connectors/record (need ≥{CAUSAL_CONNECTOR_MIN})")
    print("-" * 70)
    if gold_pct >= 75:
        print(f"  ✓ V21 Gold target met: {gold_pct:.1f}% (RAG vs v20 ~45%)")
    else:
        print(f"  ⚠ V21 Gold {gold_pct:.1f}% — review quality log")
    print("-" * 70)
    if top5:
        print("  Top-5 tickers:")
        for tk, sc in top5:
            print(f"    {tk:10s} {sc:.1f}")
    if bottom5:
        print("  Bottom-5 tickers:")
        for tk, sc in bottom5:
            print(f"    {tk:10s} {sc:.1f}")
    print("=" * 70)




# ══════════════════════════════════════════════════════════════════════════════
# TICKER LIST (same as v20, abbreviated here for clarity)
# ══════════════════════════════════════════════════════════════════════════════


def get_sp500_tickers() -> list:
    return [
        {"ticker": "AAPL",  "name": "Apple Inc"},
        {"ticker": "MSFT",  "name": "Microsoft Corp"},
        {"ticker": "NVDA",  "name": "NVIDIA Corp"},
        {"ticker": "AVGO",  "name": "Broadcom Inc"},
        {"ticker": "ORCL",  "name": "Oracle Corp"},
        {"ticker": "ADBE",  "name": "Adobe Inc"},
        {"ticker": "CRM",   "name": "Salesforce Inc"},
        {"ticker": "AMD",   "name": "Advanced Micro Devices Inc"},
        {"ticker": "INTC",  "name": "Intel Corp"},
        {"ticker": "QCOM",  "name": "Qualcomm Inc"},
        {"ticker": "TXN",   "name": "Texas Instruments Inc"},
        {"ticker": "IBM",   "name": "International Business Machines Corp"},
        {"ticker": "CSCO",  "name": "Cisco Systems Inc"},
    ]




# ── Patch _dense_search for qdrant-client ≥ 1.7 ──────────────────────────────
import types
from typing import List, Tuple, Optional


def _dense_search_patched(
    self,
    query: str,
    ticker: str,
    year: int,
    form_type: str,
    section: Optional[str] = None,
    quant_only: bool = False,
    top_k: int = TOP_K_PER_QUERY,
    period_end: Optional[str] = None,  # ← ADD THIS LINE
) -> List[Tuple[dict, float]]:
    client = self._get_client()
    vec = self._embed([query])[0]

    must_conditions = [
        FieldCondition(key="ticker",    match=MatchValue(value=ticker)),
        FieldCondition(key="year",      match=MatchValue(value=year)),
        FieldCondition(key="form_type", match=MatchValue(value=form_type)),
    ]
    if section:
        must_conditions.append(FieldCondition(key="section", match=MatchValue(value=section)))
    if quant_only:
        must_conditions.append(FieldCondition(key="is_quantitative", match=MatchValue(value=True)))

    if period_end:  # ✓ Now period_end is defined
        must_conditions.append(
            FieldCondition(key="period_end", match=MatchValue(value=period_end))
        )
    qfilter = Filter(must=must_conditions)

    try:                                          # qdrant-client ≥ 1.7
        response = client.query_points(
            collection_name=self.collection,
            query=vec,
            query_filter=qfilter,
            limit=top_k,
            with_payload=True,
        )
        points = response.points
    except AttributeError:                        # qdrant-client < 1.7 fallback
        points = client.search(
            collection_name=self.collection,
            query_vector=vec,
            query_filter=qfilter,
            limit=top_k,
            with_payload=True,
        )

    return [(p.payload, p.score) for p in points]


# Apply patch — no kernel restart needed
SECVectorStore._dense_search = _dense_search_patched
if _vector_store is not None:
    _vector_store._dense_search = types.MethodType(_dense_search_patched, _vector_store)

print(f"qdrant-client version: {__import__('importlib.metadata').metadata.version('qdrant-client')}")
print("✓ _dense_search patched — now run the entry-point cell")



# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════


# ══════════════════════════════════════════════════════════════════════════════
# NOTEBOOK ENTRY POINT  (no argparse — just edit the config block below)
# ══════════════════════════════════════════════════════════════════════════════

# ── CONFIG — edit these cells, then Run All ───────────────────────────────────
RUN_TICKERS      = ["AAPL"]          # [] = run full list from get_sp500_tickers()
RUN_FORMS        = ["10-K", "10-Q"]
RUN_MAX_PER_FORM = 8
RUN_OUTPUT       = OUTPUT_FILE       # "sec_finetune_v21.jsonl"

RUN_RESUME       = True              # False = start fresh
RUN_INDEX_ONLY   = False             # True = build index, skip LLM
RUN_STATS_ONLY   = False             # True = print report, skip everything

# ── Resolve ticker list ───────────────────────────────────────────────────────
_all = get_sp500_tickers()

if RUN_TICKERS:
    _upper  = {t.upper() for t in RUN_TICKERS}
    _known  = {c["ticker"].upper() for c in _all}
    selected = [c for c in _all if c["ticker"].upper() in _upper]
    # allow tickers not in master list
    for t in RUN_TICKERS:
        if t.upper() not in {c["ticker"].upper() for c in selected}:
            selected.append({"ticker": t.upper(), "name": t.upper()})
else:
    selected = _all

logger.info(
    f"v21 RAG | tickers={len(selected)} | forms={RUN_FORMS} | "
    f"max_per_form={RUN_MAX_PER_FORM} | filings_root={FILINGS_ROOT} | "
    f"resume={RUN_RESUME}"
)

# ── Stats-only ────────────────────────────────────────────────────────────────
if RUN_STATS_ONLY:
    report_dataset_stats_v21(
        output_file=RUN_OUTPUT,
        quality_log_file=QUALITY_LOG_FILE,
        sanity_log_file=SANITY_LOG_FILE,
        gold_file=GOLD_FILE,
        silver_file=SILVER_FILE,
    )

# ── Index-only ────────────────────────────────────────────────────────────────
elif RUN_INDEX_ONLY:
    logger.info("Index-only mode: building Qdrant index from local HTML files…")
    manifest = build_filing_manifest(FILINGS_ROOT)
    vs       = get_vector_store()
    total_chunks = 0
    for company in tqdm(selected, desc="Indexing", unit="ticker"):
        n = index_ticker(company["ticker"], manifest, vs)
        total_chunks += n
        logger.info(f"  {company['ticker']}: {n} chunks")
    logger.info(f"Index complete — total chunks: {total_chunks}")

# ── Full pipeline ─────────────────────────────────────────────────────────────
else:
    _t0 = time.time()

    total = build_dataset_v21(
        tickers=selected,
        output_file=RUN_OUTPUT,
        forms=tuple(RUN_FORMS),
        max_per_form=RUN_MAX_PER_FORM,
        resume=RUN_RESUME,
        filings_root=FILINGS_ROOT,
    )

    logger.info(
        f"Pipeline complete in {(time.time()-_t0)/60:.1f} min | "
        f"total records: {total}"
    )

    report_dataset_stats_v21(
        output_file=RUN_OUTPUT,
        quality_log_file=QUALITY_LOG_FILE,
        sanity_log_file=SANITY_LOG_FILE,
        gold_file=GOLD_FILE,
        silver_file=SILVER_FILE,
    )

17:19:34 | INFO    | v21 RAG | tickers=1 | forms=['10-K', '10-Q'] | max_per_form=8 | filings_root=filings | resume=True


qdrant-client version: 1.18.0
✓ _dense_search patched — now run the entry-point cell


17:19:34 | INFO    | Filing manifest: 13 tickers, 365 HTML files
17:19:34 | INFO    | Pre-indexing local filings into Qdrant…
17:19:34 | INFO    |   Indexing AAPL 10-K 2019-09-28 (10-K_2019-09-28_0000320193-19-000119.html)
17:19:42 | INFO    | Created Qdrant collection: sec_filings_v21
17:19:42 | INFO    | Loading embedding model: all-MiniLM-L6-v2
/Users/fanilshakirov/conda/projects/env/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
17:19:55 | INFO    |   → 11 chunks indexed
17:19:55 | INFO    |   Indexing AAPL 10-Q 2019-06-29 (10-Q_2019-06-29_0000320193-19-000076.html)
17:20:09 | INFO    |   → 6 chunks indexed
17:20:09 | INFO    |   Indexing AAPL 10-Q 2019-12-28 (10-Q_2019-12-28_0000320193-20-000010.html)
17:20:15 | INFO    |   → 6 chunks indexed
17:20:15 | INFO 

SEC Fine-Tuning Dataset Builder v21 RAG — QUALITY REPORT
  Total records written    : 0
  Tickers covered          : 0
  Gold tier (≥90)       : 0 (0.0%)
  Silver tier (≥75)   : 0 (0.0%)
  Bronze tier (<75)   : 0 (0.0%)
  Sanity-flagged records   : 0
----------------------------------------------------------------------
  V21 avg RAG context      : 0 chars per filing (target >3000)
  V21 avg causal density   : 0.0 connectors/record (need ≥7)
----------------------------------------------------------------------
  ⚠ V21 Gold 0.0% — review quality log
----------------------------------------------------------------------


In [7]:
from importlib.metadata import version
print(version("qdrant-client"))
# ≥ 1.7.0 → needs query_points
# < 1.7.0 → search still works

1.18.0


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Model loaded")

/Users/fanilshakirov/conda/projects/env/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Model loaded


In [5]:
import torch, transformers, sentence_transformers

print(torch.__version__)
print(transformers.__version__)
print(sentence_transformers.__version__)

2.2.2
4.41.2
2.7.0


In [ ]:
# uv pip install torch==2.2.2 transformers==4.41.2 sentence-transformers==2.7.0 huggingface_hub==0.23.0

In [16]:
import os

# ── 1. Working directory ──────────────────────────────────────────────────────
print(f"CWD: {os.getcwd()}")
print(f"Filings root exists: {Path(FILINGS_ROOT).exists()}")
print(f"Filings root abs:    {Path(FILINGS_ROOT).resolve()}")

# ── 2. Manifest check ─────────────────────────────────────────────────────────
_diag_manifest = build_filing_manifest(FILINGS_ROOT)
print(f"\nManifest tickers: {list(_diag_manifest.keys())}")
for tkr, yrs in _diag_manifest.items():
    for yr, forms in yrs.items():
        for ft, entries in forms.items():
            print(f"  {tkr}/{yr}/{ft}: {len(entries)} file(s)")

# ── 3. XBRL connectivity check ────────────────────────────────────────────────
_diag_sess  = EDGARSession()
_diag_cik   = ticker_to_cik("AAPL", _diag_sess)
print(f"\nAAPL CIK: {_diag_cik}")
if _diag_cik:
    _diag_xbrl = load_xbrl_facts(_diag_cik, _diag_sess)
    print(f"XBRL GAAP concepts loaded: {len(_diag_xbrl.get('us-gaap', {}))}")

# ── 4. LLM API key ────────────────────────────────────────────────────────────
print(f"\nNVIDIA_API_KEY_QWEN set: {bool(os.getenv('NVIDIA_API_KEY_QWEN'))}")

22:06:18 | INFO    | Filing manifest: 13 tickers, 365 HTML files


CWD: /Users/fanilshakirov/conda/projects/practice_projects/01_finetuning/v21/01
Filings root exists: True
Filings root abs:    /Users/fanilshakirov/conda/projects/practice_projects/01_finetuning/v21/01/filings

Manifest tickers: ['AAPL', 'ADBE', 'AMD', 'AVGO', 'CRM', 'CSCO', 'IBM', 'INTC', 'MSFT', 'NVDA', 'ORCL', 'QCOM', 'TXN']
  AAPL/2019/10-K: 1 file(s)
  AAPL/2019/10-Q: 2 file(s)
  AAPL/2020/10-K: 1 file(s)
  AAPL/2020/10-Q: 3 file(s)
  AAPL/2021/10-K: 1 file(s)
  AAPL/2021/10-Q: 3 file(s)
  AAPL/2022/10-K: 1 file(s)
  AAPL/2022/10-Q: 3 file(s)
  AAPL/2023/10-K: 1 file(s)
  AAPL/2023/10-Q: 3 file(s)
  AAPL/2024/10-K: 1 file(s)
  AAPL/2024/10-Q: 3 file(s)
  AAPL/2025/10-K: 1 file(s)
  AAPL/2025/10-Q: 3 file(s)
  AAPL/2026/10-Q: 1 file(s)
  ADBE/2019/10-K: 1 file(s)
  ADBE/2019/10-Q: 2 file(s)
  ADBE/2020/10-K: 1 file(s)
  ADBE/2020/10-Q: 3 file(s)
  ADBE/2021/10-K: 1 file(s)
  ADBE/2021/10-Q: 3 file(s)
  ADBE/2022/10-K: 1 file(s)
  ADBE/2022/10-Q: 3 file(s)
  ADBE/2023/10-K: 1 file(s

22:06:20 | INFO    |   XBRL: 503 GAAP + 2 DEI


XBRL GAAP concepts loaded: 503

NVIDIA_API_KEY_QWEN set: True


In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# DEBUG RUNNER — sequential, no ThreadPoolExecutor, full tracebacks visible
# Switch back to the normal entry point once this works cleanly.
# ══════════════════════════════════════════════════════════════════════════════

import traceback

RUN_TICKERS      = ["AAPL"]
RUN_FORMS        = ["10-K"]      # start with just 10-K to keep it fast
RUN_MAX_PER_FORM = 3             # only 3 filings for the smoke-test
RUN_OUTPUT       = OUTPUT_FILE

RUN_RESUME       = False         # fresh start for debug

# ── Resolve tickers ───────────────────────────────────────────────────────────
_all      = get_sp500_tickers()
_upper    = {t.upper() for t in RUN_TICKERS}
selected  = [c for c in _all if c["ticker"].upper() in _upper]

# ── Shared objects ────────────────────────────────────────────────────────────
_dbg_sess     = EDGARSession()
_dbg_vs       = get_vector_store()
_dbg_manifest = build_filing_manifest(FILINGS_ROOT)
_dbg_llm      = get_llm_for_context()
_dbg_llm_s    = get_llm_for_context(strict=True)

# ── Pre-index ─────────────────────────────────────────────────────────────────
for _c in selected:
    _n = index_ticker(_c["ticker"], _dbg_manifest, _dbg_vs)
    print(f"Indexed {_c['ticker']}: {_n} chunks")

# ── Open output files ─────────────────────────────────────────────────────────
_fmode       = "w"   # fresh start
_dbg_out     = open(RUN_OUTPUT,       _fmode, encoding="utf-8")
_dbg_pf      = open(PENDING_FILE,     _fmode, encoding="utf-8")
_dbg_qlog    = open(QUALITY_LOG_FILE, _fmode, encoding="utf-8")
_dbg_slog    = open(SANITY_LOG_FILE,  _fmode, encoding="utf-8")
_dbg_gold    = open(GOLD_FILE,        _fmode, encoding="utf-8")
_dbg_silver  = open(SILVER_FILE,      _fmode, encoding="utf-8")

# ── Run sequentially with full tracebacks ─────────────────────────────────────
_total = 0
_done_acc: set = set()

for _company in selected:
    print(f"\n{'='*60}")
    print(f"Processing: {_company['ticker']}")
    print(f"{'='*60}")
    try:
        _recs = process_company_v21(
            company        = _company,
            sess           = _dbg_sess,
            llm            = _dbg_llm,
            llm_strict     = _dbg_llm_s,
            manifest       = _dbg_manifest,
            vs             = _dbg_vs,
            forms          = tuple(RUN_FORMS),
            max_per_form   = RUN_MAX_PER_FORM,
            done_accessions= _done_acc,
            quality_log    = _dbg_qlog,
            sanity_log     = _dbg_slog,
            gold_out       = _dbg_gold,
            silver_out     = _dbg_silver,
            out_file       = _dbg_out,
            pf             = _dbg_pf,
        )
        _total += len(_recs)
        print(f"✓ {_company['ticker']}: {len(_recs)} records written")
    except Exception:
        print(f"✗ {_company['ticker']} FAILED:")
        traceback.print_exc()

# ── Flush & close ─────────────────────────────────────────────────────────────
for _fh in (_dbg_out, _dbg_pf, _dbg_qlog, _dbg_slog, _dbg_gold, _dbg_silver):
    _fh.flush(); _fh.close()

print(f"\nTotal records: {_total}")

# ── Quick file-size check ─────────────────────────────────────────────────────
for _fname in [RUN_OUTPUT, GOLD_FILE, SILVER_FILE, QUALITY_LOG_FILE]:
    _sz = os.path.getsize(_fname) if os.path.exists(_fname) else 0
    print(f"  {_fname}: {_sz:,} bytes")

# ── Print quality report ──────────────────────────────────────────────────────
report_dataset_stats_v21(RUN_OUTPUT, QUALITY_LOG_FILE, SANITY_LOG_FILE, GOLD_FILE, SILVER_FILE)

22:06:26 | INFO    | Filing manifest: 13 tickers, 365 HTML files


NameError: name 'cid' is not defined